# scratch space

all my random testing scripts so this repo doesn't get cluttered with different notebooks

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os
from pathlib import Path
import re
import sys
import tempfile
import time

import cv2
import finufft
from IPython.display import HTML
import matplotlib.pyplot as plt
import numpy as np
from numpy.fft import fftn, fftshift, ifftn, ifftshift, fftfreq, fftshift
from scipy.ndimage import uniform_filter
from scipy import ndimage as ndi
from scipy import signal
from tifffile import imread, imwrite
from tqdm.auto import tqdm
import visu3d as v3d
import zarr

from sptflux import flux_probing, ihpp_data, plotting, ihpp_sim, datastructs, utils
from sptflux.logger import get_logger

logger = get_logger(__name__)

# event camera test shenanigans

## voltage tests

### 1d voltage tests

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class EventPixelSimulator:
    def __init__(self, a, b, lambda_r, delta_lambda_p, delta_lambda_n=None):
        """
        Initialize the Event Pixel Simulator based on Benson & Holzinger (2024).
        
        Parameters:
        -----------
        a : float
            Voltage decay rate (s^-1)[cite: 70].
        b : float
            Gain constant (V/s/ln(lambda/lambda_r))[cite: 82].
        lambda_r : float
            Reference flux constant[cite: 82].
        delta_lambda_p : float
            Positive brightness change threshold (e.g., 0.1 for 10%)[cite: 101].
        delta_lambda_n : float, optional
            Negative brightness change threshold. If None, assumes symmetric.
        """
        self.a = a
        self.b = b
        self.lambda_r = lambda_r
        
        # Calculate voltage thresholds c+ and c- based on Eq. 7 [cite: 104]
        # c+ corresponds to +1 event, c- to -1 event
        self.c_plus = (b / a) * np.log(1 + delta_lambda_p)
        
        if delta_lambda_n is None:
            # Paper generally assumes symmetric relative change threshold
            self.c_minus = (b / a) * np.log(1 - delta_lambda_p) 
        else:
            self.c_minus = (b / a) * np.log(1 - delta_lambda_n)

    def generate_photon_arrivals(self, t_start, t_end, flux_func, lambda_c_margin=1.1):
        """
        Simulate photon arrival times using the Acceptance-Rejection (thinning) method
        for a Non-Homogeneous Poisson Process (NHPP) .
        
        Returns:
        --------
        arrival_times : np.array
            Sorted array of photon arrival times.
        """
        # 1. Determine maximum flux (lambda_c) for the thinning process
        test_times = np.linspace(t_start, t_end, 1000)
        max_flux = np.max(flux_func(test_times))
        lambda_c = max_flux * lambda_c_margin
        
        # 2. Generate Homogeneous Poisson Process (HPP) arrivals with rate lambda_c
        expected_n = int((t_end - t_start) * lambda_c)
        num_raw = np.random.poisson(expected_n * 1.2) 
        inter_arrivals = np.random.exponential(1.0 / lambda_c, num_raw)
        arrival_candidates = np.cumsum(inter_arrivals) + t_start
        arrival_candidates = arrival_candidates[arrival_candidates <= t_end]
        
        # 3. Thinning step: Accept with probability lambda(t) / lambda_c
        prob_accepts = flux_func(arrival_candidates) / lambda_c
        random_values = np.random.rand(len(arrival_candidates))
        accepted = random_values < prob_accepts
        accepted_times = arrival_candidates[accepted]
        # accepted_times = []
        # for t in arrival_candidates:
        #     prob_accept = flux_func(t) / lambda_c
        #     if np.random.rand() < prob_accept:
        #         accepted_times.append(t)
                
        return np.array(accepted_times)

    def simulate_voltage_and_events(self, t_eval, arrival_times, flux_func, v0=0.0):
        """
        Calculate continuous voltage V(t) and generate events with precise timing.
        Solves for exact crossover times during the decay phase, including after the last photon.
        """
        # Pre-calculate logarithmic scaling factor h(Tn) for all photons
        lams = flux_func(arrival_times)
        h_values = self.b * (np.log(lams) - np.log(self.lambda_r)) / lams
        h_values[h_values < 1e-9] = 0.0
        print(h_values.max())
        
        # State variables
        v_current = v0
        t_current = t_eval[0]
        v_ref = v0 
        
        events = []
        
        # Filter relevant arrivals
        valid_indices = np.where(arrival_times >= t_eval[0])[0]
        
        # Tracking variables for the simulation loop
        sim_time = t_current
        sim_voltage = v_current
        
        # --- Helper to process decay over an interval [t_start, t_end] ---
        def process_decay_interval(t_start, t_end, current_v, current_ref):
            local_time = t_start
            local_v = current_v
            local_ref = current_ref
            local_events = []
            
            while True:
                t_cross = None
                polarity = 0
                v_target_val = 0
                
                # Determine next target threshold based on current state
                candidates = []
                
                # Decay down to Lower Threshold (valid if V > threshold and V > 0)
                v_target_down = local_ref + self.c_minus
                if local_v > v_target_down and local_v > 0:
                    if v_target_down > 0:
                        dt = -(1.0/self.a) * np.log(v_target_down / local_v)
                        candidates.append((dt, -1, v_target_down))
                
                # Decay up to Upper Threshold (valid if V < threshold and V < 0)
                v_target_up = local_ref + self.c_plus
                if local_v < v_target_up and local_v < 0:
                    if v_target_up < 0:
                        dt = -(1.0/self.a) * np.log(v_target_up / local_v)
                        candidates.append((dt, 1, v_target_up))
                
                # Select earliest valid crossing
                valid_candidates = [c for c in candidates if c[0] > 1e-9]
                if not valid_candidates:
                    break
                
                valid_candidates.sort(key=lambda x: x[0])
                best_dt, best_pol, best_v_target = valid_candidates[0]
                
                potential_time = local_time + best_dt
                
                # If crossing happens within the interval
                if potential_time < t_end:
                    local_events.append((potential_time, best_pol))
                    
                    # Update Ref
                    if best_pol == 1:
                        local_ref += self.c_plus
                    else:
                        local_ref += self.c_minus
                        
                    # Update State
                    local_time = potential_time
                    local_v = best_v_target
                else:
                    break # Crossing is outside this interval
            
            # Calculate final voltage at t_end
            dt_final = t_end - local_time
            v_final = local_v * np.exp(-self.a * dt_final)
            
            return local_events, v_final, local_ref

        # --- MAIN SIMULATION LOOP (Photon Arrivals) ---
        for idx in valid_indices:
            tn = arrival_times[idx]
            h = h_values[idx]
            
            # 1. Decay Phase (Last event -> Current Photon)
            new_events, v_at_arrival, v_ref = process_decay_interval(sim_time, tn, sim_voltage, v_ref)
            events.extend(new_events)
            
            # 2. Impulse Phase (At tn)
            v_after_jump = v_at_arrival + h
            
            # Check for immediate events caused by the jump
            # Note: this can cross over multiple thresholds, but only register one event
            if (v_after_jump - v_ref) >= self.c_plus:
                events.append((tn, 1))
                v_ref = v_after_jump
                # v_ref += self.c_plus
            
            # Update state
            sim_time = tn
            sim_voltage = v_after_jump

        # --- FINAL DECAY PHASE (Last Photon -> End of Simulation) ---
        t_sim_end = t_eval[-1]
        if sim_time < t_sim_end:
            final_decay_events, final_v, v_ref = process_decay_interval(sim_time, t_sim_end, sim_voltage, v_ref)
            events.extend(final_decay_events)
            # Update sim_voltage one last time if you needed the final value, 
            # though it's not strictly needed for the event list.
            sim_voltage = final_v 

        # --- Reconstruct Continuous Voltage for Plotting ---
        voltages = np.zeros_like(t_eval)
        photon_cursor = 0
        num_photons = len(arrival_times)
        
        # We can calculate this more efficiently or clearly
        for i, t in enumerate(t_eval):
            if i == 0:
                voltages[i] = v0
                continue
            
            # Decay from previous step
            dt_step = t - t_eval[i-1]
            v_now = voltages[i-1] * np.exp(-self.a * dt_step)
            
            # Add impulses that happened in this step
            while photon_cursor < num_photons and arrival_times[photon_cursor] <= t:
                tn_p = arrival_times[photon_cursor]
                h_p = h_values[photon_cursor]
                # Add jump decayed to current time t
                v_now += h_p * np.exp(-self.a * (t - tn_p))
                if h_p * np.exp(-self.a * (t - tn_p)) > 0.5:
                    print(flux_func(tn_p), h_p)
                    print(h_p, tn_p, t, h_p * np.exp(-self.a * (t - tn_p)))
                photon_cursor += 1
                
            voltages[i] = v_now

        return voltages, events

def peaked_absolute_value(x, height=1.0):
    """
    Returns a negative absolute value (triangle wave) function.
    Zeros at x=0 and x=1, peak at x=0.5.
    
    Parameters:
    x (float): The input value.
    height (float): The maximum value at the peak (x=0.5).
    """
    return 2 * height * (0.5 - np.abs(x - 0.5))

def flux_function(t):
    # lambda_1 = 10000.0 # High contrast to force events
    # lambda_0 = 11000.0
    # omega = 2 * np.pi * 100.0 # change Hz
    # val = lambda_0 + lambda_1 * np.sin(omega * t)

    val = 11000 + 5000 * np.sin(2 * np.pi * 20.5 * t) + 3000 * np.cos(2 * np.pi * 5.0 * t) + 2000 * np.sin(2 * np.pi * 7.0 * t + np.pi)

    # val = np.full_like(t, 10000.0)  # Constant flux for baseline testing

    # val = np.exp(3 + 2.0 * np.sin(2 * np.pi * 2.0 * t))  # Exponential of sine for dynamic range

    # val = peaked_absolute_value(t, height=10000) + 1

    # val = 10000 - 10000 * t + 1

    # val = 10000 * t + 1
    # flattop_where = t >= 0.5
    # val[flattop_where] = 5000
    return val

def h(t, lam_func=None, b=1.0, lambda_r=0.1):
    flux = lam_func(t)
    return b * (np.log(flux) - np.log(lambda_r)) / flux

# --- Simulation Execution ---

# 1. Define Parameters (Section 3.1 / Fig 1) [cite: 283]
# a=1 s^-1, b=1, lambda_r=0.1, delta_lambda=10%
a = 0.0001  # voltage decay rate (s^-1)
b = 1.0  # constant gain
lambda_r = 0.1  # reference flux
delta_lambda_p = 0.01  # mean brightness change threshold
sim = EventPixelSimulator(a=a, b=b, lambda_r=lambda_r, delta_lambda_p=delta_lambda_p)

# 3. Run Simulation
tstart = 0.0
tend = 1.0
t_eval = np.linspace(tstart, tend, 10000)
print("Generating photons...")
photon_times = sim.generate_photon_arrivals(tstart, tend, flux_function)
print(f"Photons generated: {len(photon_times)}")

print("Simulating voltage and events...")
# Initialize at expected steady state voltage for lambda(0) [cite: 122]
v_init = (sim.b / sim.a) * np.log(flux_function(tstart) / sim.lambda_r)
voltages, events = sim.simulate_voltage_and_events(t_eval, photon_times, flux_function, v0=v_init)
print(f"Events generated: {len(events)}")

# 4. Visualization
plt.figure(figsize=(10, 8))

# Flux
plt.subplot(4, 1, 1)
plt.plot(t_eval, flux_function(t_eval), 'k--', label='Input Flux $\lambda(t)$')
plt.scatter(photon_times, flux_function(photon_times), color='r', s=10, label='Photon Arrivals')
plt.ylabel('Flux (photons/s)')
plt.xlim(tstart, tend)
plt.legend()
plt.title("Event Camera Simulation Results")

# log flux
plt.subplot(4, 1, 2)
plt.plot(t_eval, np.log(flux_function(t_eval)), 'm', lw=1, label='Log Flux $\ln(\lambda(t))$')
plt.ylabel('Log Flux')
plt.xlim(tstart, tend)
plt.legend()

# Voltage
plt.subplot(4, 1, 3)
plt.plot(t_eval, voltages, 'b', lw=1, label='Pixel Voltage $V(t)$')
event_t = [e[0] for e in events]
# positive events
positive_event_t = [e[0] for e in events if e[1] > 0]
negative_event_t = [e[0] for e in events if e[1] < 0]
plt.scatter(positive_event_t, voltages[np.searchsorted(t_eval, positive_event_t)], color='red', s=10, label='ON Events')
plt.scatter(negative_event_t, voltages[np.searchsorted(t_eval, negative_event_t)], color='green', s=10, label='OFF Events')
plt.ylabel('Voltage (V)')
plt.xlim(tstart, tend)
plt.legend()

# Events
plt.subplot(4, 1, 4)
if len(events) > 0:
    times, pols = zip(*events)
    times = np.array(times)
    pols = np.array(pols)
    if np.count_nonzero(pols > 0) > 0:
        plt.stem(times[pols > 0], pols[pols > 0], linefmt='r-', markerfmt='ro', basefmt=' ', label='+ events')
    if np.count_nonzero(pols < 0) > 0:
        plt.stem(times[pols < 0], pols[pols < 0], linefmt='g-', markerfmt='go', basefmt=' ', label='- events')
# plot log flux derivative for reference normalized between -1 and 1
plt.axhline(0, color='k', lw=0.5)
plt.ylim(-1.5, 1.5)
plt.ylabel('Event Polarity')
plt.xlim(tstart, tend)
plt.xlabel('Time (s)')
# plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
voltages.min(), voltages.max()

In [ ]:
sim.c_minus, sim.c_plus

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(10, 6))
hlam = h(t_eval, lam_func=flux_function, b=b, lambda_r=lambda_r) * flux_function(t_eval)
axs[0].plot(-a * voltages + hlam)
axs[0].set_title(r"$-aV(t) + h(t) \lambda(t)$")
axs[1].plot(hlam)
axs[1].set_title(r"$h(t) \lambda(t)$")

In [ ]:
import finufft
from scipy.stats import chi

ts = np.array([e[0] for e in events])
pols = np.array([e[1] for e in events])
cs = np.where(pols > 0, sim.c_plus, sim.c_minus)
texp = tend - tstart
nfreqs = 30
nmodes = 2 * nfreqs + 1
F = finufft.nufft1d1(ts * np.pi * 2, cs.astype("complex128"), n_modes=nmodes, eps=1e-6, isign=-1) / np.sqrt(texp)
F /= np.sqrt(texp)
sumc2 = np.sum(cs**2)
bound = chi.ppf(1 - 1 / nfreqs, 2) * sumc2 / (2 * texp)
cfar_mask = (np.abs(F) ** 2) >= bound
print(f"CFAR threshold: {bound:.4f}, kept {np.sum(cfar_mask)}/{len(F)} frequencies")
F_bd = F.copy()
F_bd[~cfar_mask] = 0.0
recon = finufft.nufft1d2(t_eval * np.pi * 2, F_bd, eps=1e-6, isign=1) / np.sqrt(texp)

volt_offset = (v_init - v_init * np.exp(-a * texp)) / texp
volt_mult = np.exp(a * (texp - t_eval))

fig, axs = plt.subplots(2, 1, figsize=(10, 6))
axs[0].plot(-a * voltages + hlam, label="GT")
axs[0].plot(recon.real, label="recon")
axs[0].legend()
axs[0].set_title("Reconstructed $-aV(t) + h(t) \lambda(t)$")
axs[1].plot(hlam, label="GT")
axs[1].plot(1 * (recon.real + volt_offset), label="recon")
axs[1].legend()
axs[1].set_title(r"Reconstructed $h(t) \lambda(t)$")
fig.tight_layout()

In [ ]:
plt.plot(np.abs(np.fft.fftshift(np.fft.fft(voltages))))
# log y scale
plt.yscale("log")

In [ ]:
plt.plot(np.abs(F))
# plt.plot(np.abs(F_bd))
plt.axhline(y=bound, color='r', linestyle='--', label='CFAR Threshold')

### voltage tests on videos

In [ ]:
from sptflux import ihpp_data, ihpp_sim, plotting
def fluxfunc(x, y, t):
    vals = 100000 + 10000 * np.sin(2 * np.pi * 4 * x) + 6000 * np.cos(2 * np.pi * 5.0 * t) + 2000 * np.sin(2 * np.pi * 7.0 * t + np.pi)
    return vals

xx = np.linspace(0, 1, 64)
yy = np.linspace(0, 1, 64)
tt = np.linspace(0, 1, 100)
TT, XX, YY = np.meshgrid(tt, xx, yy, indexing='ij')
pts = ihpp_sim.simulate_ihpp3d(fluxfunc)
print(f"Simulated {len(pts)} events.")
flux = fluxfunc(XX, YY, TT)
ihpp_data.to_video(flux, "output/synthetic/event/flux_video.avi", playback_fps=30)

In [ ]:
# Discretize points from 3D point process into per-pixel 1D time series
# pts has shape (N, 3) with columns [t, y, x] in [0,1] normalized coordinates
# Create a spatial grid and bin each point to its pixel

# Define spatial grid dimensions (matching xx, yy)
H = len(yy)
W = len(xx)

# Create pixel boundaries
x_edges = np.linspace(0, 1, W + 1)
y_edges = np.linspace(0, 1, H + 1)

# Initialize storage: list of lists, where per_pixel_times[i][j] contains timestamps for pixel (i,j)
per_pixel_times = [[[] for _ in range(W)] for _ in range(H)]

# Bin each point to its pixel and store the time coordinate
for point in pts:
    t_val, y_val, x_val = point
    
    # Find which pixel this point belongs to
    x_idx = np.searchsorted(x_edges, x_val, side='right') - 1
    y_idx = np.searchsorted(y_edges, y_val, side='right') - 1
    
    # Clamp to valid range (handles boundary cases)
    x_idx = np.clip(x_idx, 0, W - 1)
    y_idx = np.clip(y_idx, 0, H - 1)
    
    # Store the timestamp for this pixel
    per_pixel_times[y_idx][x_idx].append(t_val)

# Convert lists to sorted numpy arrays for easier processing
per_pixel_times = [[np.array(sorted(times)) for times in row] for row in per_pixel_times]

# Summary statistics
total_points = sum(len(times) for row in per_pixel_times for times in row)
non_empty_pixels = sum(1 for row in per_pixel_times for times in row if len(times) > 0)

print(f"Total points distributed: {total_points}")
print(f"Non-empty pixels: {non_empty_pixels}/{H*W}")
print(f"Average points per non-empty pixel: {total_points/non_empty_pixels:.2f}")

In [ ]:
# set up event simulation for each pixel

a = 10.0  # voltage decay rate (s^-1)
b = 2.0  # constant gain
lambda_r = 0.1  # reference flux
delta_lambda_p = 0.1  # mean brightness change threshold
sim = EventPixelSimulator(a=a, b=b, lambda_r=lambda_r, delta_lambda_p=delta_lambda_p)
voltages = [[[] for _ in range(W)] for _ in range(H)]

In [ ]:
import numpy as np

def make_moving_gaussian(amplitude, sigma, velocity, period):
    """
    Returns a function lambda(x, y, t) representing a Gaussian ball 
    moving back and forth along the x-axis.
    
    Args:
        amplitude (float): Peak intensity.
        sigma (float): Standard deviation (spread) of the Gaussian.
        velocity (float): (Unused in simple oscillator, derived from period/bounds) 
                          controls speed if we weren't fixed to [0,1].
                          Here, we use 'period' to dictate the full cycle duration.
        period (float): Time to complete one full back-and-forth cycle.
        
    Returns:
        function: lambda(x, y, t) -> intensity
    """
    def intensity_func(x, y, t):
        # Center x oscillation: moves between 0.2 and 0.8 to stay within [0,1] mostly
        # x_c(t) = center + oscillating_offset
        omega = 2 * np.pi / period
        # Oscillation centered at 0.5, amplitude 0.3 (range 0.2 to 0.8)
        x_c = 0.5 + 0.3 * np.sin(omega * t)
        y_c = 0.5 # Fixed y center
        
        # Gaussian formula
        exponent = -((x - x_c)**2 + (y - y_c)**2) / (2 * sigma**2)
        return amplitude * np.exp(exponent)
        
    return intensity_func

def make_flashing_gaussian(amplitude, sigma, period, fluc_intensity=0.5):
    """
    Returns a function lambda(x, y, t) representing a stationary Gaussian ball 
    that flashes (intensity modulates sinusoidally).
    
    Args:
        amplitude (float): Peak intensity.
        sigma (float): Standard deviation.
        period (float): Time for one full flash cycle.
        
    Returns:
        function: lambda(x, y, t) -> intensity
    """
    def intensity_func(x, y, t):
        x_c, y_c = 0.5, 0.5
        omega = 2 * np.pi / period
        
        # Modulation: oscillates between 0 and 1
        # (sin goes -1 to 1, +1 makes it 0 to 2, /2 makes it 0 to 1)
        modulation = (fluc_intensity * np.sin(omega * t)) + 0.5
        
        exponent = -((x - x_c)**2 + (y - y_c)**2) / (2 * sigma**2)
        return amplitude * modulation * np.exp(exponent)
        
    return intensity_func

def make_moving_circle(amplitude, radius, period):
    """
    Returns a function lambda(x, y, t) representing a solid circle (step function)
    moving back and forth along the x-axis.
    
    Args:
        amplitude (float): Intensity inside the circle.
        radius (float): Radius of the circle (0 to 0.5 typically).
        period (float): Time to complete one full back-and-forth cycle.
        
    Returns:
        function: lambda(x, y, t) -> intensity
    """
    def intensity_func(x, y, t):
        # Same oscillatory motion: Center moves between 0.2 and 0.8
        omega = 2 * np.pi / period
        x_c = 0.5 + 0.3 * np.cos(omega * t)
        y_c = 0.5
        
        # Calculate squared distance from the center
        dist_sq = (x - x_c)**2 + (y - y_c)**2
        
        # Binary selection: Amplitude if inside radius, 0 if outside
        # We use np.where to handle the vector inputs (meshgrids) correctly
        return np.where(dist_sq <= radius**2, amplitude, 0.0)
        
    return intensity_func

def make_flashing_circle(amplitude, radius, period):
    """
    Returns a function lambda(x, y, t) representing a solid circle (step function)
    that flashes (intensity modulates sinusoidally).
    
    Args:
        amplitude (float): Peak intensity inside the circle.
        radius (float): Radius of the circle (0 to 0.5 typically).
        period (float): Time for one full flash cycle.
        
    Returns:
        function: lambda(x, y, t) -> intensity
    """
    def intensity_func(x, y, t):
        x_c, y_c = 0.5, 0.5
        omega = 2 * np.pi / period
        
        # Modulation: oscillates between 0 and 1
        modulation = 0.5 * (1 + np.sin(omega * t))
        
        # Calculate squared distance from the center
        dist_sq = (x - x_c)**2 + (y - y_c)**2
        
        # Binary selection with modulation: amplitude * modulation if inside radius, 0 if outside
        return np.where(dist_sq <= radius**2, amplitude * modulation, 0.0)
        
    return intensity_func

def make_moving_flashing_circle(amplitude, radius, period_flash, period_move):
    """
    Returns a function lambda(x, y, t) representing a solid circle (step function)
    that moves back and forth and flashes.
    
    Args:
        amplitude (float): Peak intensity inside the circle.
        radius (float): Radius of the circle (0 to 0.5 typically).
        period_flash (float): Time for one full flash cycle.
        period_move (float): Time to complete one full back-and-forth movement cycle.
        
    Returns:
        function: lambda(x, y, t) -> intensity
    """
    def intensity_func(x, y, t):
        # Moving center
        omega_move = 2 * np.pi / period_move
        x_c = 0.5 + 0.3 * np.cos(omega_move * t)
        y_c = 0.5
        
        # Flashing modulation
        omega_flash = 2 * np.pi / period_flash
        modulation = 0.5 * (1 + np.sin(omega_flash * t))
        
        # Calculate squared distance from the center
        dist_sq = (x - x_c)**2 + (y - y_c)**2
        
        # Binary selection with modulation: amplitude * modulation if inside radius, 0 if outside
        return np.where(dist_sq <= radius**2, amplitude * modulation, 0.0)
        
    return intensity_func

def sample_lambda(func, T_steps, H, W):
    """Samples the continuous function onto a discrete grid."""
    # Define coordinate grids
    ts = np.linspace(0, 1, T_steps)
    ys = np.linspace(0, 1, H)
    xs = np.linspace(0, 1, W)
    
    # Create meshgrid for vectorization (broadcasting is cleaner/faster)
    # Using 'indexing="ij"' so the order remains (T, Y, X)
    tt, yy, xx = np.meshgrid(ts, ys, xs, indexing='ij')
    
    return func(xx, yy, tt)


from sptflux import ihpp_data
# --- Configuration ---
H, W = 180, 240       # Spatial Resolution
T_steps = 10000       # Temporal Resolution (frames over 1 second)
AMPLITUDE = 100.0    # Intensity peak
SIGMA = 0.1         # Gaussian spread
PERIOD_MOVE = 0.1        # seconds per cycle
PERIOD_FLASH = 0.3     # seconds per cycle
RADIUS = 0.1        # Size of the circle

# # --- 1. Moving Ball Simulation ---
# print("Simulating Moving Ball...")
# moving_lambda = make_moving_gaussian(AMPLITUDE, SIGMA, velocity=None, period=PERIOD)
# video_moving = sample_lambda(moving_lambda, T_steps, H, W)

# # --- 2. Flashing Ball Simulation ---
# print("\nSimulating Flashing Ball...")
# flashing_lambda = make_flashing_gaussian(AMPLITUDE, SIGMA, period=PERIOD)
# video_flashing = sample_lambda(flashing_lambda, T_steps, H, W)

# # 1. Create the function
# print("Simulating Moving Solid Circle...")
# circle_lambda = make_moving_circle(AMPLITUDE, RADIUS, PERIOD)
# circle_lam_flash = make_flashing_circle(AMPLITUDE, RADIUS, PERIOD)

# # 2. Sample to video grid (using the helper from the previous code)
# video_circle = sample_lambda(circle_lambda, T_steps=T_steps, H=H, W=W)
# video_circle_flash = sample_lambda(circle_lam_flash, T_steps=T_steps, H=H, W=W)

dataname = "moving_flashing_circle"
fluxfunc = make_moving_flashing_circle(AMPLITUDE, RADIUS, PERIOD_MOVE, PERIOD_FLASH)
video_fluxfunc = sample_lambda(fluxfunc, T_steps=T_steps, H=H, W=W)
video_fluxfunc += 10

texp = 10.0
ihpp_data.to_video(video_fluxfunc, f"data/event/{dataname}.mp4", cmap="gray", playback_fps=30, vmin=0)
paths = ihpp_data.to_video(video_fluxfunc, f"data/event/{dataname}", cmap="gray", vmin=0)
with open(f"data/event/{dataname}/info.txt", "w") as f:
    for i, p in enumerate(paths):
        time_us = int(texp * i / T_steps * 1e6)
        f.write(f"./data_samples/interp/{dataname}/{p.name} {str(time_us).zfill(12)}\n")

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import finufft
from scipy.stats import chi, chi2
from tqdm.auto import tqdm
from sptflux import flux_probing, ihpp_data, plotting, config

def events_to_voxel_grid(events, num_bins, width, height):
    """
    https://github.com/uzh-rpg/rpg_e2vid/blob/master/utils/inference_utils.py
    Build a voxel grid with bilinear interpolation in the time domain from a set of events.

    :param events: a [N x 4] NumPy array (np.int32) containing one event per row in the form:
        [timestamp(us), x, y, polarity(0 or 1)]
    :param num_bins: number of bins in the temporal axis of the voxel grid
    :param width, height: dimensions of the voxel grid
    """

    assert(events.shape[1] == 4)
    assert(num_bins > 0)
    assert(width > 0)
    assert(height > 0)

    voxel_grid = np.zeros((num_bins, height, width), np.float32).ravel()

    # normalize the event timestamps so that they lie between 0 and num_bins
    last_stamp = events[-1, 0]
    first_stamp = events[0, 0]
    deltaT = last_stamp - first_stamp

    if deltaT == 0:
        deltaT = 1.0

    ts = events[:, 0].astype(np.float32)
    ts = (num_bins - 1) * (ts - first_stamp) / deltaT
    xs = events[:, 1].astype(np.int32)
    ys = events[:, 2].astype(np.int32)
    pols = events[:, 3].astype(np.float32)
    pols[pols == 0] = -1  # polarity should be +1 / -1

    tis = ts.astype(np.int32)
    dts = ts - tis
    vals_left = pols * (1.0 - dts)
    vals_right = pols * dts

    valid_indices = tis < num_bins
    np.add.at(voxel_grid, xs[valid_indices] + ys[valid_indices] * width
              + tis[valid_indices] * width * height, vals_left[valid_indices])

    valid_indices = (tis + 1) < num_bins
    np.add.at(voxel_grid, xs[valid_indices] + ys[valid_indices] * width
              + (tis[valid_indices] + 1) * width * height, vals_right[valid_indices])

    voxel_grid = np.reshape(voxel_grid, (num_bins, height, width))

    return voxel_grid

def visual_voxel_grid(voxel_grid, output_folder, filename_key):
    for i in range(voxel_grid.shape[0]):
        path = os.path.join(output_folder, '%s_%02d.png' % (filename_key, i))
        gridmin = np.min(voxel_grid[i])
        gridmax = np.max(voxel_grid[i])
        absmax = max(abs(gridmin), abs(gridmax))
        normalize_im = voxel_grid[i] / absmax * 0.5 + 0.5
        # normalize_im = (voxel_grid[i] - np.min(voxel_grid[i])) / (np.max(voxel_grid[i]) - np.min(voxel_grid[i]))
        cv2.imwrite(path, (normalize_im * 255).astype(np.uint8))

def read_event_dir(event_dir):
    event_dir = Path(event_dir)
    with open(event_dir / 'info.txt', 'r') as f:
        video_info = f.readlines()
    file_timestamps_us = [int(info_i.split()[1]) for info_i in video_info]
    file_paths = [Path(info_i.split()[0]) for info_i in video_info]
    imgs = []
    for i, filepath in enumerate(file_paths):
        img = cv2.imread(event_dir / filepath.name, cv2.IMREAD_GRAYSCALE)
        imgs.append(img)
    return np.array(imgs), np.array(file_timestamps_us) / 1e6  # convert to seconds

def drop_hot_pixels_events(events_df, dominance_thresh=0.95, min_events=50):
    # Compute per-pixel event counts
    pixel_stats = (
        events_df
        .groupby(["x", "y"])["p"]
        .agg(
            total="count",
            pos=lambda s: (s == 1).sum(),
            neg=lambda s: (s == 0).sum()
        )
    )
    # Polarity dominance ratio
    pixel_stats["dominance"] = pixel_stats[["pos", "neg"]].max(axis=1) / pixel_stats["total"]
    
    hot_pixels = pixel_stats[
        (pixel_stats["dominance"] > dominance_thresh) &
        (pixel_stats["total"] >= min_events)
    ].index  # MultiIndex (x, y)
    
    print(f"Removing {len(hot_pixels)} hot pixels.")
    
    events_df_clean = events_df.set_index(["x", "y"]).drop(hot_pixels).reset_index()
    
    return events_df_clean

def read_event_dir_davis(event_dir):
    event_dir = Path(event_dir)
    img_dir = event_dir / "images"
    with open(event_dir / 'events.txt', 'r') as f:
        events_df = pd.read_csv(f, sep=" ", names=["t", "x", "y", "p"])

    with open(event_dir / 'images.txt', 'r') as f:
        video_info = f.readlines()
    file_timestamps_s = [float(info_i.split()[0]) for info_i in video_info]
    file_paths = [Path(info_i.split()[1]) for info_i in video_info]
    imgs = []
    for i, filepath in enumerate(file_paths):
        img = cv2.imread(img_dir / filepath.name, cv2.IMREAD_GRAYSCALE)
        imgs.append(img)
    return events_df, np.array(imgs), np.array(file_timestamps_s)

In [ ]:
import aedat

dataname = "shooting-rapid-eccv"
decoder = aedat.Decoder(config.DATA_DIR / f"event/{dataname}.aedat4")
print(decoder.id_to_stream())

events_df = pd.DataFrame(columns=["t", "x", "y", "p"])
timestamps = []
frames = []
event_ts = []
event_xs = []
event_ys = []
event_ps = []

for packet in tqdm(decoder):
    if "frame" in packet:
        timestamps.append(packet["frame"]["t"])
        frames.append(packet["frame"]["pixels"])
    if "events" in packet:
        event_ts.append(packet["events"]["t"])
        event_xs.append(packet["events"]["x"])
        event_ys.append(packet["events"]["y"])
        event_ps.append(packet["events"]["p"])
event_ts = np.concatenate(event_ts)
event_xs = np.concatenate(event_xs)
event_ys = np.concatenate(event_ys)
event_ps = np.concatenate(event_ps)
events_df = pd.DataFrame({
    "t": event_ts,
    "x": event_xs,
    "y": event_ys,
    "p": event_ps
})
frames = np.array(frames)
timestamps = np.array(timestamps)
T, H, W = frames.shape

t0_us = events_df["t"].min()
events_df["t"] = (events_df["t"] - t0_us) / 1e6
timestamps = (timestamps - t0_us) / 1e6

tstart, tend = 0.0, events_df["t"].max()
# shooting-rapid-eccv-2
# tstart, tend = 1.5, 5.0
# shooting-rapid-eccv
tstart, tend = 4.0, 7.5
events_df = events_df[(events_df["t"] >= tstart) & (events_df["t"] <= tend)]
aps_timestamp_mask = (timestamps >= tstart) & (timestamps <= tend)
timestamps = timestamps[aps_timestamp_mask]
frames = frames[aps_timestamp_mask]
texp = tend - tstart
events_df = drop_hot_pixels_events(events_df, dominance_thresh=0.95)
events_df

In [ ]:
from pathlib import Path
import pandas as pd
# DVXplorer csv data
dataname = "teaser6"
path = config.DATA_DIR / Path(f"event/{dataname}.csv")
events_df = pd.read_csv(path, skiprows=1, header=0, names=["t", "x", "y", "p"])
T, H, W = 690, 480, 640
with open(path, "r") as f:
    first = f.readline()
print(first)
# convert times from microseconds to seconds and set starting time to 0
events_df["t"] = (events_df["t"] - events_df["t"].min()) / 1e6
tstart, tend = 0, events_df["t"].max()
events_df.head()
print(f"total capture time: {events_df['t'].max()}")
# balloon1
# tstart, tend = 6.5, 7.5
# seed1
# tstart, tend = 0.0, 7.0
# tstart = 29.0
tend = 7.0
texp = tend - tstart
events_df = events_df[(events_df["t"] >= tstart) & (events_df["t"] <= tend)]
print(f"Total events after filtering: {len(events_df)}")
# drop hot pixels
events_df = drop_hot_pixels_events(events_df, dominance_thresh=0.95)

In [ ]:
dataname = "dynamic_6dof"
events_df, frames, timestamps = read_event_dir_davis(f"data/event/{dataname}")
events_df = drop_hot_pixels_events(events_df, dominance_thresh=0.95)

tstart, tend = timestamps[0], timestamps[-1]
texp = tend - tstart
T, H, W = frames.shape
outdir = Path(f"output/results/event/{dataname}")
outdir.mkdir(parents=True, exist_ok=True)
ihpp_data.to_video(frames, outdir / f"{dataname}-GT.mp4", cmap="gray", playback_fps=30)

In [ ]:
dataname = "moving_flashing_circle"
datapath = Path(f"data/event/{dataname}.txt")
events_df = pd.read_csv(datapath, sep=' ', names=['t', 'x', 'y', 'p'])
tstart, tend = 0.0, 10.0
texp = tend - tstart
T, H, W = 2000, 180, 240

# simulated data, starts at 0
events_df["t"] = (events_df["t"]) / 1e6
events_df

In [ ]:
# dataname = "gauss-moving"
# # fluxfunc = make_flashing_gaussian(AMPLITUDE, SIGMA, period=PERIOD, fluc_intensity=0.2)
# # fluxfunc = make_moving_circle(AMPLITUDE, RADIUS, period=PERIOD)
# fluxfunc = make_moving_gaussian(AMPLITUDE, SIGMA, velocity=None, period=PERIOD)
# video_fluxfunc = sample_lambda(fluxfunc, T_steps, H, W)
# tstart, tend = 0.0, 1.0

In [ ]:
# dataname = "office_zigzag"
# frames, timestamps = read_event_dir(f"data/event-synth/{dataname}")
# tstart, tend = timestamps[0], timestamps[-1]
# T, H, W = frames.shape

In [ ]:
# ihpp_data.to_video(video_fluxfunc, f"output/synthetic/event/{dataname}.avi", cmap="gray", playback_fps=30)
# paths = ihpp_data.to_video(video_fluxfunc, f"output/synthetic/event/{dataname}", cmap="gray", playback_fps=30)
# with open(f"output/synthetic/event/{dataname}/info.txt", "w") as f:
#     for i, p in enumerate(paths):
#         time_us = int(i / T_steps * 1e6)
#         f.write(f"./data_samples/interp/{dataname}/{p.name} {str(time_us).zfill(12)}\n")

In [ ]:
# DVS346
k1 = 0.00018 * 29250
k3 = 0.0001
kL = 1.0

# DVS240
# k1 = 0.000094 * 47065
# k3 = 0.0002
# kL = 1.0

# DVXplorer
# i have no idea what these parameters are for this camera
# just use DVS 240 xd

# calculated
A = k1 * kL
mL = (k3 / A) ** 2

c_plus = 1.0
c_minus = 1.0
ts = events_df["t"].to_numpy()
ts = (ts - tstart) / texp
x = events_df["x"].to_numpy()
y = events_df["y"].to_numpy()
pol = events_df["p"].to_numpy()
pol_delt = np.where(pol > 0, c_plus, -c_minus)

outdir = config.OUTPUT_DIR / Path(f"results/event/{dataname}")
outdir.mkdir(parents=True, exist_ok=True)

In [ ]:
# 1d probing

# import finufft
# from scipy.stats import chi

# k1 = 0.00018 * 29250
# k3 = 0.0001
# kL = 1.0
# A = k1 * kL
# mL = (k3 / A) ** 2

# tt = np.linspace(0, 1, 1000)
# step_der = np.gradient(np.cumsum(poldeltpos), tpos)
# texp = 1.0
# nfreqs = 10
# nmodes = 2 * nfreqs + 1
# F = finufft.nufft1d1(tpos * np.pi * 2, poldeltpos.astype("complex128"), n_modes=nmodes, eps=1e-6, isign=-1) / np.sqrt(texp)
# sumc2 = np.sum(poldeltpos**2)
# bound = chi.ppf(1 - 1 / nfreqs, 2) * sumc2 / (2 * texp)
# # bound = 0
# cfar_mask = (np.abs(F) ** 2) >= bound
# print(f"CFAR threshold: {bound:.4f}, kept {np.count_nonzero(cfar_mask)}/{len(F)} frequencies")
# F_bd = F.copy()
# F_bd[~cfar_mask] = 0.0
# freqs = np.fft.fftshift(np.fft.fftfreq(nmodes, d=1/nmodes))
# inv_i_omega = np.zeros_like(freqs, dtype=complex)
# # Create a mask to avoid division by zero at the DC component (freqs == 0)
# mask = freqs != 0
# # Apply the integration rule: 1 / (i * 2 * pi * f)
# inv_i_omega[mask] = 1.0 / (2j * np.pi * freqs[mask])
# F_bd_integrated = F_bd * inv_i_omega
# recon_der = finufft.nufft1d2(tt * np.pi * 2, F_bd, eps=1e-6, isign=1) / np.sqrt(texp)
# recon_integ = finufft.nufft1d2(tt * np.pi * 2, F_bd_integrated, eps=1e-6, isign=1) / np.sqrt(texp)
# # find tt closest to tpos[0]
# idx_closest = np.argmin(np.abs(tt - tpos[0]))
# recon_integ = recon_integ + (recon_integ[0] - recon_integ[idx_closest])

# fig, axs = plt.subplots(3, 1, figsize=(10, 6))
# axs[0].plot(tt, np.gradient(np.log(video_flashing[:, ypos, xpos]), tt), label="GT")
# axs[0].plot(tt, recon_der.real, label="probing")
# axs[0].plot(tpos, step_der, label="step")
# axs[0].set_ylim(min(np.min(recon_der.real), np.min(step_der)) - 5, max(np.max(recon_der.real), np.max(step_der)) + 5)
# axs[0].legend()
# axs[0].set_title("derivative")

# gt = video_flashing[:, ypos, xpos]
# log_gt = np.log(gt)
# axs[1].plot(tt, log_gt - log_gt[0], label="GT (offset)")
# axs[1].plot(tt, recon_integ.real, label="probing")
# axs[1].plot(tpos, np.cumsum(poldeltpos), label="step")
# axs[1].legend()
# axs[1].set_title("ln flux integrated")

# axs[2].plot(tt, gt, label="GT")
# axs[2].plot(tt, np.e ** (recon_integ.real / A + log_gt[0]), "-.", label="probing")
# axs[2].plot(tpos, np.e ** (np.cumsum(poldeltpos) / A + log_gt[0]), "--", label="step")
# axs[2].legend()
# axs[2].set_title("flux integrated (offset)")
# fig.tight_layout()

In [ ]:
# plt.plot(freqs, np.abs(F))
# # plt.plot(np.abs(F_bd))
# plt.axhline(y=bound**0.5, color='r', linestyle='--', label='CFAR Threshold')

---

In [ ]:
# fucked up STFT implementation
# don't use

import finufft
from scipy.stats import chi
from tqdm.auto import tqdm
from sptflux import flux_probing

nt, ny, nx = 2500, H, W
tt = np.linspace(0, 1, nt)
yy = np.linspace(0, 1, ny)
xx = np.linspace(0, 1, nx)
TT, YY, XX = np.meshgrid(tt, yy, xx, indexing='ij')
texp = tend - tstart
nfreqs = (nt // 2, ny // 2, nx // 2)
nmodes = (2 * nfreqs[0] + 1, 2 * nfreqs[1] + 1, 2 * nfreqs[2] + 1)
V = texp * H * W
patch_H, patch_W = 60, 60
hop_H, hop_W = 20, 20
V_chunk = texp * patch_H * patch_W
out = np.zeros((nt, ny, nx))
out_integ = np.zeros((nt, ny, nx))
wsum = np.zeros((nt, ny, nx))
pbar = tqdm(total=((H - patch_H) // hop_H + 1) * ((W - patch_W) // hop_W + 1))

nmodes_patch = (nt, patch_H, patch_W)  # full modes for time being
ft_patch = np.fft.fftshift(np.fft.fftfreq(nmodes_patch[0], d=1/nmodes_patch[0]))
fy_patch = np.fft.fftshift(np.fft.fftfreq(nmodes_patch[1], d=1/nmodes_patch[1]))
fx_patch = np.fft.fftshift(np.fft.fftfreq(nmodes_patch[2], d=1/nmodes_patch[2]))
ftgrid_patch, fygrid_patch, fxgrid_patch = np.meshgrid(ft_patch, fy_patch, fx_patch, indexing='ij')
inv_i_omega_patch = np.zeros_like(ftgrid_patch, dtype="complex128")
highpass_thresh = 5.0
mask = np.abs(ftgrid_patch) > highpass_thresh
inv_i_omega_patch[mask] = 1.0 / (2j * np.pi * ftgrid_patch[mask])

window_output = flux_probing.get_3d_window_volume(nmodes_patch, window="hann", window_t=False)

for h0 in range(0, H - patch_H + 1, hop_H):
    for w0 in range(0, W - patch_W + 1, hop_W):
        ev_patch = events_df[
            (events_df["y"] >= h0) & (events_df["y"] < h0 + patch_H) &
            (events_df["x"] >= w0) & (events_df["x"] < w0 + patch_W)
        ]
        t_patch = ev_patch["t"].to_numpy()
        t_patch = (t_patch - tstart) / (tend - tstart)
        y_patch = (ev_patch["y"].to_numpy() - h0) / patch_H
        x_patch = (ev_patch["x"].to_numpy() - w0) / patch_W
        pol_patch = ev_patch["p"].to_numpy()
        pol_patch_delt = np.where(pol_patch > 0, c_plus, -c_minus)
        window_input = flux_probing.get_3d_window(x_patch, y_patch, t_patch, window="hann", window_t=False)
        F = finufft.nufft3d1(
            t_patch * np.pi * 2, y_patch * np.pi * 2, x_patch * np.pi * 2,
            np.sqrt(window_input) * pol_patch_delt.astype("complex128"),
            n_modes=nmodes_patch, eps=1e-6, isign=-1
        ) / np.sqrt(V_chunk)
        sumc2 = np.sum((window_input * pol_patch_delt)**2)
        alpha = 0.01
        bound = chi2.ppf(1 - alpha, 2) * sumc2 / (2 * V_chunk)
        cfar_mask = (np.abs(F) ** 2) >= bound

        F_bd = F.copy()
        F_bd[~cfar_mask] = 0.0
        F_bd_integ = F_bd * inv_i_omega_patch
        recon_stft = np.fft.ifftn(np.fft.ifftshift(F_bd), axes=(0, 1, 2), norm="forward") / np.sqrt(V_chunk)
        recon_integ_stft = np.fft.ifftn(np.fft.ifftshift(F_bd_integ), axes=(0, 1, 2), norm="forward") / np.sqrt(V_chunk)
        out[:, h0:h0+patch_H, w0:w0+patch_W] += recon_stft.real * np.sqrt(window_output)
        out_integ[:, h0:h0+patch_H, w0:w0+patch_W] += recon_integ_stft.real * np.sqrt(window_output)
        wsum[:, h0:h0+patch_H, w0:w0+patch_W] += window_output
        pbar.update(1)
# out = out / (wsum + 1e-6)
recon_der_stft = out
recon_integ_stft = out_integ

In [ ]:
plt.imshow(recon_integ_stft[600])
plt.colorbar()

In [ ]:
ihpp_data.to_video(recon_der_stft, outdir / f"STFT-der.mp4", cmap="gray", playback_fps=30)
ihpp_data.to_video(recon_integ_stft, outdir / f"STFT-integ.mp4", cmap="gray", playback_fps=30)

---

In [ ]:
# shooting-rapid-eccv-2, shooting-rapid-eccv
nt, ny, nx = 3500, H, W
tt = np.linspace(0, 1, nt)
yy = np.linspace(0, 1, ny)
xx = np.linspace(0, 1, nx)
texp = tend - tstart
nmodes = (nt, ny, nx)
V = texp * H * W
F = finufft.nufft3d1(
    ts * np.pi * 2, y / H * np.pi * 2, x / W * np.pi * 2, pol_delt.astype("complex128"), n_modes=nmodes, eps=1e-6, isign=-1
) / np.sqrt(V)

In [ ]:
sumc2 = np.sum(pol_delt**2)
alpha = 0.001
bound = chi2.ppf(1 - alpha, 2) * sumc2 / (2 * V)
cfar_mask = (np.abs(F) ** 2) >= bound
print(f"CFAR threshold: {bound:.4f}, kept {np.count_nonzero(cfar_mask)}/{F.size} frequencies")
F_bd = F.copy()
F_bd[~cfar_mask] = 0.0
ft = np.fft.fftshift(np.fft.fftfreq(nmodes[0], d=1/nmodes[0])) * 2 * np.pi
fy = np.fft.fftshift(np.fft.fftfreq(nmodes[1], d=1/nmodes[1])) * 2 * np.pi
fx = np.fft.fftshift(np.fft.fftfreq(nmodes[2], d=1/nmodes[2])) * 2 * np.pi
ftgrid, fygrid, fxgrid = np.meshgrid(ft, fy, fx, indexing='ij')
F_bd[ftgrid == 0] = 0.0

# # integrator with hard highpass
highpass_thresh = 5.0
inv_i_omega = np.zeros_like(ftgrid, dtype="complex128")
# Create a mask to avoid division by zero at the DC component (freqs == 0)
mask = ftgrid != 0
# Apply the integration rule: 1 / (i * 2 * pi * f)
inv_i_omega[mask] = 1.0 / (2j * np.pi * ftgrid[mask])
inv_i_omega[np.abs(ftgrid) <= highpass_thresh] = 0.0

# leaky integrator 1
# leak = 10.0
# inv_i_omega = 1.0 / (leak + 2j * np.pi * ftgrid)

# some other kind of leaky integrator idk
# omega = 2 * np.pi * ftgrid
# cutoff = 2.0
# omega0 = 2 * np.pi * cutoff
# hp = (omega ** 2) / (omega ** 2 + omega0 ** 2)
# inv_i_omega = np.zeros_like(omega, dtype=np.complex128)
# mask = omega != 0
# inv_i_omega[mask] = (1.0 / (1j*omega[mask])) * hp[mask]

# reg1 = 1
# reg2 = 1
# inv_i_omega = (-1j * ftgrid) / (ftgrid ** 2 + reg1 * (fxgrid ** 2 + fygrid ** 2) + reg2)
# inv_i_omega[ftgrid == 0] = 0.0

F_bd_integrated = F_bd * inv_i_omega

# k = np.sqrt(fxgrid**2 + fygrid**2)
# kc = 2.0  # cutoff in same units as fxgrid/fygrid (tune)
# LP = np.exp(-0.5 * (k / kc)**2)     # spatial low-pass
# HP = 1.0 - LP                       # spatial high-pass
# a = 10  # sharpening strength (tune)
# H_sharp = 1.0 + a * HP                    # unsharp mask gain
# F_sharp = F_bd_integrated * H_sharp

recon_der = np.fft.ifftn(np.fft.ifftshift(F_bd), axes=(0, 1, 2), norm="forward") / np.sqrt(V)
recon_der = recon_der.real
recon_integ = np.fft.ifftn(np.fft.ifftshift(F_bd_integrated), axes=(0, 1, 2), norm="forward") / np.sqrt(V)
recon_integ = recon_integ.real
# recon_integ_sharp = np.fft.ifftn(np.fft.ifftshift(F_sharp), axes=(0, 1, 2), norm="forward") / np.sqrt(V)
# recon_integ_sharp = recon_integ_sharp.real
recon_integ_exp = np.exp(recon_integ / A)

del inv_i_omega, mask, cfar_mask, ftgrid, fygrid, fxgrid

In [ ]:
ihpp_data.to_video(recon_der, outdir / f"recon_der.mp4", cmap="gray", playback_fps=30)
# ihpp_data.to_video(recon_integ_sharp, outdir / f"recon_integ_sharp.mp4", cmap="gray", playback_fps=30)
ihpp_data.to_video(recon_integ, outdir / f"recon_integ.mp4", cmap="gray", playback_fps=30)
ihpp_data.to_video(recon_integ_exp, outdir / f"recon_integ_exp.mp4", cmap="gray", playback_fps=30)

In [ ]:
ihpp_data.to_video(np.exp(recon_integ * 2), outdir / f"recon_integ_exp-scale.mp4", cmap="gray", playback_fps=30)

In [ ]:
events_arr = np.array(events_df[['t', 'x', 'y', 'p']])
events_voxel = events_to_voxel_grid(events_arr, num_bins=nt*5, width=W, height=H)
abshighest = np.max(np.abs(events_voxel))
# abshighest = 1.0
ihpp_data.to_video(events_voxel, outdir / "events.mp4", cmap="bwr", playback_fps=30, vmin=-abshighest, vmax=abshighest)
events_integ_step = np.cumsum(events_voxel, axis=0)
ihpp_data.to_video(events_integ_step, outdir / f"step_integ.mp4", playback_fps=30, cmap="gray", use_quantile=True)
# ihpp_data.to_video(np.exp(events_integ_step / A), outdir / f"step_integ_exp.mp4", playback_fps=30, cmap="gray", use_quantile=True)

In [ ]:
events_integ_step = np.cumsum(events_voxel[6800:], axis=0)
ihpp_data.to_video(events_integ_step, outdir / f"step_integ.mp4", playback_fps=30, cmap="gray", use_quantile=True)

---

# motion magnification???

In [ ]:
import numpy as np
from sptflux import ihpp_data, plotting

def make_moving_circle(amplitude, radius, period, sweep_dist):
    """
    Returns a function lambda(x, y, t) representing a solid circle (step function)
    moving back and forth along the x-axis.
    
    Args:
        amplitude (float): Intensity inside the circle.
        radius (float): Radius of the circle (0 to 0.5 typically).
        period (float): Time to complete one full back-and-forth cycle.
        sweep_dist (float): Distance over which the circle moves back and forth [0, 1].
    Returns:
        function: lambda(x, y, t) -> intensity
    """
    def intensity_func(x, y, t):
        # Same oscillatory motion: Center moves between 0.2 and 0.8
        omega = 2 * np.pi / period
        x_c = 0.5 + sweep_dist * np.sin(omega * t)
        y_c = 0.5
        
        # Calculate squared distance from the center
        dist_sq = (x - x_c)**2 + (y - y_c)**2
        
        # Binary selection: Amplitude if inside radius, 0 if outside
        # We use np.where to handle the vector inputs (meshgrids) correctly
        return np.where(dist_sq <= radius**2, amplitude, 0.0)

    return intensity_func

def make_moving_rect(amplitude, width, height, period, sweep_dist):
    """
    Returns a function lambda(x, y, t) representing a solid rectangle
    moving back and forth along the x-axis.
    
    Args:
        amplitude (float): Intensity inside the rectangle.
        width (float): Width of the rectangle.
        height (float): Height of the rectangle.
        period (float): Time to complete one full back-and-forth cycle.
        sweep_dist (float): Distance over which the rectangle moves back and forth [0, 1].
    Returns:
        function: lambda(x, y, t) -> intensity
    """
    def intensity_func(x, y, t):
        # Oscillatory motion: Center moves between 0.2 and 0.8
        omega = 2 * np.pi / period
        x_c = 0.5 + sweep_dist * np.sin(omega * t)
        y_c = 0.5
        
        # Rectangle boundaries
        x_min = x_c - width / 2
        x_max = x_c + width / 2
        y_min = y_c - height / 2
        y_max = y_c + height / 2
        
        # Binary selection: Amplitude if inside rectangle, 0 if outside
        return np.where((x >= x_min) & (x <= x_max) & (y >= y_min) & (y <= y_max), amplitude, 0.0)

    return intensity_func

In [ ]:
circle_close = make_moving_circle(50, 0.1, 0.5, 0.1)
circle_far = make_moving_circle(50, 0.1, 0.5, 0.3)
tt = np.linspace(0, 1, 50)
xx = np.linspace(0, 1, 32)
yy = np.linspace(0, 1, 32)
tgrid, ygrid, xgrid = np.meshgrid(tt, yy, xx, indexing="ij")
video_close = circle_close(xgrid, ygrid, tgrid)
video_far = circle_far(xgrid, ygrid, tgrid)

In [ ]:
ihpp_data.to_video(video_close, "output/synthetic/magnify/circle-moving-close.avi", playback_fps=30, cmap="gray")
ihpp_data.to_video(video_far, "output/synthetic/magnify/circle-moving-far.avi", playback_fps=30, cmap="gray")

In [ ]:
F_close = np.fft.rfftn(video_close.T, axes=(0, 1, 2)).T
F_far = np.fft.rfftn(video_far.T, axes=(0, 1, 2)).T
fx = np.fft.fftfreq(video_close.shape[2], d=1/video_close.shape[2])
fy = np.fft.fftfreq(video_close.shape[1], d=1/video_close.shape[1])
ft = np.fft.rfftfreq(video_close.shape[0], d=1/video_close.shape[0])

In [ ]:
plotting.fig_powerspec(fx, fy, ft, F_close.T, maxpts=None)

In [ ]:
plotting.fig_powerspec(fx, fy, ft, F_far.T, maxpts=None)

In [ ]:
plotting.fig_powerspec(fx, fy, ft, F_far.T, maxpts=None, func=np.angle)

In [ ]:
plotting.fig_powerspec(fx, fy, ft, F_close.T, maxpts=None, func=np.angle)

In [ ]:
# Create new frequency array with temporal frequencies scaled by alpha
# This effectively "speeds up" or "slows down" the temporal changes
F_close_magnified = np.zeros_like(F_close, dtype=np.complex128)
alpha = 1.1
T, H, W = F_close.shape

for t in range(T):
    # Calculate the source index in the temporal dimension
    t_source = t / alpha
    
    # Use linear interpolation for non-integer indices
    t_low = int(np.floor(t_source))
    t_high = int(np.ceil(t_source))
    
    # Handle boundary conditions
    if t_high >= T:
        t_high = T - 1
        t_low = T - 1
    
    # Interpolation weight
    weight = t_source - t_low
    
    # Interpolate between adjacent temporal slices
    F_close_magnified[t, :, :] = (1 - weight) * F_close[t_low, :, :] + weight * F_close[t_high, :, :]
print(f"Created magnified frequency array with shape {F_close_magnified.shape}")

In [ ]:
plotting.fig_powerspec(fx, fy, ft, F_close_magnified.T, maxpts=None)

In [ ]:
magnified = np.fft.irfftn(F_close_magnified.T, s=video_close.T.shape, axes=(0, 1, 2)).T
ihpp_data.to_video(magnified, "output/synthetic/magnify/magnified-circle.avi", playback_fps=30, cmap="gray")

In [ ]:
rect_close = make_moving_rect(50, 0.1, 1.0, 0.5, 0.1)
rect_far = make_moving_rect(50, 0.1, 1.0, 0.5, 0.3)
tt = np.linspace(0, 1, 50)
xx = np.linspace(0, 1, 32)
yy = np.linspace(0, 1, 32)
tgrid, ygrid, xgrid = np.meshgrid(tt, yy, xx, indexing="ij")
video_close = rect_close(xgrid, ygrid, tgrid)
video_close[:, 5:10, 5:10] += 20
video_close[:, 16:20, 10:20] += 20
video_far = rect_far(xgrid, ygrid, tgrid)
ihpp_data.to_video(video_close, "output/synthetic/magnify/rect-moving-close.avi", playback_fps=30, cmap="gray")
ihpp_data.to_video(video_far, "output/synthetic/magnify/rect-moving-far.avi", playback_fps=30, cmap="gray")

In [ ]:
video_close.shape

In [ ]:
F_close = np.fft.rfftn(video_close.T, axes=(0, 1, 2)).T
F_far = np.fft.rfftn(video_far.T, axes=(0, 1, 2)).T
fx = np.fft.fftfreq(video_close.shape[2], d=1/video_close.shape[2])
fy = np.fft.fftfreq(video_close.shape[1], d=1/video_close.shape[1])
ft = np.fft.rfftfreq(video_close.shape[0], d=1/video_close.shape[0])

In [ ]:
plotting.fig_powerspec(fx, fy, ft, F_close.T, maxpts=None)

In [ ]:
import cv2
import numpy as np

video_path = "output/synthetic/magnify/rect-moving-close.avi"
cap = cv2.VideoCapture(video_path)
ret, first_frame = cap.read()

# Convert to grayscale and float for FFT
gray_ref = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY).astype(np.float32)

# 1. Compute Fourier Transform of reference frame [Eq. 10]
fft_ref = np.fft.fft2(gray_ref)
phase_ref = np.angle(fft_ref)
alpha = 2

frames = []
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    gray_curr = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32)
    
    # 2. Compute FFT of current frame [Eq. 11]
    fft_curr = np.fft.fft2(gray_curr)
    mag_curr = np.abs(fft_curr)
    phase_curr = np.angle(fft_curr)

    # 3. Calculate Phase Difference: delta_phi = omega * delta(t) [Eq. 12]
    # We unwrap to prevent phase-wrapping artifacts [6]
    phase_diff = phase_curr - phase_ref
    
    # 4. Amplify Phase Difference [Eq. 13]
    amplified_phase = phase_curr + (alpha * phase_diff)

    # 5. Reconstruct the Signal
    # Combine original amplitude with amplified phase
    fft_magnified = mag_curr * np.exp(1j * amplified_phase)
    
    # Inverse FFT to return to spatial domain [3]
    magnified_frame = np.real(np.fft.ifft2(fft_magnified))
    
    # Normalize and display
    magnified_frame = np.clip(magnified_frame, 0, 255).astype(np.uint8)
    frames.append(magnified_frame)

cap.release()

In [ ]:
frames = np.array(frames)

In [ ]:
ihpp_data.to_video(frames, "output/synthetic/magnify/maybe-rect.avi", playback_fps=30, cmap="gray")

In [ ]:
import cv2
import numpy as np
from pathlib import Path
import zarr

flux_path = Path("output/results/quanta/guitar/thin1.000/fps1000/guitar-keep1.000-playfps1000-chunks5000x16x16-hops2500x8x8-res512x512-bins1000x512x512-sliceall-window_hann-cfar1.0/reconinfo.zarr")
stuff = zarr.open_group(flux_path, mode="r")
# very small part of top string
flux = stuff["flux"][:, 169:169+35, 396:396+101]

ref = flux[5]

# 1. Compute Fourier Transform of reference frame [Eq. 10]
fft_ref = np.fft.fft2(ref)
phase_ref = np.angle(fft_ref)
alpha = 2

frames = []
for i in range(0, len(flux)):
    curr = flux[i]
    
    # 2. Compute FFT of current frame [Eq. 11]
    fft_curr = np.fft.fft2(curr)
    mag_curr = np.abs(fft_curr)
    phase_curr = np.angle(fft_curr)

    # 3. Calculate Phase Difference: delta_phi = omega * delta(t) [Eq. 12]
    # We unwrap to prevent phase-wrapping artifacts [6]
    phase_diff = np.unwrap(phase_curr - phase_ref)
    
    # 4. Amplify Phase Difference [Eq. 13]
    amplified_phase = phase_curr + (alpha * phase_diff)

    # 5. Reconstruct the Signal
    # Combine original amplitude with amplified phase
    fft_magnified = mag_curr * np.exp(1j * amplified_phase)
    
    # Inverse FFT to return to spatial domain [3]
    magnified_frame = np.real(np.fft.ifft2(fft_magnified))
    
    # Normalize and display
    frames.append(magnified_frame)
frames = np.array(frames)
ihpp_data.to_video(flux, "output/synthetic/magnify/magnified-original-flux.avi", playback_fps=30, cmap="gray")
ihpp_data.to_video(flux ** (1/2.2), "output/synthetic/magnify/magnified-original-gamma.avi", playback_fps=30, cmap="gray")
ihpp_data.to_video(frames, "output/synthetic/magnify/magnified-volume.avi", playback_fps=30, cmap="gray")
# concatenate flux and frames
ihpp_data.to_video(
    np.concatenate([flux, frames], axis=1),
    "output/synthetic/magnify/magnified-combined.avi",
    playback_fps=30,
    cmap="gray"
)

In [ ]:
from tqdm.auto import tqdm

In [ ]:
import numpy as np
import pyrtools as pt

alpha = 2.0
levels = 2
orientations = 4

phases = []
mags = []

for frame in flux:
    pyr = pt.pyramids.SteerablePyramidFreq(
        frame, height=levels, order=orientations-1
    )
    phases.append({k: np.angle(v) for k, v in pyr.pyr_coeffs.items() if v.ndim == 2})
    mags.append({k: np.abs(v) for k, v in pyr.pyr_coeffs.items() if v.ndim == 2})
from scipy.signal import butter, filtfilt

def temporal_bandpass(x, fs, f1, f2, order=2):
    b, a = butter(order, [f1/(fs/2), f2/(fs/2)], btype='band')
    return filtfilt(b, a, x, axis=0)
filtered_phases = []

for band in phases[0].keys():
    stack = np.stack([p[band] for p in phases], axis=0)
    stack_f = temporal_bandpass(stack, fs=1000, f1=50, f2=300)
    filtered_phases.append((band, stack_f))
frames_out = []

for t in range(len(flux)):
    pyr = pt.pyramids.SteerablePyramidFreq(
        flux[t], height=levels, order=orientations-1
    )

    for band, phase_f in filtered_phases:
        mag = np.abs(pyr.pyr_coeffs[band])
        phase = np.angle(pyr.pyr_coeffs[band])

        pyr.pyr_coeffs[band] = mag * np.exp(1j * (phase + alpha * phase_f[t]))

    frames_out.append(np.real(pyr.recon_pyr()))
frames_out = np.array(frames_out)
ihpp_data.to_video(
    frames_out,
    "output/synthetic/magnify/magnified-pyrtools.avi",
    playback_fps=30,
    cmap="gray"
)

In [ ]:
def gabor_kernel(ksize, sigma, theta, lambd, gamma=1.0):
    y, x = np.mgrid[-ksize:ksize+1, -ksize:ksize+1]
    x_theta = x*np.cos(theta) + y*np.sin(theta)
    y_theta = -x*np.sin(theta) + y*np.cos(theta)

    gb = np.exp(-(x_theta**2 + gamma**2 * y_theta**2) / (2*sigma**2)) \
         * np.exp(1j * 2*np.pi * x_theta / lambd)
    return gb

thetas = np.linspace(0, np.pi, 4, endpoint=False)
lambdas = [6, 12, 24]   # spatial scales
sigma = 4

filters = []
for theta in thetas:
    for lambd in lambdas:
        filters.append(gabor_kernel(15, sigma, theta, lambd))

from scipy.signal import convolve2d

# reference responses
ref_responses = [convolve2d(ref, g, mode='same') for g in filters]
ref_phases = [np.angle(r) for r in ref_responses]

frames_out = []

for frame in tqdm(flux):
    responses = [convolve2d(frame, g, mode='same') for g in filters]
    out = np.zeros_like(frame, dtype=np.float64)

    for r, r_ref, g in zip(responses, ref_responses, filters):
        mag = np.abs(r)
        phase = np.angle(r)
        delta = np.unwrap(phase - np.angle(r_ref), axis=0)

        amplified = mag * np.exp(1j * (phase + alpha * delta))

        # reconstruction via adjoint (approximate)
        out += np.real(convolve2d(amplified, np.flip(np.conj(g)), mode='same'))

    frames_out.append(out)

frames_out = np.array(frames_out)
ihpp_data.to_video(
    np.concatenate([flux, frames_out], axis=1),
    "output/synthetic/magnify/magnified-combined-gabor.avi",
    playback_fps=30,
    cmap="gray"
)

In [ ]:
ihpp_data.to_video(
    np.concatenate([flux, frames], axis=1),
    "output/synthetic/magnify/magnified-combined.mp4",
    playback_fps=30,
    cmap="gray"
)

In [ ]:
F = np.fft.fftn(flux, axes=(0,1,2))  # (t, y, x)
F = np.fft.fftshift(F, axes=0)
T, H, W = flux.shape
freq_t = np.fft.fftfreq(T, d=1/1000)  # fps = 1000

mask_t = (np.abs(freq_t) > 50) & (np.abs(freq_t) < 300)
mask_t = mask_t[:, None, None]
real = np.real(F)
imag = np.imag(F)

F_mag = np.abs(F)
F_phase = np.angle(F)

F_phase_mag = F_phase + alpha * mask_t * F_phase
F_amp = F_mag * np.exp(1j * F_phase_mag)
out = np.real(np.fft.ifftn(F_amp, axes=(0,1,2)))
ihpp_data.to_video(
    out,
    "output/synthetic/magnify/fft3doogabooga.avi",
    playback_fps=30,
    cmap="gray"
)

In [ ]:
F.shape

In [ ]:
fx.shape, fy.shape

In [ ]:
F = np.fft.rfftn(flux.T, axes=(0, 1, 2)).T
fx = np.fft.fftfreq(flux.shape[2], d=1/flux.shape[2])
fy = np.fft.fftfreq(flux.shape[1], d=1/flux.shape[1])
ft = np.fft.rfftfreq(flux.shape[0], d=1/flux.shape[0])
plotting.fig_powerspec(fx, fy, ft[:10], F[:10].T, maxpts=None)

In [ ]:
F[:20, fy_mask, fx_mask].shape

In [ ]:
alpha = 5
# 1. Representation: Decompose the spatiotemporal volume into 3D sinusoids [2]
# We apply the FFT across time (axis 0), height (axis 1), and width (axis 2)
fft_volume = np.fft.fftn(video_close)

# 2. Phase Extraction: Position is represented by the phase of sinusoids [1, 4]
amplitude = np.abs(fft_volume)
phase = np.angle(fft_volume)

# 3. Phase Difference: In a global case, motion corresponds to a shift
# proportional to the translation delta(t) [2, 3]
# Here we isolate the temporal phase variations. 
# For a 3D block, we compare the phase of the volume to the phase of the 
# reference state (time 0).
reference_phase = phase[0, :, :]
phase_diff = phase - reference_phase

# 4. Amplification: Multiply the phase difference by alpha [3, 5]
# Equation 13: Amplified phase = phase + alpha * (phase_diff)
magnified_phase = phase + (alpha * phase_diff)

# 5. Reconstruction: Use amplified phase to shift basis functions [3, 6]
# Combine original amplitude with the new, exaggerated phase
fft_magnified = amplitude * np.exp(1j * magnified_phase)

# Inverse 3D FFT to return to the spatial domain
magnified_volume = np.real(np.fft.ifftn(fft_magnified))

magnified_vid = np.clip(magnified_volume, 0, 255).astype(np.uint8)

In [ ]:
ihpp_data.to_video(magnified_vid, "output/synthetic/magnify/magnified-volume.avi", playback_fps=30, cmap="gray")

# explosions????

In [ ]:
# https://github.com/openmsi/shimadzu/blob/main/hpv2tif.py

import numpy as np
import array
import sys
from PIL import Image
import tifffile

filename = "0.5cmx6cm-1"
# filename = "Still2"
inputFile = f"data/phantom/{filename}.dat"
outputFile = inputFile[:-3] + 'tif'
print('Converting file ' + inputFile)

# The .dat file is raw binary, so we read it in as byte string
with open(inputFile, 'rb') as fobj:
    raw_bytes = fobj.read()

# Find data tag for recording speed by searching. The .dat file format has tags that specify the beginning
# of each section. These tags are documented in the manual. We find the relevant section of the byte string
# by searching for the tags and set the offset accordingly. Then there's an additional offset (14 bytes in 
# this section) before the corresponding data begin. Finally, the variable nbytes tells us how many bytes of
# actual data there are.
offset = raw_bytes.find(b'\x30\x30\x07\x30')
offset += 14 # account for the ten bytes of metadata (see below)
nbytes = 4
byte_data=raw_bytes[offset:nbytes+offset]
print('Recording speed: ' + byte_data.decode('ascii') + ' ns')

# If you ever want to see the actual string of bytes, you can do this:
# print(''.join(['\\x%02x' % b for b in byte_data]))

# Find data tag for exposure time by searching
offset = raw_bytes.find(b'\x30\x30\x08\x30')
offset += 14 # account for the 14 bytes of metadata 
nbytes = 4
byte_data=raw_bytes[offset:nbytes+offset]
print('Exposure time: ' + byte_data.decode('ascii') + ' ns')

# Find data tag for time of recording
offset = raw_bytes.find(b'\x40\x40\x0e\x40')
offset += 14 # account for the fourteen bytes of metadata 
nbytes = 16
byte_data=raw_bytes[offset:nbytes+offset]
year = int.from_bytes(byte_data[0:2],byteorder = "little", signed=False)
month = int.from_bytes(byte_data[2:4],byteorder = "little", signed=False)
day = int.from_bytes(byte_data[6:8],byteorder = "little", signed=False)
hour = int.from_bytes( byte_data[8:10], byteorder = "little", signed=False )
minute = int.from_bytes( byte_data[10:12], byteorder = "little", signed=False )
second = int.from_bytes( byte_data[12:14], byteorder = "little", signed=False )
print('Date: ' + str(year) + '-' + str(month).zfill(2) + '-' + str(day).zfill(2) )
print('Time: ' + str(hour).zfill(2) + ':' + str(minute).zfill(2) + ':' + str(second).zfill(2) )

# Find data tag for image data
offset = raw_bytes.find(b'\xa0\xa0\x01\xa0')
offset += 14  # account for 14 bytes of metadata
# Next imageNum x 250 x 400 x 2 bytes are the actual values
W = 400
H = 250
img_bytes = W * H * 2
imageNum = int(len(raw_bytes[offset:]) / img_bytes)
print('Number of images: ' + str(imageNum))
nbytes = imageNum * img_bytes
byte_data = raw_bytes[offset:nbytes+offset]

In [ ]:
video = []
for i in range(0,imageNum):
    imagebytes=np.frombuffer(byte_data[i*img_bytes:img_bytes*(i+1)],dtype=np.uint16)
    # The data format in the .dat file is 16-bit integers, but that the CMOS chip in the camera
    # is only ten bits. For some reason, rather than simply writing out the raw values when saving the data,
    # the Shimadzu software left-shifts by 6 bits (equivalent to multiplying by 64). To make sure that
    # the converted TIFFs match the true, raw data, we undo that change by right-shifting the data by
    # 6 bits on the next line:
    imagebytes = np.reshape(imagebytes, shape=(H, W)) >> 6
    imagebytes = np.flipud(imagebytes) # flip vertically to match view in HPV viewer
    video.append(imagebytes)
video = np.array(video)
ihpp_data.to_video(video, f'data/phantom/{filename}.avi', playback_fps=10, cmap="gray")
ihpp_data.to_video(video, f'data/phantom/{filename}', playback_fps=10, cmap="gray")

# wavelets maybe???

## translation of original matlab wavelet multiscale implementation

In [ ]:
from sptflux.wavelets import statistical_thresholding

frames, _, _ = ihpp_data.read_quanta_zarr("data/quanta/gun-2s.zarr", load_data=False)
frames = frames[:50000]
all_timestamps = ihpp_data.timeseries_from_binframes(frames, 1.0)

In [ ]:
Nt = 1000
thresh_method = "lrt-g"
alpha = 0.05
j0 = 3
J = 9
tt = np.linspace(0, 1, Nt)
result = np.zeros((Nt, len(all_timestamps), len(all_timestamps[0])))
pbar = tqdm(total=len(all_timestamps) * len(all_timestamps[0]))
for i in range(len(all_timestamps)):
    for j in range(len(all_timestamps[i])):
        if len(all_timestamps[i][j]) == 0:
            pbar.update(1)
            continue
        _, reconflux = statistical_thresholding(all_timestamps[i][j], 1.0, thresh_method, alpha, j0, J, tt, False)
        result[:, i, j] = reconflux
        pbar.update(1)

In [ ]:
ihpp_data.to_video(
    result ** (1/2.2), f"output/wavelet/gun-2s-wavelet-recongamma-Nt{Nt}-thresh{thresh_method}-alpha{alpha}-j0{j0}-J{J}.avi", playback_fps=60, cmap="gray"
)

## vibe coding attempts

In [ ]:
import pywt

data, (t, y, x), _ = ihpp_data.read_quanta_zarr("data/quanta/gun-2s.zarr", load_data=False)

In [ ]:
arrivals = data[:, 100, 100]

In [ ]:
where = np.nonzero(arrivals)

In [ ]:
timestamps = where[0] / 100000

In [ ]:
timestamps

In [ ]:
import numpy as np
import pywt

def inverse_anscombe_exact(y):
    """
    Approximate unbiased inverse of the Anscombe transform.
    Based on Mäkitalo & Foi (2011). For binary data, it's fine to use
    this simple approximation.
    """
    return ((y / 2)**2) - 1/8

# =========================
# Example usage
# =========================
# Simulate a small binary photon data array
T, H, W = 1000, 64, 64
np.random.seed(0)

# True underlying flux (arbitrary smooth 3D function)
# spatial 2D Gaussian, constant over time
x = np.arange(W) - W/2
y = np.arange(H) - H/2
X, Y = np.meshgrid(x, y)
spatial = np.exp(- (X**2 + Y**2) / (2 * 5.0**2))
true_flux = np.tile(spatial[None, ...], (T, 1, 1))
true_flux /= true_flux.max()
true_flux *= 0.5  # max probability ~0.5 per bin

# Generate binary photon arrivals
data = np.random.rand(T,H,W) < true_flux
wavelet = "haar"
level = None
threshold_type = 'universal'

# Denoise using wavelet method
# 1) Anscombe variance-stabilizing transform
z = 2.0 * np.sqrt(data + 3/8.0)

# 2) 3D DWT
coeffs = pywt.wavedecn(z, wavelet=wavelet, level=level, mode='periodization')

# 3) Estimate noise sigma from finest detail coefficients
detail_coeffs = coeffs[-1]  # last level detail dict
all_detail_values = np.concatenate([v.ravel() for v in detail_coeffs.values()])
sigma = np.median(np.abs(all_detail_values)) / 0.6745  # MAD estimate

# 4) Compute threshold
N_total = z.size
if threshold_type == 'universal':
    tau = sigma * np.sqrt(2 * np.log(N_total))
elif threshold_type == 'sure':
    # For simplicity, we use universal threshold as proxy
    tau = sigma * np.sqrt(2 * np.log(N_total))
else:
    raise ValueError("threshold_type must be 'universal' or 'sure'")

tau = 0.9

# 5) Soft-threshold all detail coefficients
def soft_thresh(x, t):
    return np.sign(x) * np.maximum(np.abs(x) - t, 0.0)
def hard_thresh(x, t):
    return x * (np.abs(x) > t)

for lvl in range(1, len(coeffs)):
    for key in coeffs[lvl]:
        # coeffs[lvl][key] = soft_thresh(coeffs[lvl][key], tau)
        coeffs[lvl][key] = hard_thresh(coeffs[lvl][key], tau)

# 6) Inverse DWT
z_hat = pywt.waverecn(coeffs, wavelet=wavelet, mode='periodization')

# 7) Inverse Anscombe (approximate unbiased)
lambda_hat = inverse_anscombe_exact(z_hat)
lambda_hat[lambda_hat < 0] = 0.0  # enforce non-negativity

# Compare
import matplotlib.pyplot as plt
plt.figure(figsize=(12,5))
plt.subplot(1,3,1)
plt.imshow(data[T//2], cmap='gray')
plt.title("Raw binary data (t=T/2)")
plt.subplot(1,3,2)
plt.imshow(lambda_hat[T//2], cmap='hot')
plt.title("Denoised lambda_hat (t=T/2)")
plt.colorbar()
plt.subplot(1,3,3)
plt.imshow(true_flux[T//2], cmap='hot')
plt.title("True underlying flux (t=T/2)")
plt.colorbar()
plt.show()

In [ ]:
#!/usr/bin/env python3
"""
wavelet_denoise_binary_photons.py

Pipeline:
  binary photon arrivals (T x H x W) -->
  Anscombe VST -->
  separable orthonormal 3D DWT (pywt.wavedecn) -->
  soft-thresholding (VisuShrink) on detail coeffs -->
  inverse DWT -->
  approximate inverse Anscombe --> intensity estimate

Author: ChatGPT (prepared for you)
"""

import numpy as np
import pywt
import argparse
import sys


def anscombe_transform(y):
    """Anscombe variance-stabilizing transform for Poisson counts.
    y : array-like (counts, here 0/1)
    returns z = 2*sqrt(y + 3/8)
    """
    return 2.0 * np.sqrt(y + 3.0 / 8.0)


def inverse_anscombe_approx(z):
    """Approximate inverse Anscombe (simple closed-form).
    This is the common asymptotic inverse:
        lambda_hat ≈ (z/2)^2 - 1/8
    It's simple and fast but biased for very small counts.
    For better accuracy see Mäkitalo & Foi exact unbiased inverse (not implemented here).
    """
    lam = (z / 2.0) ** 2 - 1.0 / 8.0
    lam[lam < 0] = 0.0
    return lam


def estimate_sigma_from_coeffs(detail_coeff_dict):
    """Estimate noise sigma from a detail-coefficients dict (one level).
    detail_coeff_dict is a dict mapping e.g. ('a','d','d') -> array...
    We collect all arrays, flatten, and compute MAD-based sigma.
    """
    all_details = np.concatenate([np.ravel(v) for v in detail_coeff_dict.values()])
    # If empty (shouldn't happen), fallback:
    if all_details.size == 0:
        return 1.0
    med = np.median(np.abs(all_details))
    sigma = med / 0.6745
    # avoid zero sigma
    if sigma <= 0:
        sigma = np.std(all_details) if np.std(all_details) > 0 else 1.0
    return float(sigma)


def soft_threshold(coeff, thresh):
    """Soft-threshold a numpy array in-place (returns result)."""
    return pywt.threshold(coeff, value=thresh, mode="hard")


def denoise_binary_photons(y,
                           wavelet='db2',
                           mode='periodization',
                           level=None,
                           thresholding='visushrink'):
    """
    y : ndarray, shape (T, H, W), expected binary {0,1}
    Returns: lambda_hat (same shape): estimated intensity per bin

    Parameters:
      - wavelet: wavelet name for pywt, e.g. 'haar', 'db2'
      - mode: pywt signal extension mode
      - level: decomposition level (None -> choose max possible)
      - thresholding: 'visushrink' (universal). (SURE not implemented here.)
    """
    # Basic checks
    if y.ndim != 3:
        raise ValueError("Input array y must be 3D (T,H,W). Got shape: {}".format(y.shape))
    y = y.astype(np.float64)

    # 1) Anscombe transform
    z = anscombe_transform(y)

    # 2) choose level if None: conservative max level based on smallest dim
    if level is None:
        wave = pywt.Wavelet(wavelet)
        # discrete filter length
        filt_len = wave.dec_len
        max_levels_per_axis = [pywt.dwt_max_level(s, filt_len) for s in z.shape]
        level = min(max_levels_per_axis)
        # ensure at least 1
        level = max(1, level)

    # 3) 3D DWT (multilevel) via wavedecn (separable n-D)
    coeffs = pywt.wavedecn(z, wavelet=wavelet, mode=mode, level=level)
    # coeffs list: [cA_n, detail_n (dict), detail_{n-1} (dict), ..., detail_1 (dict)]

    # 4) Estimate sigma from the finest (detail_1) level -> that's coeffs[-1]
    finest_detail = coeffs[-1]
    sigma = estimate_sigma_from_coeffs(finest_detail)

    # Global number of samples for threshold scaling
    N = z.size

    # 5) Compute universal threshold (VisuShrink)
    if thresholding == 'visushrink':
        thresh = sigma * np.sqrt(2.0 * np.log(N))
        # Apply threshold to all detail levels (not approximation coeffs)
        coeffs_thresh = [coeffs[0]]  # keep approximation as-is
        for detail_dict in coeffs[1:]:
            new_detail = {}
            for key, arr in detail_dict.items():
                new_detail[key] = soft_threshold(arr, thresh)
            coeffs_thresh.append(new_detail)
    else:
        raise ValueError("Only 'visushrink' thresholding implemented in this script.")

    # 6) inverse DWT
    z_hat = pywt.waverecn(coeffs_thresh, wavelet=wavelet, mode=mode)

    # If inverse yields shape slightly different due to boundary handling, crop/pad:
    if z_hat.shape != z.shape:
        # prefer cropping to original shape
        z_hat = z_hat[tuple(slice(0, s) for s in y.shape)]

    # 7) Inverse Anscombe (approx)
    lam_hat = inverse_anscombe_approx(z_hat)

    return lam_hat, {
        'wavelet': wavelet,
        'level': level,
        'sigma_est': sigma,
        'threshold': thresh,
        'z': z,
        'z_hat': z_hat
    }


def example_synthetic_data(T=64, H=64, W=64, seed=1):
    """Generate a synthetic varying flux pattern, sample binary photon arrivals.
    The intensity lambda(t,x,y) will be in [0, mu_max] and we use Bernoulli for binary.
    (Alternatively you could sample Poisson counts and then clip to 0/1.)
    """
    rng = np.random.RandomState(seed)

    # Create a spatial spot + a moving temporal pulse to test multiscale recovery
    x = np.linspace(0, 1, W)
    y = np.linspace(0, 1, H)
    X, Y = np.meshgrid(x, y)
    spatial = np.exp(-((X - 0.5) ** 2 + (Y - 0.5) ** 2) / (2 * 0.08 ** 2))
    temporal = np.zeros(T)
    temporal[T // 4:T // 4 + 6] = 1.0
    temporal[T // 2:T // 2 + 10] = 0.6
    temporal = np.convolve(temporal, np.ones(3) / 3.0, mode='same')

    # Make intensity lambda(t,x,y) = temporal(t) * spatial(x,y) + background
    background = 0.01
    mu_max = 0.2  # max Bernoulli probability (choose <1 to remain binary)
    intensity = np.zeros((T, H, W), dtype=np.float64)
    for ti in range(T):
        intensity[ti] = background + mu_max * temporal[ti] * spatial

    # Sample binary arrivals per bin (Bernoulli)
    arrivals = rng.binomial(1, np.clip(intensity, 0.0, 1.0)).astype(np.uint8)
    return arrivals, intensity


def main_run_demo():
    print("Running demo with synthetic binary photon data...")
    y, lambda_true = example_synthetic_data(T=64, H=64, W=64, seed=2)

    lam_hat, meta = denoise_binary_photons(y, wavelet='haar', level=None)

    print("metadata:")
    for k, v in meta.items():
        if k in ('z', 'z_hat'):
            print(f"  {k}: shape {v.shape}")
        else:
            print(f"  {k}: {v}")

    # Simple diagnostics (global)
    mse = np.mean((lam_hat - lambda_true) ** 2)
    mae = np.mean(np.abs(lam_hat - lambda_true))
    print(f"Global MSE vs true intensity (for demo synthetic): {mse:.5e}")
    print(f"Global MAE vs true intensity (for demo synthetic): {mae:.5e}")

    # If run interactively you can inspect arrays; we will print a few stats:
    print("Estimated intensity min/median/max:", lam_hat.min(), np.median(lam_hat), lam_hat.max())
    print("True intensity min/median/max:", lambda_true.min(), np.median(lambda_true), lambda_true.max())

    # Optionally save outputs, but here just return
    return y, lambda_true, lam_hat, meta


y, lambda_true, lam_hat, meta = main_run_demo()

In [ ]:
plt.imshow(lambda_true[20])
plt.colorbar()

In [ ]:
plt.imshow(lam_hat[20])

In [ ]:
# plot various 1D Haar wavelets (scales & translations)
w = pywt.Wavelet('haar')
phi, psi, x = w.wavefun(level=8)  # scaling function, mother wavelet, grid

fig, axes = plt.subplots(2 + 4, 1, figsize=(8, 10), sharex=True)
axes[0].plot(x, phi, color='k')
axes[0].set_title("Scaling function (phi) - Haar")
axes[1].plot(x, psi, color='k')
axes[1].set_title("Mother wavelet (psi) - Haar")

levels = [0, 1, 2, 3]
for ii, j in enumerate(levels, start=2):
    ax = axes[ii]
    for k in range(min(4, 2**j)):  # show up to 4 translations per scale
        psi_jk = (2**(j/2)) * np.interp(2**j * x - k, x, psi, left=0.0, right=0.0)
        ax.plot(x, psi_jk, label=f"j={j}, k={k}")
    ax.set_ylabel(f"scale 2^{j}")
    ax.legend(loc='upper right', fontsize='small', ncol=2)

plt.xlabel("x")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from scipy.stats import chi2

def dyadic_counts(points, J, domain):
    """
    points : (N, d) array in [0,1]^d
    J      : resolution level
    domain : dimensionality d

    Returns:
        counts: array of shape (2^J,)*d
    """
    bins = 2 ** J
    idx = np.floor(points * bins).astype(int)
    idx = np.clip(idx, 0, bins - 1)

    counts = np.zeros((bins,) * domain, dtype=int)
    for i in idx:
        counts[tuple(i)] += 1
    return counts

def haar_intensity_1d(event_times, J, T=1.0):
    """
    event_times: array of times in [0,T]
    """
    points = (event_times / T).reshape(-1, 1)
    counts = dyadic_counts(points, J, domain=1).squeeze()

    bin_width = T / (2 ** J)
    intensity = counts / bin_width
    return intensity

def haar_threshold_1d(counts, alpha=0.01):
    """
    LRT for equality of two child Poisson means
    """
    thresholded = counts.copy()

    for j in range(int(np.log2(len(counts))) - 1, -1, -1):
        step = 2 ** (j + 1)
        for k in range(0, len(counts), step):
            c1 = counts[k:k + step//2].sum()
            c2 = counts[k + step//2:k + step].sum()

            if c1 + c2 == 0:
                continue

            llr = 2 * (
                c1 * np.log(c1 / ((c1 + c2)/2 + 1e-12) + 1e-12) +
                c2 * np.log(c2 / ((c1 + c2)/2 + 1e-12) + 1e-12)
            )

            if llr < chi2.ppf(1 - alpha, df=1):
                thresholded[k:k + step] = (c1 + c2) / step

    return thresholded

np.random.seed(0)
N = 2000
t = np.sort(np.random.rand(N))

J = 6
raw = haar_intensity_1d(t, J)
smooth = haar_threshold_1d(raw)

In [ ]:
def lam1d(t):
    return 5000 * np.cos(2 * np.pi * 4 * t) + 3000 * np.cos(2 * np.pi * 3 * t) + 8000
t = np.linspace(0, 1, 1024)
intensities = lam1d(t)
plt.plot(intensities, label='true')
pts = ihpp_sim.simulate_ihpp1d(lam1d, T=1, max_rate=16000)

In [ ]:
J = 6
raw = haar_intensity_1d(pts, J)
smooth = haar_threshold_1d(raw, 0.01)
plt.plot(raw, label='raw')
plt.plot(smooth, label='smooth')

In [ ]:
J = 13
raw = haar_intensity_1d(timestamps, J)
smooth = haar_threshold_1d(raw, 0.01)
plt.plot(raw, label='raw')
plt.plot(smooth, label='smooth', ls="--")
plt.legend()
raw.shape

In [ ]:
F = np.fft.rfft(arrivals)
bd = flux_probing.compute_amplitude_bound(np.count_nonzero(arrivals), V=1, nfreqs=F.shape[0]) * 2
F[np.abs(F) < bd] = 0
recon = np.fft.irfft(F, n=8192, norm="forward").real
plt.plot(recon)

In [ ]:
def haar_intensity_2d(points, J):
    """
    points: (N,2) in [0,1]^2
    """
    counts = dyadic_counts(points, J, domain=2)
    cell_area = (1 / (2 ** J)) ** 2
    return counts / cell_area

def haar_threshold_2d(counts, alpha=0.01):
    out = counts.copy()
    J = int(np.log2(counts.shape[0]))

    for j in range(J - 1, -1, -1):
        step = 2 ** (j + 1)
        for i in range(0, counts.shape[0], step):
            for k in range(0, counts.shape[1], step):
                block = counts[i:i+step, k:k+step]
                c = [
                    block[:step//2, :step//2].sum(),
                    block[:step//2, step//2:].sum(),
                    block[step//2:, :step//2].sum(),
                    block[step//2:, step//2:].sum()
                ]

                if sum(c) == 0:
                    continue

                mean = sum(c) / 4
                llr = 2 * sum(ci * np.log(ci / (mean + 1e-12) + 1e-12) for ci in c)

                if llr < chi2.ppf(1 - alpha, df=3):
                    out[i:i+step, k:k+step] = mean / (step ** 2)

    return out
points = np.random.rand(5000, 2)
J = 5

raw = haar_intensity_2d(points, J)
smooth = haar_threshold_2d(raw)

In [ ]:
def lam2d(x, y):
    return (5000 * np.exp(-((x - 0.4) ** 2 + (y - 0.4) ** 2) / (2 * 0.2 ** 2)) +
            3000 * np.exp(-((x - 0.7) ** 2 + (y - 0.7) ** 2) / (2 * 0.18 ** 2)))
x = np.linspace(0, 1, 64)
y = np.linspace(0, 1, 64)
X, Y = np.meshgrid(x, y)
intensities = lam2d(X, Y)
plt.imshow(intensities, origin='lower')
plt.colorbar()
pts = ihpp_sim.simulate_ihpp2d(lam2d, lmax=60000)

In [ ]:
plt.scatter(pts[:,0], pts[:,1], s=1)
plt.gca().set_aspect('equal', adjustable='box')

In [ ]:
J = 4
raw = haar_intensity_2d(pts, J)
print(raw.shape)
smooth = haar_threshold_2d(raw)
plt.imshow(raw, origin='lower')
plt.colorbar()
plt.show()
plt.imshow(smooth, origin='lower')
plt.colorbar()
plt.show()

In [ ]:
import numpy as np
import pywt
import matplotlib.pyplot as plt

def simulate_ppp_1d(intensity_fn, T=1.0, grid_size=4096):
    x = np.linspace(0, T, grid_size)
    lam = intensity_fn(x)
    lam_max = lam.max()

    N = np.random.poisson(lam_max * T)
    pts = np.random.rand(N) * T
    keep = np.random.rand(N) < intensity_fn(pts) / lam_max
    return pts[keep]

true_lambda = lambda x: 3000 * np.sin(4 * np.pi * x) + 3000
points = simulate_ppp_1d(true_lambda)

def bin_points_1d(points, J):
    n = 2**J
    bins = np.zeros(n)
    idx = np.floor(points * n).astype(int)
    idx = idx[idx < n]
    for i in idx:
        bins[i] += 1
    return bins

def poisson_wavelet_denoise_1d(counts, wavelet="haar"):
    coeffs = pywt.wavedec(counts, wavelet, mode="periodization")

    denoised = [coeffs[0]]  # keep scaling coeff untouched

    for j, c in enumerate(coeffs[1:], start=1):
        # Poisson variance estimate
        sigma = np.sqrt(np.maximum(np.mean(c), 1e-6))
        tau = np.sqrt(2 * np.log(len(c))) * sigma
        
        c_thr = np.where(np.abs(c) > tau, c, 0.0)
        denoised.append(c_thr)
        
    recon = pywt.waverec(denoised, wavelet, mode="periodization")
    return np.maximum(recon, 0)

J = 10
counts = bin_points_1d(points, J)
lam_hat = poisson_wavelet_denoise_1d(counts)

x = np.linspace(0, 1, len(lam_hat))
plt.plot(x, true_lambda(x), label="True λ")
plt.plot(x, lam_hat, label="Estimated λ")
plt.legend()
plt.show()

In [ ]:
import numpy as np

# KDE via Gaussian-smoothing of binned events (uses pts, x, y, ndi available in notebook)
import matplotlib.pyplot as plt

# bin points on the existing x,y grid
x_edges = np.linspace(0.0, 1.0, len(x) + 1)
y_edges = np.linspace(0.0, 1.0, len(y) + 1)
# histogram2d expects (x_coords, y_coords) or (cols, rows) ordering;
# we want H[row, col] with row ~ y, col ~ x -> pass pts[:,1] as x for rows
H_counts, _, _ = np.histogram2d(pts[:, 1], pts[:, 0], bins=[y_edges, x_edges])

# convert to density per unit area
dx = x_edges[1] - x_edges[0]
dy = y_edges[1] - y_edges[0]
bin_area = dx * dy
density = H_counts / bin_area  # events per unit area

# automatic bandwidth (Scott rule like for 2D): factor = n^{-1/(d+4)} with d=2 -> n^{-1/6}
n = max(1, len(pts))
factor = n ** (-1.0 / 6.0)
stds = np.std(pts, axis=0)
bandwidth = max(1e-3, stds.mean() * factor)  # in unit coordinates

# map bandwidth to grid-space sigma (pixels)
sigma_y = bandwidth / dy
sigma_x = bandwidth / dx

# Gaussian smoothing
from scipy import ndimage as ndi  # available in notebook, re-import safe
density_kde = ndi.gaussian_filter(density, sigma=(sigma_y, sigma_x), mode='reflect')

# expose result and visualize
intensity_kde = density_kde  # intensity estimate on grid: shape (len(y), len(x))

plt.figure(figsize=(6,5))
plt.imshow(intensity_kde, origin='lower', extent=[0,1,0,1], cmap='magma')
plt.colorbar(label='events / unit area')
plt.title(f'KDE intensity (bandwidth ~ {bandwidth:.4f})')
plt.xlabel('x'); plt.ylabel('y')
plt.show()

# point process fft error vs direct fft

In [ ]:
# continuous 2D function on [0,1]x[0,1]: sum of 4 Gaussians with varying amplitudes and sigmas
def get_fluxfunc(scale=1.0):
    # parameters: (cx, cy, amplitude, sigma)
    gs = [
        (0.25, 0.25, 0.95, 0.08),
        (0.75, 0.25, 1.0, 0.2),
        (0.4, 0.5, 0.8, 0.12),
        (0.75, 0.75, 0.9, 0.4),
    ]
    # add filled circular region centered near previous square center
    circle_cx, circle_cy = 0.30, 0.70
    circle_r = 0.15
    circle_amp = 1.0
    # add a diagonal line (slope = 1) with given thickness and amplitude
    line_thickness = 0.1  # in normalized coordinates
    line_amplitude = 0.6
    def fluxfunc(x, y):
        """
        x, y : scalar or numpy array (broadcastable)
        returns : same-shape array of sum of 4 Gaussians evaluated at (x,y)
        """
        x = np.asarray(x)
        y = np.asarray(y)
        out = np.zeros_like(x, dtype=float)
        for cx, cy, A, sigma in gs:
            dx2 = (x - cx) ** 2
            dy2 = (y - cy) ** 2
            out += A * np.exp(-0.5 * (dx2 + dy2) / (sigma ** 2))
        # add filled circular region centered near previous square center
        dist2 = (x - circle_cx) ** 2 + (y - circle_cy) ** 2
        out[dist2 <= circle_r ** 2] += circle_amp
        # add a diagonal line (slope = 1) with given thickness and amplitude
        dist_to_diag = np.abs(y - x) / np.sqrt(2)  # perpendicular distance to line y = x
        out[dist_to_diag <= (line_thickness / 2)] += line_amplitude
        return out * scale
    # deterministic upper bound: sum of each Gaussian peak + full circle + full line (worst-case overlap)
    gaussian_peak_sum = sum(A for (_, _, A, _) in gs)
    lmax = scale * (gaussian_peak_sum + circle_amp + line_amplitude)
    return fluxfunc, lmax

# example: sample on a grid and display
nx = ny = 128
xs = np.linspace(0, 1, nx)
ys = np.linspace(0, 1, ny)
X, Y = np.meshgrid(xs, ys, indexing='xy')
lamfunc, lmax = get_fluxfunc(scale=10_000_000)
Z = lamfunc(X, Y)

plt.figure(figsize=(5,5))
plt.imshow(Z, extent=[0,1,0,1], cmap='magma', origin='lower')
plt.colorbar(label='value')
plt.xlabel('x')
plt.ylabel('y')
plt.show()

F = np.fft.rfftn(Z, axes=(0,1))
plt.imshow(np.log1p(np.abs(np.fft.fftshift(F, axes=(0)))), origin='lower', cmap='magma')
plt.colorbar()
plt.show()

In [ ]:
pts = ihpp_sim.simulate_ihpp2d(lambda_fn=lamfunc, lmax=lmax)
print(f"Simulated {len(pts)} points.")
# plt.figure(figsize=(6,6))
# plt.imshow(Z, origin='lower', extent=[0,1,0,1], cmap='magma')
# plt.scatter(pts[:,0], pts[:,1], s=10, edgecolors='white')
# plt.show()

In [ ]:
mult = np.pi * 2
F_NU = finufft.nufft2d1(mult * pts[:,0], mult * pts[:,1], np.ones(len(pts), dtype="complex128"), (nx, ny), isign=-1, eps=1e-12)
plt.imshow(np.log1p(np.abs(F_NU)), origin='lower', cmap='magma')
plt.colorbar()
plt.show()

In [ ]:
cfarscale = 2.0
bound = flux_probing.compute_amplitude_bound(npoints=len(pts), V=1.0, nfreqs=nx * ny//2)
F_bd_cfar1 = F_NU.copy()
F_bd_cfar_scale = F_NU.copy()
mask_cfar1 = np.abs(F_NU) < bound
mask_cfar2 = np.abs(F_NU) < bound * cfarscale
F_bd_cfar1[mask_cfar1] = 0.0
F_bd_cfar_scale[mask_cfar2] = 0.0
plt.imshow(np.log1p(np.abs(F_bd_cfar_scale)), origin='lower', cmap='magma')
plt.show()

In [ ]:
def eval_basis(fx, ft, x, t):
    return np.exp(-2j * np.pi * (fx * x + ft * t))

In [ ]:
from scipy.stats import wilcoxon
allfx = np.fft.fftfreq(nx, 1/nx)
allfy = np.fft.fftfreq(ny, 1/ny)
mask_sym_real = np.zeros((nx, ny), dtype=bool)
mask_sym_imag = np.zeros((nx, ny), dtype=bool)
pbar = tqdm(total=nx * ny)
for i, fx in enumerate(allfx):
    for j, fy in enumerate(allfy):
        if fx == 0 and fy == 0:
            mask_sym_real[i, j] = True
            mask_sym_imag[i, j] = True
            pbar.update(1)
            continue
        vals = eval_basis(fx, fy, pts[:,0], pts[:,1])
        vals_real = np.real(vals)
        vals_imag = np.imag(vals)
        stat_real, pval_real = wilcoxon(vals_real)
        stat_imag, pval_imag = wilcoxon(vals_imag)
        if pval_real < 0.01:
            mask_sym_real[i, j] = True
        if pval_imag < 0.01:
            mask_sym_imag[i, j] = True
        pbar.update(1)
F_bd_sym = np.real(F_NU) * np.fft.fftshift(mask_sym_real) + 1j * np.imag(F_NU) * np.fft.fftshift(mask_sym_imag)

In [ ]:
F_bd_sym = np.real(F_NU) * np.fft.fftshift(mask_sym_real) + 1j * np.imag(F_NU) * np.fft.fftshift(mask_sym_imag)

In [ ]:
cfarscale = 1.4
bound = flux_probing.compute_amplitude_bound(npoints=len(pts), V=1.0, nfreqs=nx * ny//2)
F_bd_cfar1 = F_NU.copy()
F_bd_cfar_scale = F_NU.copy()
mask_cfar1 = np.abs(F_NU) < bound
mask_cfar2 = np.abs(F_NU) < bound * cfarscale
F_bd_cfar1[mask_cfar1] = 0.0
F_bd_cfar_scale[mask_cfar2] = 0.0
plt.imshow(np.log1p(np.abs(F_bd_cfar_scale)), origin='lower', cmap='magma')
plt.show()
plt.imshow(np.log1p(np.abs(F_bd_sym)), origin='lower', cmap='magma')
plt.show()

In [ ]:
plt.imshow(np.log1p(np.abs(F_bd_sym)), origin='lower', cmap='magma')

In [ ]:
plt.imshow(np.log1p(np.abs(F_bd)), origin='lower', cmap='magma')

In [ ]:
# recon1 = np.fft.ifftn(np.fft.ifftshift(F_bd_sym, axes=(0, 1)), s=(nx, ny), axes=(0, 1), norm='forward')
# recon1 = np.clip(recon1.real, 0, None)
recon2 = np.fft.ifftn(np.fft.ifftshift(F_bd_cfar1, axes=(0, 1)), s=(nx, ny), axes=(0, 1), norm='forward')
recon2 = np.clip(recon2.real, 0, None)
recon3 = np.fft.ifftn(np.fft.ifftshift(F_bd_cfar_scale, axes=(0, 1)), s=(nx, ny), axes=(0, 1), norm='forward')
recon3 = np.clip(recon3.real, 0, None)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
im_gt = axes[0].imshow(Z, cmap='magma', origin='lower')
axes[0].set_title('ground truth')
fig.colorbar(im_gt, ax=axes[0])
# im0 = axes[1].imshow(recon1.T, cmap='magma', origin='lower')
# axes[1].set_title(r'symmetry stat ($\alpha=0.01$) (wilcoxon test)')
# fig.colorbar(im0, ax=axes[1])
im1 = axes[2].imshow(recon2.T, cmap='magma', origin='lower')
axes[2].set_title('CFAR x 1')
fig.colorbar(im1, ax=axes[2])
im2 = axes[3].imshow(recon3.T, cmap='magma', origin='lower')
axes[3].set_title(f'CFAR x {cfarscale}')
fig.colorbar(im2, ax=axes[3])

plt.tight_layout()
plt.show()

In [ ]:
recon = np.fft.ifftn(np.fft.ifftshift(F_bd, axes=(0, 1)), s=(nx, ny), axes=(0, 1), norm='forward')
recon = np.clip(recon.real, 0, None)
plt.imshow(recon.T, cmap='magma')
plt.colorbar()
plt.show()

## Regularized least squares solver

In [ ]:
from scipy.sparse.linalg import LinearOperator, cg
import numpy as np

# regularized LS solve: (A^H A + alpha L^H L) c = A^H d
# A: NUFFT type-2 (grid coeffs -> values at pts)
# A^H: NUFFT type-1 (values at pts -> grid coeffs)
# y = A^H d is assumed to be F_NU (grid-shaped); solve for grid coeffs c.

# parameters (tweak as needed)
alpha = 10                # regularization weight
reg_kind = "lowpass"        # "identity" or "lowpass"
maxiter = 200
tol = 1e-6

# shapes / coords (use existing variables)
ms, mt = F_NU.shape         # grid shape used for the NUFFT grid
xj = mult * pts[:, 0]
yj = mult * pts[:, 1]
N = ms * mt

# helper wrappers for finufft calls (support tuple or separate dims)
def _nufft2d2_eval(c_grid):
    # c_grid : shape (ms, mt)
    return finufft.nufft2d2(xj, yj, c_grid.astype(np.complex128), isign=1, eps=1e-12)

def _nufft2d1_adj(vals_at_pts):
    # vals_at_pts : shape (n_points,)
    return finufft.nufft2d1(xj, yj, vals_at_pts.astype(np.complex128), (ms, mt), isign=-1, eps=1e-12)

# build low-pass weight if requested
if reg_kind == "identity":
    weight = np.ones((ms, mt), dtype=float)
else:
    # frequency coordinates (normalized cycles)
    kx = np.fft.fftshift(np.fft.fftfreq(ms))
    ky = np.fft.fftshift(np.fft.fftfreq(mt))
    KX, KY = np.meshgrid(kx, ky, indexing="ij")
    # radial squared frequency -> stronger penalty at high freq
    # weight = 1.0 + (KX ** 2 + KY ** 2) * (ms * mt)  # scale factor to make weight sensible
    weight = 1.0 + (KX ** 2 + KY ** 2) * 1000  # scale factor to make weight sensible

weight = weight.astype(np.float64)
weight_flat = weight.ravel()

# linear operator for normal matrix-vector product
def matvec(c_flat):
    c_grid = c_flat.reshape((ms, mt)).astype(np.complex128)
    # A c -> values at points (type-2)
    vals = _nufft2d2_eval(c_grid)
    # A^H (A c) -> grid (type-1)
    ata = _nufft2d1_adj(vals) / len(pts)  # scale by n_points for proper adjoint
    # regularizer term alpha * (L^H L) c -> here diagonal in freq domain: alpha * weight * c
    reg = alpha * (weight * c_grid)
    out = ata + reg
    return out.ravel()

Aop = LinearOperator((N, N), matvec=matvec, dtype=np.complex128)

# right-hand side b = A^H d = F_NU (already computed). Flatten to vector.
b = F_NU.ravel().astype(np.complex128)

# run conjugate gradient on Hermitian positive-definite normal system
x0 = F_NU.ravel().copy()  # warm start (adjoint result)
sol_flat, info = cg(Aop, b, x0=x0, rtol=tol, maxiter=maxiter)

if info == 0:
    print("CG converged")
else:
    print("CG info:", info)

c_grid = sol_flat.reshape((ms, mt))

# diagnostics: residual norm and comparison to adjoint
res = matvec(sol_flat) - b
print("residual norm (normal eq):", np.linalg.norm(res))

# expose solution for later cells
c_nufft_sol = c_grid        # optimized frequency coefficients on NUFFT grid

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 6))

# prepare images (log-scaled magnitudes)
A = np.log1p(np.abs(np.fft.fftshift(F, axes=(0))))
B = np.log1p(np.abs(c_nufft_sol[:, mt//2:]))
C = np.log1p(np.abs(F_NU[:, mt//2:]))
D = np.log1p(np.abs(F_bd))

# align widths for fair visual comparison
w = min(A.shape[1], B.shape[1], C.shape[1], D.shape[1])
A = A[:, :w]
B = B[:, :w]
C = C[:, :w]
D = D[:, :w]

# vmin = min(A.min(), B.min(), C.min())
# vmax = max(A.max(), B.max(), C.max())

im0 = axes[0].imshow(A, cmap='magma', origin='lower')
axes[0].set_title("GT spectrum (log1p(|F|))")
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(B, cmap='magma', origin='lower')
axes[1].set_title("least squares spectrum (log1p(|F|))")
plt.colorbar(im1, ax=axes[1])
im2 = axes[2].imshow(C, cmap='magma', origin='lower')
axes[2].set_title("adjoint spectrum (log1p(|F|))")
plt.colorbar(im2, ax=axes[2])
im3 = axes[3].imshow(D, cmap='magma', origin='lower')
axes[3].set_title("CFAR (log1p(|F|))")
plt.colorbar(im3, ax=axes[3])
plt.tight_layout()
plt.show()

In [ ]:
recon = np.fft.irfftn(np.fft.ifftshift(c_nufft_sol[:, mt//2:], axes=(0)), s=(nx, ny), axes=(0, 1), norm='forward')
recon = np.clip(recon, 0, None)
plt.imshow(recon, cmap='magma')
plt.colorbar()
plt.show()

In [ ]:
recon = np.fft.irfftn(np.fft.ifftshift(F_bd, axes=(0)), s=(nx, ny), axes=(0, 1), norm='forward')
recon = np.clip(recon, 0, None)
plt.imshow(recon, cmap='magma')
plt.colorbar()
plt.show()

In [ ]:
import numpy as np

np.fft.fftfreq(20, 1/20)

In [ ]:
# diagnostic impulses
e_dc = np.zeros((ms, mt), dtype=np.complex128); e_dc[ms//2, mt//2] = 1.0
e_hf = np.zeros_like(e_dc); e_hf[ms//5, mt//5] = 1.0

out_dc = matvec(e_dc.ravel()).reshape(ms,mt)
out_hf = matvec(e_hf.ravel()).reshape(ms,mt)

print("||A^H A e_dc||_2:", np.linalg.norm(out_dc))
print("||A^H A e_hf||_2:", np.linalg.norm(out_hf))
print("ratio dc/hf:", np.linalg.norm(out_dc)/ (np.linalg.norm(out_hf)+1e-30))

In [ ]:
plt.imshow(np.log1p(np.abs(out_dc)), cmap='magma')
plt.colorbar()
plt.show()

In [ ]:
plt.imshow(np.log1p(np.abs(out_hf)), cmap='magma')

In [ ]:
# temporarily disable reg inside matvec (or make a small wrapper)
def ata_only(c_flat):
    c_grid = c_flat.reshape((ms, mt)).astype(np.complex128)
    vals = _nufft2d2_eval(c_grid)
    ata = _nufft2d1_adj(vals)
    return ata.ravel()

# test response to a delta at some freq
test = np.zeros((ms, mt), dtype=np.complex128); test[ms//8, mt//8] = 1.0
resp = ata_only(test.ravel()).reshape(ms,mt)
print("max(abs(resp)):", np.max(np.abs(resp)))
print("DC element of resp:", resp[ms//2,mt//2])

In [ ]:
plt.imshow(np.log1p(np.abs(resp)))
plt.colorbar()
plt.show()

In [ ]:
# random test vector
c_rand = (np.random.randn(ms,mt)+1j*np.random.randn(ms,mt)).astype(np.complex128)
vals = _nufft2d2_eval(c_rand)
ata = _nufft2d1_adj(vals)

print("||c_rand||_2:", np.linalg.norm(c_rand))
print("||ata||_2:", np.linalg.norm(ata))

In [ ]:
np.linalg.norm(ata) / np.linalg.norm(c_rand), len(pts)

## global fft tests

In [ ]:
# align shapes (crop to smallest width) and compute linear magnitudes
M = np.abs(np.fft.fftshift(F, axes=(0,)))                # linear |F| (shape (256,129))
N = np.abs(F_NU[:, ny // 2:])                       # linear |F_NU| (shape (256,127))
w = min(M.shape[1], N.shape[1])
Mc = M[:, :w]
Nc = N[:, :w]

# log-domain images for direct visual comparison (shared color scale)
A = np.log1p(Mc)
B = np.log1p(Nc)
vmin = min(A.min(), B.min())
vmax = max(A.max(), B.max())
vminNU = B.min()
vmaxNU = B.max()

# error measures (linear domain)
abs_err = np.abs(Mc - Nc)
eps = 1e-12
perc_err = np.nan_to_num(abs_err / (Mc + eps) * 100.0)

# plot: top row = original comparisons (shared colorbar), bottom row = abs and percent error
fig, axes = plt.subplots(1, 3, figsize=(18, 10))

im0 = axes[0].imshow(A, origin='lower', cmap='magma')
fig.colorbar(im0, ax=axes[0], shrink=0.8)
axes[0].set_title('log1p |fftshift(F)|')
im1 = axes[1].imshow(B, origin='lower', cmap='magma', vmax=vmaxNU, vmin=vminNU)
fig.colorbar(im1, ax=axes[1], shrink=0.8)
axes[1].set_title('log1p |F_NU[:, ny//2:]|')

im2 = axes[2].imshow(np.log1p(np.abs(F_bd)), origin='lower', cmap='magma', vmax=vmaxNU, vmin=vminNU)
fig.colorbar(im2, ax=axes[2], shrink=0.8)
axes[2].set_title('log1p |F_NU[:, ny//2:]| thresholded')

# absolute error (log-scaled for visibility)
# im2 = axes[2].imshow(np.log1p(abs_err), origin='lower', cmap='magma')
# axes[2].set_title('log1p Absolute error | |F| - |F_NU| |')
# fig.colorbar(im2, ax=axes[2], shrink=0.8)

# # percentage (normalized) error (show raw percent; clip extreme outliers for readability)
# perc_clip = np.clip(perc_err, 0, np.percentile(perc_err, 99.5))
# im3 = axes[3].imshow(perc_clip, origin='lower', cmap='magma')
# axes[3].set_title('Normalized error (percentage, clipped at 99.5th pct)')
# fig.colorbar(im3, ax=axes[3], label='%')

plt.tight_layout()
plt.show()

In [ ]:
recon = np.fft.irfftn(np.fft.fftshift(F_bd, axes=(0)), s=(nx, ny), axes=(0, 1), norm='forward')
recon = np.clip(recon, 0, None)
plt.imshow(recon, origin='lower', cmap='magma')
plt.colorbar()
plt.show()

In [ ]:
# patch-wise NUFFT synthesis (type-1) + FFT analysis (inverse) using 2D Hann window, 50% overlap.
# Assumes Z is the image (H,W) sampled on xs, ys as in notebook (unit coords).
patch_size = 64           # configurable
hop = patch_size // 2     # 50% overlap

H_img, W_img = Z.shape
# per-user assumption: divisible by hop/patch
assert H_img % hop == 0 and W_img % hop == 0, "image must be divisible by hop (per assumption)"

n_steps_h = (H_img - patch_size) // hop + 1
n_steps_w = (W_img - patch_size) // hop + 1

# separable Hann window
win_h = signal.windows.hann(patch_size, sym=False)
win_w = signal.windows.hann(patch_size, sym=False)
win_2d = np.outer(win_h, win_w).astype(float)

# storages for per-patch spectra (NUFFT full -> keep rfft half) and thresholded spectra
spec_shape = (n_steps_h, n_steps_w, patch_size, patch_size // 2 + 1)
spectra_patches = np.zeros(spec_shape, dtype=np.complex128)
spectra_thresh = np.zeros_like(spectra_patches)

# accumulators for overlap-add reconstruction
recon_accum = np.zeros_like(Z, dtype=float)
weight_accum = np.zeros_like(Z, dtype=float)

# precompute dx, dy (assumes xs, ys are regular linspace used earlier)
dx = float(xs[1] - xs[0])
dy = float(ys[1] - ys[0])
patch_width_norm = patch_size * dx
patch_height_norm = patch_size * dy
patch_area = patch_width_norm * patch_height_norm

pbar = tqdm(total=n_steps_h * n_steps_w, desc="patch NUFFT synth + FFT recon")
for iy in range(n_steps_h):
    y0 = iy * hop
    y_min = ys[y0]
    for ix in range(n_steps_w):
        x0 = ix * hop
        x_min = xs[x0]

        # select points that fall inside this patch (pts in unit coords)
        in_x = (pts[:, 0] >= x_min) & (pts[:, 0] < (x_min + patch_width_norm))
        in_y = (pts[:, 1] >= y_min) & (pts[:, 1] < (y_min + patch_height_norm))
        mask_pts = in_x & in_y
        pts_idx = np.nonzero(mask_pts)[0]
        n_pts_patch = pts_idx.size

        if n_pts_patch == 0:
            # no events in this patch -> zero spectrum, contributes nothing
            F_r = np.zeros((patch_size, patch_size // 2 + 1), dtype=np.complex128)
            F_r_thresh = F_r.copy()
        else:
            # map points to local [0,1) coordinates inside patch
            pts_local_x = (pts[pts_idx, 0] - x_min) / patch_width_norm
            pts_local_y = (pts[pts_idx, 1] - y_min) / patch_height_norm

            # --- Use events directly in NUFFT with continuous window weights ---
            # continuous separable Hann window on [0,1]
            def hann_cont(u):
                u = np.asarray(u)
                u = np.clip(u, 0.0, 1.0)  # guard against tiny numerical drift
                return 0.5 * (1.0 - np.cos(2.0 * np.pi * u))

            # per-event continuous weight (preserves phase by using real scalar weights)
            evt_weights = hann_cont(pts_local_y) * hann_cont(pts_local_x)
            # use complex-valued strengths for NUFFT
            strengths = np.sqrt(evt_weights.astype(np.complex128))

            # type-1 NUFFT (nonuniform->uniform Fourier modes) over patch grid
            # Use same mult = 2*pi convention as earlier cells (x in [0,1] -> phase 2π*x*k)
            F_full = finufft.nufft2d1(mult * pts_local_x, mult * pts_local_y,
                                      strengths / np.sqrt(patch_area),
                                      (patch_size, (patch_size+1)*2//2), isign=-1, eps=1e-12)
            # convert to rfft-like half-spectrum to match np.fft.irfftn convention
            F_r = F_full[:, patch_size // 2:].astype(np.complex128)
            # --- end direct-event NUFFT ---

            # compute CFAR-like amplitude bound per patch and threshold
            # nfreqs: approximate number of positive-frequency bins
            nfreqs = (patch_size * patch_size) // 2
            bound_patch = flux_probing.compute_amplitude_bound(npoints=n_pts_patch, V=patch_area, nfreqs=nfreqs) * 2.0

            F_r_thresh = F_r.copy()
            F_r_thresh[np.abs(F_r) < bound_patch] = 0.0

        # store spectra
        spectra_patches[iy, ix] = F_r
        spectra_thresh[iy, ix] = F_r_thresh

        # inverse FFT (analysis) from thresholded rfft coefficients -> spatial patch
        rec_patch = np.real(np.fft.irfftn(F_r_thresh / np.sqrt(patch_area), s=(patch_size, patch_size), axes=(0, 1), norm='forward'))
        rec_patch = np.clip(rec_patch, 0, None)

        # overlap-add using same Hann window (classic OLA)
        recon_accum[y0 : y0 + patch_size, x0 : x0 + patch_size] += rec_patch * np.sqrt(win_2d)
        weight_accum[y0 : y0 + patch_size, x0 : x0 + patch_size] += win_2d

        pbar.update(1)
pbar.close()

# normalize overlap-add
eps = 1e-12
reconstructed_patchwise = recon_accum / (weight_accum + eps)
reconstructed_patchwise = recon_accum

# expose for later cells / diagnostics
patch_spectra_nufft = spectra_patches
patch_spectra_thresh = spectra_thresh
recon_nufft_patchwise = reconstructed_patchwise

# quick diagnostic: log errors vs original Z
logger.info("patchwise NUFFT reconstruction max abs error: %g, mean abs error: %g",
            np.max(np.abs(reconstructed_patchwise - Z)), np.mean(np.abs(reconstructed_patchwise - Z)))

plt.imshow(reconstructed_patchwise, origin='lower', cmap='magma')
plt.colorbar()
plt.show()

In [ ]:
# plt.figure(figsize=(20,0))
plt.imshow(reconstructed_patchwise[10:250, 10:250], origin='lower', cmap='magma')
plt.colorbar()
plt.show()

# throw 2d patch hann window at a png

In [ ]:
path = Path("data/screenshot-test.png")
img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)

In [ ]:
patch_h, patch_w = 32, 32
hop_h, hop_w = 16, 16

H, W = img.shape
# compute number of steps and padded output size
n_steps_h = int(np.ceil(max(1, (H - patch_h) / hop_h)) + 1)
n_steps_w = int(np.ceil(max(1, (W - patch_w) / hop_w)) + 1)
out_h = (n_steps_h - 1) * hop_h + patch_h
out_w = (n_steps_w - 1) * hop_w + patch_w
pad_h = out_h - H
pad_w = out_w - W

logger.info("image %s x %s -> padded %s x %s (steps %s x %s)", H, W, out_h, out_w, n_steps_h, n_steps_w)

# pad by reflection to avoid edge artifacts
padded = np.pad(img.astype(float), ((0, pad_h), (0, pad_w)), mode="reflect")

# hann window (2D separable)
win_h = signal.windows.hann(patch_h, sym=False)
win_w = signal.windows.hann(patch_w, sym=False)
# win_h = signal.windows.hamming(patch_h, sym=False)
# win_w = signal.windows.hamming(patch_w, sym=False)
window = np.outer(win_h, win_w).astype(float)

# accumulators for overlap-add reconstruction
recon_accum = np.zeros_like(padded, dtype=float)
weight_accum = np.zeros_like(padded, dtype=float)
gain_accum = np.zeros_like(padded, dtype=float)

# store spectra per patch if needed
spectra = np.zeros((n_steps_h, n_steps_w, patch_h, patch_w // 2 + 1), dtype=np.complex128)
spectra_thresh = np.zeros((n_steps_h, n_steps_w, patch_h, patch_w // 2 + 1), dtype=np.complex128)
threshold = 20.0
norm = "ortho"
sqrtwindow = False
pbar = tqdm(total=n_steps_h * n_steps_w, desc="processing patches")
for iy in range(n_steps_h):
    y = iy * hop_h
    for ix in range(n_steps_w):
        x = ix * hop_w
        patch = padded[y : y + patch_h, x : x + patch_w].astype(float)
        if sqrtwindow:
            windowed = patch * np.sqrt(window)
        else:
            windowed = patch * window

        # forward 2D FFT of the windowed patch
        F = np.fft.rfftn(windowed, norm=norm, axes=(0, 1))
        spectra[iy, ix] = F
        # Soft-thresholding options (operate on complex spectrum, preserve phase)
        eps = 1e-12
        mag = np.abs(F)

        # Method 1: simple soft shrinkage (lasso-style)
        # mag_shrink = np.maximum(mag - threshold, 0.0)
        # gain = (mag_shrink / (mag + eps))

        # Method 2: relative shrinkage (multiplicative attenuation)
        # gain = np.maximum(0.0, 1.0 - threshold / (mag + eps))

        # Method 3: Wiener-like attenuation using a noise estimate
        # noise_est = np.median(mag) / 0.6745
        # SNR = (mag**2) / (noise_est**2 + eps)
        # gain = SNR / (SNR + threshold)   # threshold acts like regularizer here

        # Method 4: soft (smooth) Gaussian roll-off around threshold
        # sigma = threshold * 0.5
        # gain = 1.0 - np.exp(-0.5 * ((mag - threshold) / (sigma + eps))**2)
        # gain = np.clip(gain, 0.0, 1.0)

        # guard against NaNs / tiny numerical noise
        F_bd = F.copy()
        # F_bd *= gain
        F_bd[mag < threshold] = 0.0

        spectra_thresh[iy, ix] = F_bd
        # gain_hard = np.zeros_like(gain)
        # gain_hard[mag >= threshold] = 1.0

        # inverse to obtain the windowed patch back (no spectral processing here)
        if sqrtwindow:
            rec_patch = np.real(np.fft.irfftn(F_bd, norm=norm, s=(patch_h, patch_w), axes=(0, 1))) * np.sqrt(window)
        else:
            rec_patch = np.real(np.fft.irfftn(F_bd, norm=norm, s=(patch_h, patch_w), axes=(0, 1)))

        # overlap-add
        recon_accum[y : y + patch_h, x : x + patch_w] += rec_patch
        weight_accum[y : y + patch_h, x : x + patch_w] += window
        pbar.update(1)
pbar.close()

# normalize by accumulated window weights (avoid division by zero)
eps = 1e-12
reconstructed_padded = recon_accum / (weight_accum + eps)

# crop back to original image size
reconstructed = reconstructed_padded[:H, :W]

# diagnostics
diff = reconstructed - img.astype(float)
logger.info("reconstruction max abs error: %g, mean abs error: %g", np.max(np.abs(diff)), np.mean(np.abs(diff)))

# Optionally convert back to uint8 for display/saving (clamp)
reconstructed_uint8 = np.clip(np.round(reconstructed), 0, 255).astype(np.uint8)

# Expose useful names for later cells
patch_spectra = spectra
recon = reconstructed
recon_uint8 = reconstructed_uint8
weight_map = weight_accum[:H, :W]

# Basic visualization (uncomment to display)
import matplotlib.pyplot as plt
plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.title("original"); plt.imshow(img, cmap="gray", origin="lower")
plt.subplot(1,3,2); plt.title("reconstructed"); plt.imshow(recon_uint8, cmap="gray", origin="lower")
plt.subplot(1,3,3); plt.title("diff"); plt.imshow(np.abs(recon - img), cmap="magma", origin="lower"); plt.colorbar()
cv2.imwrite(f"output/colortests/test2d/screenshot-test-recon-patch{patch_h}-hop{hop_h}-thresh{threshold}-sqrtwin{sqrtwindow}.png", recon_uint8)

In [ ]:
windowed = patch * np.sqrt(window)
# forward 2D FFT of the windowed patch
F = np.fft.rfftn(windowed, norm=norm, axes=(0, 1))
plt.imshow(np.log1p(np.fft.fftshift(np.abs(F), axes=(0))), cmap="magma")
plt.colorbar()
plt.show()

In [ ]:
windowed = patch * window
# forward 2D FFT of the windowed patch
F = np.fft.rfftn(windowed, norm=norm, axes=(0, 1))
plt.imshow(np.log1p(np.fft.fftshift(np.abs(F), axes=(0))), cmap="magma")
plt.colorbar()
plt.show()

In [ ]:
plt.imshow(patch)

In [ ]:
plt.imshow(weight_accum[400:450, 750:800])
plt.colorbar()
plt.show()

In [ ]:
plt.imshow(np.fft.fftshift(gain))
plt.colorbar()
plt.show()
plt.imshow(np.fft.fftshift(gain_hard))
plt.colorbar()
plt.show()

In [ ]:
idx = (5, 0)
F = spectra[idx]
F_bd = spectra_thresh[idx]
mag1 = np.log1p(np.abs(np.fft.fftshift(F)))
mag2 = np.log1p(np.abs(np.fft.fftshift(F_bd)))

vmin = min(mag1.min(), mag2.min())
vmax = max(mag1.max(), mag2.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
im0 = axes[0].imshow(mag1, cmap='magma', origin='lower', vmin=vmin, vmax=vmax)
axes[0].set_title('log1p|FFT(F)| (shifted)')
im1 = axes[1].imshow(mag2, cmap='magma', origin='lower', vmin=vmin, vmax=vmax)
axes[1].set_title('log1p|FFT(F_bd)| (shifted)')

# shared colorbar
fig.colorbar(im0, ax=axes.ravel().tolist(), shrink=0.8)
# plt.tight_layout()
# plt.show()
# plt.imshow(np.log1p(np.abs(np.fft.fftshift(F_bd))))
# plt.colorbar()
# plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(gain_accum, interpolation='nearest')
plt.colorbar()
plt.title('gain_accum (inverse FFT of frequency gain)')
plt.gcf().set_size_inches(12, 6)
plt.tight_layout()
plt.savefig("output/screenshot-test-gain-accum.png", )

# throw window hann at a color video now

In [ ]:
output_dir = Path("output/colortests/fan-slowmo")
output_dir.mkdir(parents=True, exist_ok=True)
path = Path("data/fan-slowmo-gjPOotFgJEM.mp4")
# get frames from video
cap = cv2.VideoCapture(str(path))
frames = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    frames.append(gray)
cap.release()
frames = np.array(frames)  # shape (num_frames, H, W)
frames = frames[:, ::2, ::2]  # downsample spatially for speed

In [ ]:
# global fft
threshold = 0.0
recon_t = 125
do_s = True
# forward 3D FFT
F = np.fft.rfftn(frames.T, axes=(0, 1, 2), norm="forward")

# # thresholding
# mag = np.abs(F)
# mask = mag < threshold
# print(f"dropped {np.count_nonzero(mask)} freqs")
# F[mask] = 0.0
# # zero time-frequencies beyond nyquist set by recon_t (axis 0 is unshifted)
# T = F.shape[0]
# nyq_idx = recon_t // 2
# k = np.arange(T)
# mag_idx = np.minimum(k, T - k)
# time_mask = mag_idx > nyq_idx
# n_freq_idxs = np.count_nonzero(time_mask)
# F[time_mask, ...] = 0.0
# print(f"zeroed {n_freq_idxs} time-freq indices (nyquist idx = {nyq_idx})")

# inverse
if do_s:
    rec = np.real(np.fft.irfftn(F, s=(frames.shape[2], frames.shape[1], recon_t), axes=(0, 1, 2), norm="forward")).T
else:
    rec = np.real(np.fft.irfftn(F, s=(frames.shape[0], frames.shape[1], frames.shape[2]), axes=(0, 1, 2), norm="forward"))
    sample_idxs = np.linspace(0, frames.shape[0], recon_t, dtype=int, endpoint=False)
    rec = rec[sample_idxs]
ihpp_data.to_video(rec, output_dir / f"recon-global-s{do_s}-frames{recon_t}-thresh{threshold}.avi", playback_fps=30, cmap="grey")
utils.avi_to_mov(output_dir / f"recon-global-s{do_s}-frames{recon_t}-thresh{threshold}.avi")

In [ ]:
i = 0
im1 = np.log1p(np.fft.fftshift(np.abs(F[i]), axes=(0)))
im2 = np.log1p(np.fft.fftshift(mag[i], axes=(0)))

vmin = min(im1.min(), im2.min())
vmax = max(im1.max(), im2.max())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
im0 = axes[0].imshow(im1, cmap='magma', origin='lower', vmin=vmin, vmax=vmax)
axes[0].set_title(f'log1p |F| thresholded (shifted)')
fig.colorbar(im0, ax=axes[0], fraction=0.02, pad=0.04)

im1p = axes[1].imshow(im2, cmap='magma', origin='lower', vmin=vmin, vmax=vmax)
axes[1].set_title(f'log1p |F| original')
fig.colorbar(im1p, ax=axes[1], fraction=0.02, pad=0.04)

plt.tight_layout()

In [ ]:
# 3D patch-wise FFT (time, y, x) with overlap-add reconstruction
# Uses existing variables: frames (T,H,W), fftn, ifftn, plt, np, logger, tqdm

# patch / hop sizes requested
patch_t, patch_h, patch_w = frames.shape[0], 32, 32
hop_t, hop_h, hop_w = 0, 16, 16

T, H, W = frames.shape

# compute number of steps and padded output size (same convention as earlier cell)
# n_steps_t = int(np.ceil(max(1, (T - patch_t) / hop_t)) + 1)
n_steps_t = 1
n_steps_h = int(np.ceil(max(1, (H - patch_h) / hop_h)))
n_steps_w = int(np.ceil(max(1, (W - patch_w) / hop_w)))

out_t = (n_steps_t - 1) * hop_t + patch_t
out_h = (n_steps_h - 1) * hop_h + patch_h
out_w = (n_steps_w - 1) * hop_w + patch_w

pad_t = out_t - T
pad_h = out_h - H
pad_w = out_w - W

logger.info("frames %s x %s x %s -> padded %s x %s x %s (steps %s x %s x %s)",
            T, H, W, out_t, out_h, out_w, n_steps_t, n_steps_h, n_steps_w)

# separable Hann windows
win_t = np.ones(patch_t)
win_h = np.hanning(patch_h + 1)[:-1]
win_w = np.hanning(patch_w + 1)[:-1]
window = win_t[:, None, None] * win_h[None, :, None] * win_w[None, None, :]

recon_t = 250
win_t = np.ones(recon_t)
win_h = np.hanning(patch_h + 1)[:-1]
win_w = np.hanning(patch_w + 1)[:-1]
window_out = win_t[:, None, None] * win_h[None, :, None] * win_w[None, None, :]

# accumulators
out_shape = (recon_t, H, W)
recon_accum = np.zeros(out_shape, dtype=float)
weight_accum = np.zeros(out_shape, dtype=float)

# spectra storage (may be large)
spectra = np.zeros((n_steps_t, n_steps_h, n_steps_w, patch_t, patch_h, patch_w//2+1), dtype=np.complex128)

threshold = 0.0
pbar = tqdm(total=n_steps_t * n_steps_h * n_steps_w, desc="processing patches")
# iterate patches
for it in range(n_steps_t):
    t0 = it * hop_t
    for iy in range(n_steps_h):
        y0 = iy * hop_h
        for ix in range(n_steps_w):
            x0 = ix * hop_w
            patch = frames[t0 : t0 + patch_t, y0 : y0 + patch_h, x0 : x0 + patch_w].astype(float)
            windowed = patch * window

            # forward 3D FFT
            F = np.fft.rfftn(windowed, axes=(0, 1, 2), norm="ortho")
            spectra[it, iy, ix] = F

            # thresholding
            eps = 1e-12
            mag = np.abs(F)
            F[mag < threshold] = 0.0

            # inverse
            rec_patch = np.real(np.fft.irfftn(F, s=(recon_t, patch_h, patch_w), axes=(0, 1, 2), norm="ortho"))
            # rec_patch *= window_out

            # overlap-add
            recon_accum[:, y0 : y0 + patch_h, x0 : x0 + patch_w] += rec_patch
            weight_accum[:, y0 : y0 + patch_h, x0 : x0 + patch_w] += window_out

            pbar.update(1)
pbar.close()

# normalize and crop
eps = 1e-12
reconstructed = recon_accum / (weight_accum + eps)
# uint8 copy for saving/display (clip)
recon_uint8 = np.clip(np.round(reconstructed), 0, 255).astype(np.uint8)

out_path = output_dir / f"fanrecon-frames{recon_t}-chunk{patch_t}x{patch_h}x{patch_w}-hop{hop_t}x{hop_h}x{hop_w}-thresh{threshold}.avi"
# recon3d_uint8 is (T, H, W) uint8 from previous cell
ihpp_data.to_video(recon_uint8, str(out_path), playback_fps=30, cmap="grey")

In [ ]:
# generate a synthetic video: flashing Gaussian spot + sliding expanding gray rectangle
# uses existing imports: np, Path, ihpp_data, utils

out_dir = Path("output/synthetic")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "flash_gauss_sliding_rect.avi"

# configurable params
W, H = 512, 512
duration_s = 10.0
fps = 30
num_frames = int(duration_s * fps)

flash_freq = 4.0          # Hz, configurable
flash_binary = False       # if True use on/off pulses, else smooth sinusoidal
gauss_sigma_px = 40.0     # Gaussian stddev in pixels
gauss_amplitude = 255.0

rect_gray = 120           # gray level of sliding rectangle (0-255)
rect_start_width = W // 8 # initial rectangle width (off-screen on the right)

# precompute gaussian spot
ys = np.arange(H)[:, None]
xs = np.arange(W)[None, :]
cx, cy = (W - 1) / 2.0, (H - 1) / 2.0
dx2 = (xs - cx) ** 2
dy2 = (ys - cy) ** 2
gauss = np.exp(-0.5 * (dx2 + dy2) / (gauss_sigma_px ** 2))
gauss_img = (gauss_amplitude * gauss).astype(np.float32)

frames = np.zeros((num_frames, H, W), dtype=np.uint8)

for i in range(num_frames):
    t = i / fps
    p = i / (num_frames - 1)  # progress 0..1

    # flash multiplier
    if flash_binary:
        flash_on = np.sin(2 * np.pi * flash_freq * t) > 0
        flash_scale = 1.0 if flash_on else 0.0
    else:
        flash_scale = 0.5 * (1.0 + np.sin(2 * np.pi * flash_freq * t))

    # gaussian contribution
    frame = gauss_img * flash_scale

    # sliding + expanding rectangle
    left = int(np.floor(W * (1.0 - p)))                      # moves from W -> 0
    width = rect_start_width + p * (W - rect_start_width)    # grows to full width
    right = int(np.floor(left + width))
    left_clip = max(0, left)
    right_clip = min(W, right)
    if left_clip < right_clip:
        frame[:, left_clip:right_clip] = rect_gray

    # clamp and store
    np.clip(frame, 0, 255, out=frame)
    frames[i] = frame.astype(np.uint8)

# write video (uses ihpp_data.to_video from notebook)
ihpp_data.to_video(frames, str(out_path), playback_fps=fps, cmap="grey")
# optional: create a .mov if utilities available
try:
    utils.avi_to_mov(out_path)
except Exception:
    pass

print(f"wrote {out_path} ({num_frames} frames, {fps} fps, {duration_s}s)")

In [ ]:
# global fft
threshold = 1000000
# forward 3D FFT
F = np.fft.rfftn(frames, axes=(0, 1, 2))
F_orig = F.copy()
# add noise to F
F += np.random.normal(0, 100000, size=F.shape)

# thresholding
mag = np.abs(F)
mask = mag < threshold
print(f"dropped {np.count_nonzero(mask)} freqs")
# F[mask] = 0.0

# inverse
rec = np.real(np.fft.irfftn(F, s=frames.shape, axes=(0, 1, 2)))
ihpp_data.to_video(rec, out_dir / f"recon-global-thresh{threshold}.avi", playback_fps=30, cmap="grey")
utils.avi_to_mov(out_dir / f"recon-global-thresh{threshold}.avi")

In [ ]:
i = 0
im1 = np.log1p(np.fft.fftshift(np.abs(F[i]), axes=(0)))
im2 = np.log1p(np.fft.fftshift(np.abs(F_orig[i]), axes=(0)))

vmin = min(im1.min(), im2.min())
vmax = max(im1.max(), im2.max())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
im0 = axes[0].imshow(im1, cmap='magma', origin='lower', vmin=vmin, vmax=vmax)
axes[0].set_title(f'log1p |F| thresholded (shifted)')
fig.colorbar(im0, ax=axes[0], fraction=0.02, pad=0.04)

im1p = axes[1].imshow(im2, cmap='magma', origin='lower', vmin=vmin, vmax=vmax)
axes[1].set_title(f'log1p |F| original')
fig.colorbar(im1p, ax=axes[1], fraction=0.02, pad=0.04)

plt.tight_layout()

# some power spectrum things

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Common grid (unit square)
Nx, Nt = 128, 128
x = np.linspace(0, 1, Nx, endpoint=False)
t = np.linspace(0, 1, Nt, endpoint=False)
T, X = np.meshgrid(t, x, indexing='ij')

# --- Parameters in unit domain ---
sigma = 3.0 / Nx              # Gaussian width (normalized)
x0 = 32.0 / Nx                # Gaussian center (normalized)
f0 = 40 / Nt                  # temporal flicker frequency (cycles per time-unit)
vx = -15 / Nx                 # velocity (fraction of spatial extent per time-step)

# --- Base spatial pattern: small Gaussian "ball" ---
ball_static = np.exp(-0.5 * ((X - x0) / sigma) ** 2)
C = 20000000

def case1(x, t):
    """Only flicker (separable)"""
    ball = C * np.exp(-0.5 * ((x - x0) / sigma) ** 2)
    flicker = 1.0 + 0.8 * np.cos(2 * np.pi * f0 * t * Nt)  # multiply by Nt since f0 was defined in cycles per full signal length
    return ball * flicker
def case2(x, t):
    """Only motion"""
    center_t = x0 + vx * t * Nt   # same normalization convention
    return C * np.exp(-0.5 * ((x - center_t) / sigma) ** 2)
def case3(x, t):
    """Motion + flicker"""
    center_t = x0 + vx * t * Nt   # same normalization convention
    ball = C * np.exp(-0.5 * ((x - center_t) / sigma) ** 2)
    flicker = 1.0 + 0.8 * np.cos(2 * np.pi * f0 * t * Nt)
    return ball * flicker

# --- Case 1: Only flicker (separable) ---
I_flicker = case1(X, T)

# --- Case 2: Only motion ---
I_motion = case2(X, T)

# --- Case 3: Motion + flicker ---
I_both = case3(X, T)

def simulate_inhomogeneous_poisson(intensity_fn, n_grid=256, seed=None):
    """
    Simulate an inhomogeneous Poisson point process on the unit square [0,1]x[0,1]
    using thinning.

    intensity_fn: function(x, t) -> lambda (can accept numpy arrays or scalars).
                  x and t are in [0,1].
    n_grid: grid size used to estimate the maximum intensity (n_grid x n_grid).
    seed: RNG seed or numpy Generator; if None a new Generator is created.

    Returns:
      points: (M,2) array with columns [x, t] for accepted points.
      info: dict with keys 'lambda_max', 'n_proposed', 'n_accepted'
    """
    # RNG
    if isinstance(seed, np.random.Generator):
        rng = seed
    else:
        rng = np.random.default_rng(seed)

    # estimate lambda_max on a regular grid (midpoints)
    coords = (np.arange(n_grid) + 0.5) / n_grid
    Xg, Tg = np.meshgrid(coords, coords, indexing='xy')  # Xg: x, Tg: t
    try:
        Lam_grid = np.asarray(intensity_fn(Xg, Tg))
    except Exception:
        # fallback: evaluate elementwise
        Lam_grid = np.vectorize(lambda xx, tt: intensity_fn(xx, tt))(Xg, Tg)
    lam_max = float(np.max(Lam_grid))
    if lam_max <= 0:
        return np.empty((0, 2))

    # propose a homogeneous Poisson number of points with rate lam_max over unit area
    n_prop = rng.poisson(lam_max)  # area = 1
    if n_prop == 0:
        return np.empty((0, 2))

    # sample uniform candidate points in unit square
    xs = rng.random(n_prop)
    ts = rng.random(n_prop)

    # evaluate intensity at candidates (handle vectorized and scalar functions)
    try:
        lam_vals = np.asarray(intensity_fn(xs, ts))
    except Exception:
        lam_vals = np.array([intensity_fn(xi, ti) for xi, ti in zip(xs, ts)])

    # thinning accept-reject
    accept_mask = rng.random(n_prop) < (lam_vals / lam_max)
    accepted = np.column_stack((xs[accept_mask], ts[accept_mask]))

    info = {'lambda_max': lam_max, 'n_proposed': n_prop, 'n_accepted': accepted.shape[0]}
    return accepted

def simulate_inhomogeneous_poisson_1d(intensity_fn, n_grid=1000, seed=None):
    """
    Simulate an inhomogeneous Poisson point process on the unit interval [0,1]
    using thinning.

    intensity_fn: function(x) -> lambda (can accept numpy arrays or scalars).
                    x in [0,1].
    n_grid: grid size used to estimate the maximum intensity.
    seed: RNG seed or numpy Generator; if None a new Generator is created.

    Returns:
        points: (M,) sorted array of accepted points in [0,1]
        info: dict with keys 'lambda_max', 'n_proposed', 'n_accepted'
    """
    if isinstance(seed, np.random.Generator):
        rng = seed
    else:
        rng = np.random.default_rng(seed)

    # estimate lambda_max on regular midpoints
    coords = (np.arange(n_grid) + 0.5) / n_grid
    try:
        Lam_grid = np.asarray(intensity_fn(coords))
    except Exception:
        Lam_grid = np.array([float(intensity_fn(x)) for x in coords])
    lam_max = float(np.max(Lam_grid)) if Lam_grid.size > 0 else 0.0
    if lam_max <= 0:
        return np.empty((0,))

    # propose homogeneous Poisson number of points (rate = lam_max over unit length)
    n_prop = int(rng.poisson(lam_max))
    if n_prop == 0:
        return np.empty((0,))

    # sample candidate points
    xs = rng.random(n_prop)

    # evaluate intensity at candidates
    try:
        lam_vals = np.asarray(intensity_fn(xs), dtype=float)
    except Exception:
        lam_vals = np.array([float(intensity_fn(xi)) for xi in xs], dtype=float)

    # guard against negative or NaN values
    lam_vals = np.nan_to_num(lam_vals, nan=0.0)
    lam_vals = np.clip(lam_vals, 0.0, None)

    # thinning
    accept_mask = rng.random(n_prop) < (lam_vals / lam_max)
    accepted = np.sort(xs[accept_mask])

    info = {'lambda_max': lam_max, 'n_proposed': n_prop, 'n_accepted': accepted.size}
    return accepted

def analyze_pos_quadrant(I):
    """
    I: (Nt, Nx) array.
    Returns:
      ft (>=0), P1(ft): 1D per-pixel avg temporal power
      fx_pos (>=0), ft_pos (>=0), F2_q: 2D |F(ft,fx)| for fx,ft >= 0
    """
    Nt, Nx = I.shape

    # 1D per-pixel temporal FFT, ft >= 0
    Ft = np.fft.fftshift(np.fft.fft(I, axis=0), axes=(0))              # (Nt/2+1, Nx)
    P1 = np.mean(np.abs(Ft)**2, axis=1)
    ft = np.fft.fftshift(np.fft.fftfreq(Nt, d=1.0 / Nt))     # normalized frequency (cycles per unit t)

    # Full 2D FFT
    F2 = np.fft.fftshift(np.fft.fft2(I))
    ft_full = np.fft.fftshift(np.fft.fftfreq(Nt, d=1.0 / Nt))
    fx_full = np.fft.fftshift(np.fft.fftfreq(Nx, d=1.0 / Nx))

    # Keep only ft >= 0, fx >= 0
    pos_t = ft_full >= 0
    pos_x = fx_full >= 0

    F2_q = np.abs(F2[np.ix_(pos_t, pos_x)])
    ft_pos = ft_full[pos_t]
    fx_pos = fx_full[pos_x]

    return Ft, ft, P1, fx_full, fx_pos, ft_full, ft_pos, F2, F2_q

def add_case(ax_row, I, title, fluxlam):
    """
    ax_row: [ax_xt, ax_1d, ax_2d]
    """
    nx = I.shape[0]
    ny = I.shape[1]
    Ft, ft_1d, P1, fx, fx_pos, ft, ft_pos, F2, F2_q = analyze_pos_quadrant(I)
    pts = simulate_inhomogeneous_poisson(fluxlam)
    print("npoints 2d", pts.shape)
    # discretize pts into a binary 128x128 plane (rows = t, cols = x)
    pts_plane = np.zeros((I.shape[0], I.shape[1]), dtype=np.uint8)
    if pts.size > 0:
        cols = np.floor(pts[:, 0] * I.shape[1]).astype(int)  # x -> column index
        rows = np.floor(pts[:, 1] * I.shape[0]).astype(int)  # t -> row index
        cols = np.clip(cols, 0, I.shape[1] - 1)
        rows = np.clip(rows, 0, I.shape[0] - 1)
        pts_plane[rows, cols] = 1
    F_quant = np.fft.fftshift(np.fft.fft2(pts_plane))
    F = finufft.nufft2d1(pts[:, 0] * 2 * np.pi, pts[:, 1] * 2 * np.pi, np.ones(pts.shape[0], dtype=np.complex128), n_modes=(64 * 2 + 1, 64 * 2 + 1), isign=-1)
    F_pos = F[:, 64:]

    coordx = (Nx / 4) / Nx
    flux_on_x = lambda t: fluxlam(coordx, t)
    pts_1d = simulate_inhomogeneous_poisson_1d(flux_on_x)
    print("npoints1d", pts_1d.shape)
    F_1d = finufft.nufft1d1(pts_1d * 2 * np.pi, np.ones(pts_1d.shape[0], dtype=np.complex128), n_modes=64 * 2 + 1, isign=-1)
    bound = flux_probing.compute_amplitude_bound(len(pts_1d), 1, 1 - 1 / 64)

    # (1) x–t image
    ax = ax_row[0]
    ax.imshow(I, aspect='equal', cmap='gray',
              extent=[0, 1, 0, 1], origin="lower")   # both x and t in [0,1]
    ax.scatter(pts[:, 0], pts[:, 1], s=1, c="red")
    ax.set_title(f"{title}: x–t")
    ax.set_xlabel("x (unit)")
    ax.set_ylabel("t (unit)")

    # (2) 1D per-pixel temporal spectrum (ft >= 0)
    # ax = ax_row[1]
    # ax.plot(np.arange(0, 65), np.abs(F_1d[64:]))
    # ax.set_xlim(0, ft_1d.max())
    # ax.set_title("1D |F(f_t)|² (avg)")
    # ax.axhline(bound, color='red', linestyle='--', label='CFAR bound')
    # ax.set_xlabel("f_t (cycles/unit t)")
    # ax.set_ylabel("Power")
    # ax.set_yscale("log")

    # (3) 2D joint spectrum (fx,ft >= 0)
    ax = ax_row[1]
    ax.imshow(np.log1p(np.abs(F2)),
              aspect='equal', origin='lower',
              extent=[fx[0], fx[-1], ft[0], ft[-1]],
              cmap='magma')
    ax.set_title("|F(f_x,f_t)| (fx, f_t ≥ 0)")
    ax.set_xlabel("f_x (cycles/unit x)")
    ax.set_ylabel("f_t (cycles/unit t)")

    # (4) nufft
    ax = ax_row[2]
    # # replace 2D axes with a 3D axes in the same subplot slot
    # ss = ax.get_subplotspec()
    # fig = ax.figure
    # ax.remove()
    # ax3d = fig.add_subplot(ss, projection='3d')

    # # prepare grid for plotting
    # FX, FT = np.meshgrid(fx, ft)   # fx -> x axis, ft_pos -> y axis
    # Z = np.abs(F.T)
    # Z = np.nan_to_num(Z)

    # # ensure shapes align (trim if necessary)
    # if Z.shape != FX.shape:
    #     Z = Z[:FX.shape[0], :FX.shape[1]]

    # Z_plot = np.log1p(Z)

    # # plot surface
    # surf = ax3d.plot_surface(FX, FT, Z_plot, cmap='magma', linewidth=0, antialiased=True)
    # ax3d.set_title("NUFFT |F(f_x,f_t)| (3D)")
    # ax3d.set_xlabel("f_x (cycles/unit x)")
    # ax3d.set_ylabel("f_t (cycles/unit t)")
    # ax3d.set_zlabel("log1p|F|")
    # ax3d.view_init(elev=90, azim=-20)

    # add colorbar
    # fig.colorbar(surf, ax=ax3d, shrink=0.6, pad=0.1)
    ax.imshow(np.log1p(np.abs(F.T)),
              aspect='equal', origin='lower',
              extent=[fx[0], fx[-1], ft[0], ft[-1]],
              cmap='magma')
    ax.set_title("NUFFT |F(f_x,f_t)|")
    ax.set_xlabel("f_x (cycles/unit x)")
    ax.set_ylabel("f_t (cycles/unit t)")

    # cfar filtered frequencies from nufft
    ax = ax_row[3]
    bound = flux_probing.compute_amplitude_bound(len(pts), 1, 1 - 1 / (65 * 65)) * 2.0
    print(f"bound: {bound}")
    F_bd = F.T.copy()
    F_bd[np.abs(F.T) < bound] = 0.0
    ax.imshow(np.log1p(np.abs(F_bd)),
              aspect='equal', origin='lower',
              extent=[fx[0], fx[-1], ft[0], ft[-1]],
              cmap='magma')
    ax.set_title("CFAR applied to NUFFT")
    ax.set_xlabel("f_x (cycles/unit x)")
    ax.set_ylabel("f_t (cycles/unit t)")

# ---- Assemble figure: 3 cases x 3 views ----
fig, axes = plt.subplots(3, 4, figsize=(25, 15))

add_case(axes[0, :], I_flicker, "Only flicker", case1)
add_case(axes[1, :], I_motion,  "Only motion", case2)
add_case(axes[2, :], I_both,    "Motion + flicker", case3)

plt.tight_layout()
plt.show()


In [ ]:
pts = simulate_inhomogeneous_poisson(case1)

In [ ]:
plt.scatter(pts[:, 0], pts[:, 1], s=1)
plt.xlim(0, 1)
plt.ylim(0, 1)

In [ ]:
F = finufft.nufft2d1(pts[:, 0], pts[:, 1], np.ones(pts.shape[0], dtype=np.complex128), n_modes=(64 * 2 + 1, 64 * 2 + 1))

# bit2bit stuff

In [ ]:
data = imread("output/results/quanta/elevator/bit2bit/inference.tif")
data100 = data[::100]

In [ ]:
gamma = data100 ** (1 / 4)
gamma = np.rot90(gamma, k=1, axes=(1, 2))

In [ ]:
ihpp_data.to_video(gamma, "output/results/quanta/elevator/bit2bit/inference_fps10k.avi", playback_fps=30, cmap="grey")
utils.avi_to_mov("output/results/quanta/elevator/bit2bit/inference_fps10k.avi")

# recreate gamma video

In [ ]:
import zarr
from pathlib import Path
from sptflux import ihpp_data
import numpy as np
folder = Path("output/results/quanta/gun-2s/thin1.000/fps100000/gun-2s-keep1.000-playfps100000-chunks5000x32x32-hops2500x16x16-res512x512-bins10000x512x512-sliceall-window_hann-cfar1.0")
path = folder / "reconinfo.zarr"
d = zarr.open_group(path, mode="r")

In [ ]:
flux = d["flux"][:]
Nt, Ny, Nx = flux.shape
hop_t, hop_h, hop_w = 2500, 16, 16
nframes, binvidH, binvidW = 10000, 512, 512
exclude_t = int(Nt * hop_t / nframes)
exclude_y = int(Ny * hop_h / binvidH)
exclude_x = int(Nx * hop_w / binvidW)
maxflux = np.max(flux[exclude_t:Nt-exclude_t, exclude_y:Ny-exclude_y, exclude_x:Nx-exclude_x])
maxgamma = maxflux ** (1 / 2.2)
gamma = flux ** (1 / 2.2)

In [ ]:
ihpp_data.to_video(
    gamma, folder / "gamma-modified.avi", playback_fps=30, cmap="grey", vmax=maxgamma
)
# ihpp_data.to_video(
#     gamma, folder / "gammaframes-1-3", playback_fps=30, cmap="grey", format="png"
# )
# utils.avi_to_mov(folder / "gamma-1-3.avi")

# monkey flux

In [ ]:
path = "output/results/bit2bit/Monkey_002us_100000fs_dim/Monkey_002us_100000fs_dim-keep1.000-res512-playfps4032.0-bins1000x512x512-chunksNonex8x8-pad0t4xy-windowNone-cfar2.0/reconinfo.zarr"
d = zarr.open_group(path)

In [ ]:
fluxfunc = d["flux"][:]

In [ ]:
y = 50
x = 450
pixel = fluxfunc[:, y, x]

In [ ]:
pixel.shape

In [ ]:
plt.plot(np.linspace(0, T_exp, 1000), pixel)

# why does bit2bit write unnormalized tiffs

In [ ]:
path = "output/results/bit2bit/Monkey_002us_100000fs_dim/inference.tif"
from PIL import Image, ImageSequence
img = Image.open(path)
frames = [frame.copy() for frame in ImageSequence.Iterator(img)]
frames = np.array(frames)

In [ ]:
ihpp_data.to_video(frames, "output/results/bit2bit/Monkey_002us_100000fs_dim/inference.avi", playback_fps=30, cmap="grey")

# 1d uwb thing i should do

In [ ]:
quantapath = "data/bit2bit-data/Monkey_002us_100000fs_dim.zarr"
quantaframes, _, _ = ihpp_data.read_quanta_zarr(quantapath, load_data=False)
quantaframes = quantaframes[:]
nframes = quantaframes.shape[0]
fps = 100_000
T_exp = nframes / fps
pixel_timeseries = ihpp_data.timeseries_from_binframes(quantaframes, T_exp=T_exp)

In [ ]:
ft_max = 50_000 * T_exp
fy = np.arange(0, ft_max + 1, 1)
Nt = 500
tt = np.linspace(0, 1, Nt)
rate_uwb, F_all, bound_uwb = flux_probing.spt_ihpp_uwb1d([[pixel_timeseries[200][300]]], tt=tt, ft=fy, T_exp=T_exp, cfarscale=2.0)

In [ ]:
from sptflux import uwb_utils
tslice = (0, 50_000, 1 / T_exp)
y = 50
x = 450
t = pixel_timeseries[y][x]
F, fy = uwb_utils.probe_frequencies_sweep(t, T_exp, tslice, return_freqs=True)

In [ ]:
bound = uwb_utils.compute_amplitude_bound(t, T_exp, 1 - 1 / len(fy))

In [ ]:
1/T_exp

In [ ]:
plt.plot(fy, np.abs(F))
# plt.scatter(ft, np.abs(F))
plt.axhline(bound, color='red', linestyle='--')
plt.xlim(0, 200)

In [ ]:
len(t)

In [ ]:
tt = np.linspace(0, T_exp, 1000)
F[bound >= np.abs(F)] = 0
rate = uwb_utils.reconstruct_rate_function(tt, fy, F)

In [ ]:
fluxfunc = np.clip(rate.real, a_min=0, a_max=None)

In [ ]:
plt.plot(tt, fluxfunc)
plt.xlim(0, 0.01)

In [ ]:
plt.plot(np.linspace(0, T_exp, 1000), pixel)
plt.xlim(0, 0.1)

In [ ]:
plt.imshow(np.sum(quantaframes, axis=0))

In [ ]:
np.sum(quantaframes[:, 200, 300])

In [ ]:
from 

In [ ]:
plt.plot(fy / T_exp, np.abs(F_all[0][0]))

# data conversion stuff

In [ ]:
paths = [
    "gamma-uwb3d-chunked-quanta_guitar-keep0.004-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.avi",
    "gamma-uwb3d-chunked-quanta_guitar-keep0.008-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.avi",
    "gamma-uwb3d-chunked-quanta_guitar-keep0.016-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.avi",
    "gamma-uwb3d-chunked-quanta_guitar-keep0.031-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.avi",
    "gamma-uwb3d-chunked-quanta_guitar-keep0.062-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.avi",
    "gamma-uwb3d-chunked-quanta_guitar-keep0.125-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.avi",
    "gamma-uwb3d-chunked-quanta_guitar-keep0.250-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.avi",
    "gamma-uwb3d-chunked-quanta_guitar-keep0.500-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.avi",
    "gamma-uwb3d-chunked-quanta_guitar-keep1.000-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.avi",
]
paths = [Path("output/chunk-thin-tests/guitar/") / p for p in paths]

In [ ]:
for path in paths:
    utils.avi_to_mov(path)

In [ ]:
paths = [
    "uwb3d_reconinfo-quanta_guitar-keep0.004-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.zarr",
    "uwb3d_reconinfo-quanta_guitar-keep0.008-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.zarr",
    "uwb3d_reconinfo-quanta_guitar-keep0.016-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.zarr",
    "uwb3d_reconinfo-quanta_guitar-keep0.031-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.zarr",
    "uwb3d_reconinfo-quanta_guitar-keep0.062-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.zarr",
    "uwb3d_reconinfo-quanta_guitar-keep0.125-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.zarr",
    "uwb3d_reconinfo-quanta_guitar-keep0.250-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.zarr",
    "uwb3d_reconinfo-quanta_guitar-keep0.500-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.zarr",
    "uwb3d_reconinfo-quanta_guitar-keep1.000-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.zarr"
]
paths = [Path("output/chunk-thin-tests/guitar/") / p for p in paths]

In [ ]:
paths[0].stem

In [ ]:
for path in paths:
    data = zarr.open_group(path, mode="r")
    gamma = data["gamma"][:]
    ihpp_data.to_video(
        gamma, path.parent / path.stem, fileformat="png", cmap="grey"
    )

# some denoising

In [ ]:
import numpy as np
from skimage import io
from skimage.restoration import denoise_nl_means, estimate_sigma
from skimage.util import random_noise

In [ ]:
# Load an image (or create a noisy one for demonstration)
# For example:
# image = io.imread('your_image.png', as_gray=True) 
# Add some Gaussian noise
image = np.zeros((100, 100))
image[20:80, 20:80] = 1
noisy_image = random_noise(image, mode='gaussian', var=0.01)

# Estimate the noise standard deviation
sigma_est = np.mean(estimate_sigma(noisy_image, channel_axis=None))

# Perform Non-Local Means denoising
denoised_image = denoise_nl_means(noisy_image, h=0.8 * sigma_est,
                                  fast_mode=True, patch_size=7,
                                  patch_distance=15, channel_axis=None)

# You can then display or save the denoised_image
# io.imshow(denoised_image)
# io.show()

In [ ]:
path = "output/uwb3d_reconinfo-quanta_gun-2s-keep1.000-res512-frames0_100000-bins500x512x512-chunks2000x4x4-pad1000t2xy-windowNone.zarr"
data = zarr.open_group(path, mode="r")
fluxfunc = data["flux"][:]

In [ ]:
denoised = []
for frame in tqdm(fluxfunc[100:150]):
    noisy_image = frame
    # Estimate the noise standard deviation
    sigma_est = np.mean(estimate_sigma(noisy_image, channel_axis=None))

    # Perform Non-Local Means denoising
    denoised_image = denoise_nl_means(noisy_image, h=0.8 * sigma_est,
                                    fast_mode=True, patch_size=5,
                                    patch_distance=10, channel_axis=None)
    denoised.append(denoised_image)
denoised = np.array(denoised)
ihpp_data.to_video(
    denoised ** (1 / 2.2), "output/uwb3d_denoised_gamma.avi", playback_fps=30, cmap="grey"
)

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(denoised[15] ** (1 / 2.2), cmap="gray")

In [ ]:
path = "output/chunk-thin-tests/gun-2s/uwb3d_reconinfo-quanta_gun-2s-keep1.000-res512-frames0_100000-bins500x512x512-chunksNonex6x6-pad0t3xy-windowNone-cfar2.0.zarr"
data = zarr.open_group(path, mode="r")
fluxfunc = data["flux"][:]

In [ ]:
gamma15 = fluxfunc ** (1 / 1.5)

In [ ]:
ihpp_data.to_video(
    gamma15, "output/uwb3d_gamma_1.5_exponent.avi", playback_fps=30, cmap="grey"
)

# loading bit2bit

In [ ]:
from spadtools import SPADFile, SPADData

In [ ]:
dir_path = Path("data/bit2bit-data/Resolution_target_drill_002us_100000fs")
dataset = SPADFile(dir_path, load_data=True)

In [ ]:
frames = dataset.combine()

In [ ]:
frames.shape

In [ ]:
plt.imshow(frames[500], origin='lower')

# binary video itself

In [ ]:
from pathlib import Path
import numpy as np
from sptflux import ihpp_data, config
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
datapaths = [
    # "/Volumes/CrucialX10/uwb3d-files/data/quanta/Feb27_blender_acq15_4351ppps.zarr"
    # "/Volumes/CrucialX10/uwb3d-files/data/quanta/Feb28_balloonBounce_acq0_2221ppps.zarr",
    "data/phantom-synth/upsampled/cavitation1_rife0_linear1_ppp0.1.zarr",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb28_balloonBounce_acq0_2221ppps/thin0.050/fps100000/Feb28_balloonBounce_acq0_2221ppps-slice19600_29500-chunks8000x100x100-hops2000x25x25-bins100000x512x512-keep0.050-playfps100000-window_hann-alpha0.00010/thinned_frames.zarr",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb28_balloonBounce_acq0_2221ppps/thin0.020/fps100000/Feb28_balloonBounce_acq0_2221ppps-slice19600_29500-chunks8000x100x100-hops2000x25x25-bins100000x512x512-keep0.020-playfps100000-window_hann-alpha0.00010/thinned_frames.zarr",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb28_balloonBounce_acq0_2221ppps/thin0.010/fps100000/Feb28_balloonBounce_acq0_2221ppps-slice19600_29500-chunks8000x100x100-hops2000x25x25-bins100000x512x512-keep0.010-playfps100000-window_hann-alpha0.00010/thinned_frames.zarr"
]

thinlevelstr = "thin1.000"
for datapath in datapaths:
    datapath = Path(datapath)
    dataname = datapath.stem
    frames, metadata = ihpp_data.read_quanta_auto(datapath, load_data=False)
    frames_sub = frames[:]
    # ihpp_data.to_video(frames_sub, config.DATA_DIR / f"binary-vids/binary-fps1000-{dataname}-{thinlevelstr}.mp4", playback_fps=30, cmap="gray")
    # ihpp_data.to_video(frames_sub, config.DATA_DIR / f"binary-vids/binary-fps1000-{dataname}-{thinlevelstr}", cmap="gray")
    # frames_sub = frames[19600:29500:10]
    # ihpp_data.to_video(frames_sub, config.DATA_DIR / f"binary-vids/binary-fps10k-{dataname}-{thinlevelstr}.mp4", playback_fps=30, cmap="gray")
    # frames_sub = frames[19600+1179:19600+2445]
    # ihpp_data.to_video(frames_sub, config.DATA_DIR / f"binary-vids/binary-fps100k-{dataname}-{thinlevelstr}.mp4", playback_fps=30, cmap="gray")
    # frames_sub = frames[65000:75000:50]
    # ihpp_data.to_video(frames_sub, config.DATA_DIR / f"binary-vids/binary-blencderidk-{dataname}.mp4", playback_fps=30, cmap="gray")
    # frames_sub = frames[1000:1500]
    # ihpp_data.to_video(frames_sub, config.DATA_DIR / f"binary-vids/{dataname}-contiguous.mp4", playback_fps=30, cmap="gray")
    ihpp_data.to_video(frames_sub, f"data/binary-vids/{dataname}-contiguous.mp4", playback_fps=30, cmap="gray")
    # T_exp = frames.shape[0] / 100_000
    # npoints = np.count_nonzero(frames)
    # npts_persec_perpix = np.count_nonzero(frames, axis=0) / T_exp
    # # statistics
    # avg_pts_persec_perpixel = npoints / (T_exp * frames.shape[1] * frames.shape[2])
    # median_pts_persec_perpixel = np.median(npts_persec_perpix)
    # print(f"{dataname}: {npoints} points, {T_exp:.2f}s exp, avg {avg_pts_persec_perpixel:.4f} pts/s/pix, median {median_pts_persec_perpixel:.4f} pts/s/pix")
    # stats_dir = config.DATA_DIR / "binary-vids"
    # stats_dir.mkdir(parents=True, exist_ok=True)
    # stats_path = stats_dir / f"{dataname}_stats.txt"

    # with open(stats_path, "w") as f:
    #     f.write(f"dataname: {dataname}\n")
    #     f.write(f"num_frames: {frames.shape[0]}\n")
    #     f.write(f"height: {frames.shape[1]}\n")
    #     f.write(f"width: {frames.shape[2]}\n")
    #     f.write(f"T_exp_s: {T_exp:.6f}\n")
    #     f.write(f"npoints: {npoints}\n")
    #     f.write(f"avg_pts_persec_perpixel: {avg_pts_persec_perpixel:.6f}\n")
    #     f.write(f"median_pts_persec_perpixel: {median_pts_persec_perpixel:.6f}\n")
    # print(f"saved stats: {stats_path}")
    del frames, frames_sub

In [ ]:
dataname = "fan.zarr"
# dataname = "Feb29_balloonbounce_acq0_2875ppps.zarr"

data_path = config.DATA_DIR / f"quanta/{dataname}"
data_name = data_path.stem
frames, metadata = ihpp_data.read_quanta_auto(data_path, load_data=False)
frames = frames[:]

In [ ]:
i = 20000
exposure = 10000
asdf = np.sum(frames[i:i+exposure], axis=0)
plt.imshow(asdf)

In [ ]:
plt.imshow(asdf)
plt.scatter([200], [200], c='red', s=10)

In [ ]:
# tslice = slice(0, 30000)
tslice = slice(80000, 100000)
tseries = np.sum(frames[tslice, 196:204, 196:204], axis=(1,2))
F = np.fft.rfft(tseries)
plt.plot(tseries)

In [ ]:
plt.plot(np.fft.rfftfreq(len(tseries) , 1/100000), np.abs(F))
plt.xlim(0, 150)
plt.axvline(23, color='red', linestyle='--')
plt.axvline(40, color='red', linestyle='--')
# log scale
# plt.yscale('log')

In [ ]:
dcr_prob = ihpp_data.prob_from_dcr(25, 100000)
frames_thin = ihpp_data.thin_frames_uniform_da(frames, keep_prob=1/20, dcr_prob=dcr_prob, seed=0)

In [ ]:
out = []
interval = 1000
for i in range(len(frames_thin) // interval):
    chunk = frames_thin[i*interval:(i+1)*interval]
    chunk_sum = np.sum(chunk, axis=0)
    out.append(chunk_sum)
out = np.array(out)

In [ ]:
ihpp_data.to_video(out, "output/guitar10kwhat.mp4", gamma=1/2.2, playback_fps=30, cmap="gray")

In [ ]:
# bright spot
# x = 300
# y = 200
# wall pixel
x = 340
y = 290
plt.imshow(np.sum(frames[0:1000], axis=0))
plt.scatter(x, y, s=2, c="red")

In [ ]:
rad = 2
superpix = frames[:, y-rad:y+rad+1, x-rad:x+rad+1]
superpix_sum = superpix.sum(axis=(1, 2))
abssum = np.zeros(superpix.shape[0] // 2 + 1)
for i in range(superpix.shape[1]):
    for j in range(superpix.shape[2]):
        abssum += np.abs(np.fft.rfft(superpix[:, i, j]))
plt.plot(abssum)
plt.yscale("log")

In [ ]:
plt.plot(abssum)
plt.yscale("log")

In [ ]:
i = 70000
radius = 50
plt.imshow(np.sum(frames[i-50:i+50+1, :, :], axis=0) ** (1/2.2))

In [ ]:
fps = 100000
np.count_nonzero(frames) / frames.shape[1] / frames.shape[2] / (frames.shape[0] / fps)

In [ ]:
plt.imshow(frames[0])
# y = 30
# x = 450
y = 190
x = 468
plt.scatter([x], [y], c="r")

In [ ]:
y = 190
x = 468

In [ ]:
patch_rad = 4
y = 400
x = 477
patch = frames[:, y-patch_rad:y+patch_rad+1, x-patch_rad:x+patch_rad+1]
patchsum = np.sum(patch, axis=(1,2))
F = np.fft.rfft(patchsum)

In [ ]:
plt.plot(np.abs(F))
plt.yscale("log")

In [ ]:
Fsum = np.zeros(patch.shape[0] // 2 +1)
for i in range(patch.shape[1]):
    for j in range(patch.shape[2]):
        Fsum += np.abs(np.fft.rfft(patch[:, i, j]))

In [ ]:
plt.plot(np.abs(Fsum))
plt.yscale("log")

In [ ]:
from sptflux import utils

In [ ]:
path = "/Users/yan569/Downloads/gamma-newteaser.mp4"
vid = utils.read_video(path)

In [ ]:
import cv2
vid = np.array([cv2.cvtColor(fr, cv2.COLOR_RGB2GRAY) for fr in vid])

In [ ]:
plt.imshow(vid[0])
y = 400
x = 477
plt.scatter([x], [y], c="r", s=1)

In [ ]:
plt.plot(vid[:, y, x])
plt.xlim(0, 1000)
# plt.ylim(80, 105)

In [ ]:
20000*500/100000

In [ ]:
# Subsample every x frames with x/2-frame surrounding context
frame_indices = np.arange(0, frames.shape[0], 1000)
frames_aggregated = []

for idx in tqdm(frame_indices):
    # Sum x/2 frames before and after (x frames total + center)
    start = max(0, idx - 500)
    end = min(frames.shape[0], idx + 501)
    aggregated_frame = np.sum(frames[start:end], axis=0)
    frames_aggregated.append(aggregated_frame)

frames_aggregated = np.array(frames_aggregated)

# Convert to video
ihpp_data.to_video(frames_aggregated, config.DATA_DIR / f"binary-vids/{data_name}-aggregated.mp4", playback_fps=30, cmap="gray", quantile=0.99)

In [ ]:
frames_sub = frames[::100]
ihpp_data.to_video(frames_sub, config.DATA_DIR / f"binary-vids/{data_name}.mp4", playback_fps=30, cmap="gray")

In [ ]:
ihpp_data.to_video(frames_aggregated, config.DATA_DIR / f"binary-vids/{data_name}-aggregated.mp4", playback_fps=30, cmap="gray", use_quantile=True)

In [ ]:
i = 90000
plt.imshow(np.sum(frames[i:i+1000], axis=0), cmap="grey")
plt.scatter([210], [68], s=1, c="red")
plt.scatter([267], [72], s=1, c="blue")

In [ ]:
ts1 = frames[:, 68, 210]
ts2 = frames[:, 72, 267]
F = np.fft.rfft(ts1)
F2 = np.fft.rfft(ts2)

In [ ]:
bd1 = flux_probing.compute_amplitude_bound(np.count_nonzero(ts1), 1, 1/len(F))
bd2 = flux_probing.compute_amplitude_bound(np.count_nonzero(ts2), 1, 1/len(F2))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
axs[0].plot(np.abs(F))
axs[0].set_yscale("log")
axs[0].set_title(f"FFT of red point (photon count {np.count_nonzero(ts1)})")
axs[0].set_xlabel("Hz")
axs[0].set_ylabel("|F|")
# axs[0].set_xlim(0, 5000)
axs[0].axhline(bd1, color='red', linestyle='--', label='CFAR bound')
axs[1].plot(np.abs(F2))
axs[1].set_yscale("log")
axs[1].set_title(f"FFT of blue point (photon count {np.count_nonzero(ts2)})")
axs[1].set_xlabel("Hz")
axs[1].set_ylabel("|F|")
# axs[1].set_xlim(0, 5000)
axs[1].axhline(bd2, color='red', linestyle='--', label='CFAR bound')
plt.tight_layout()
plt.show()

In [ ]:
plt.plot(np.abs(F))
# log scale
plt.yscale("log")
plt.xlim(0, 4000)
# vertical line at 800 hz
harm = 700
plt.axvline(harm, color='red', linestyle='--')
plt.axvline(harm * 2, color='red', linestyle='--')
plt.axvline(harm * 3, color='red', linestyle='--')
plt.axvline(harm * 4, color='red', linestyle='--')

In [ ]:
binsum = np.sum(frames, axis=0)

In [ ]:
plt.figure(figsize=(15, 15))
plt.imshow(binsum[:300, :512])
plt.colorbar()
# make figsize bigger

In [ ]:
data = zarr.open_group("data/quanta/guitar.zarr")

In [ ]:
frames = data["quantaframes"]
frame = frames[6443]

In [ ]:
import cv2
cv2.imwrite("output/guitar-frame6443.png", frame * 255)

In [ ]:
thinned = ihpp_data.thin_frames_uniform(np.array([frame]), keep_prob=1/32)[0]
cv2.imwrite("output/guitar-frame6443-thinned-1-32.png", thinned * 255)

In [ ]:
import numpy as np

In [ ]:
a = np.load("data/quanta/lighter.npy")

In [ ]:
import matplotlib.pyplot as plt
A = np.unpackbits(a, axis=2)[:, ::-1, ::-1]

In [ ]:
A.shape

In [ ]:
plt.imshow(A[167000, :, 16:])

In [ ]:
288*512

In [ ]:
from sptflux import ihpp_data
plt.imshow(A[167000, :, 16:])
ihpp_data.write_quantaframes_zarr("data/quanta/lighter.zarr", A.astype(np.uint8), T_exp=2*(167000 / A.shape[0]))

In [ ]:
2*(167000 / A.shape[0])

In [ ]:
1 / (2*(167000 / A.shape[0]) / 167000)

In [ ]:
1/1.05e-5

# sparse arrays or something

In [ ]:
import sparse

s = sparse.random((5, 5), density=0.3)

In [ ]:
import zarr
d = zarr.open_group("data/quanta/fan.zarr", mode="r")

In [ ]:
import numpy as np
n = np.count_nonzero(d["quantaframes"])

In [ ]:
import dask.array as da

In [ ]:
n = da.count_nonzero(d["quantaframes"]).compute()

In [ ]:
n / d["quantaframes"].size

In [ ]:
type(s)

In [ ]:
isinstance(s, sparse.COO)

In [ ]:
s.coords

In [ ]:
s.data

In [ ]:
from sptflux import utils

In [ ]:
g = utils.save_sparse_to_zarr(s, "data/test.zarr")

In [ ]:
g.attrs["fill_value"]

In [ ]:
a = np.concat([np.zeros((6, 0), dtype="int64"), np.array([[1],[2],[1],[2],[1],[2]])], axis=1)

In [ ]:
a = np.concat([a, np.array([[3, 4, 5],[3, 4, 5],[3, 4, 5],[3, 4, 5],[3, 4, 5],[3, 4, 5]])], axis=1)

In [ ]:
a

In [ ]:
frames = d["quantaframes"][:]

In [ ]:
%timeit np.count_nonzero(frames)

In [ ]:
%timeit np.sum(frames)

In [ ]:
[1,2,3,4][slice(None)]

# videos or something

In [ ]:
import cv2
import os
from pathlib import Path
import numpy as np
from tqdm import tqdm

input_video_path = "data/youtube-data/wineglass-fall.mp4"

cap = cv2.VideoCapture(input_video_path)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

In [ ]:
frame_count

In [ ]:
ret, frame = cap.read()

In [ ]:
frame.shape

In [ ]:
4320/7680

In [ ]:
512 * 4320/7680

# convert flux to float32 because they take too much space

In [ ]:
import os
import zarr
import numpy as np

def convert_flux_to_float32(root_directory):
    """
    Recursively finds 'reconinfo.zarr' folders and converts the 
    'flux' array to float32.
    """
    for root, dirs, files in os.walk(root_directory):
        for dirname in dirs:
            if dirname == 'reconinfo.zarr':
                zarr_path = os.path.join(root, dirname)
                print(f"Processing: {zarr_path}")
                
                try:
                    # Open the zarr group in append mode
                    store = zarr.open_group(zarr_path, mode='a')
                    
                    if 'flux' in store:
                        # 1. Load the existing data
                        # Note: This loads the entire array into memory.
                        flux_data = store['flux'][:]
                        T, H, W = flux_data.shape
                        
                        # 2. Check if conversion is actually needed
                        if flux_data.dtype == np.float32:
                            print(f"  - 'flux' is already float32. Skipping.")
                            continue
                        
                        # 3. Convert to float32
                        flux_converted = flux_data.astype(np.float32)
                        
                        # 4. Overwrite the existing 'flux' array
                        # We delete the old array and create a new one with the same name
                        del store['flux']
                        store.create_array('flux', data=flux_converted, overwrite=True, chunks=(400, H, W))
                        
                        print(f"  - Successfully converted 'flux' to float32.")
                    else:
                        print(f"  - Warning: 'flux' array not found in {zarr_path}")
                        
                except Exception as e:
                    print(f"  - Error processing {zarr_path}: {e}")

if __name__ == "__main__":
    # Replace with the actual path to your top-level directory
    # target_dir = "output/results/spike/"
    convert_flux_to_float32(target_dir)

In [ ]:
import os
import zarr
import numpy as np
from tqdm import tqdm

FLOAT16_MAX = np.float16(65504.0).item()

def _float16_suitability_check(
    arr,
    *,
    n_samples=16,
    t_block=64,
    abs_err_p99_tol=1.0,
    rel_err_p99_tol=1e-3,
    max_abs_err_tol=None,
    rng_seed=0,
):
    """
    Decide if arr can be represented as float16 with acceptable error.

    Criteria (you can tune):
      - reject if any sampled max > 65504 (float16 overflow risk)
      - compute float16 roundtrip error on sampled blocks:
            x32 -> x16 -> x32
      - accept if:
            p99(abs_err) <= abs_err_p99_tol
        AND p99(rel_err) <= rel_err_p99_tol
        AND (if provided) max_abs_err <= max_abs_err_tol
    """
    rng = np.random.default_rng(rng_seed)

    T, H, W = arr.shape
    if T == 0:
        return False, {"reason": "empty"}

    starts = np.arange(0, T, t_block)
    abs_errs = []
    rel_errs = []
    max_vals = []

    for s in tqdm(starts):
        t0 = int(s)
        t1 = min(T, t0 + t_block)
        block = arr[t0:t1, :, :].astype(np.float32, copy=False)

        # Track max to catch float16 overflow
        bmax = float(np.max(block)) if block.size else 0.0
        max_vals.append(bmax)
        if bmax > FLOAT16_MAX:
            return False, {
                "reason": "overflow_risk",
                "sampled_max": bmax,
                "float16_max": FLOAT16_MAX,
            }

        # float16 roundtrip error
        block16 = block.astype(np.float16)
        block_rt = block16.astype(np.float32)

        abs_err = np.abs(block_rt - block)
        abs_errs.append(abs_err.reshape(-1))

        # relative error: abs_err / max(x, eps)
        # since non-negative, this is well-behaved; ignore near-zeros via eps
        eps = 1e-6
        rel_err = abs_err / np.maximum(block, eps)
        rel_errs.append(rel_err.reshape(-1))

    abs_errs = np.concatenate(abs_errs) if abs_errs else np.array([np.inf], dtype=np.float32)
    rel_errs = np.concatenate(rel_errs) if rel_errs else np.array([np.inf], dtype=np.float32)

    stats = {
        "sampled_max": float(np.max(max_vals)) if max_vals else 0.0,
        "abs_err_p99": float(np.percentile(abs_errs, 99)),
        "abs_err_max": float(np.max(abs_errs)),
        "rel_err_p99": float(np.percentile(rel_errs, 99)),
        "rel_err_max": float(np.max(rel_errs)),
    }

    ok = (stats["abs_err_p99"] <= abs_err_p99_tol) and (stats["rel_err_p99"] <= rel_err_p99_tol)
    if max_abs_err_tol is not None:
        ok = ok and (stats["abs_err_max"] <= max_abs_err_tol)

    return ok, stats


def _convert_array_to_float16_in_blocks(group, name="flux", out_name="flux", t_block=64):
    src = group[name]
    T, H, W = src.shape
    chunks = getattr(src, "chunks", (min(400, T), H, W))

    dst = group.create_array(
        out_name,
        shape=src.shape,
        chunks=chunks,
        dtype=np.float16,
        overwrite=True,
    )

    for t0 in range(0, T, t_block):
        t1 = min(T, t0 + t_block)
        dst[t0:t1] = src[t0:t1].astype(np.float16)

    return dst


def convert_flux_to_float32_or_float16(
    root_directory,
    *,
    try_float16=True,
    # suitability thresholds (tune these)
    abs_err_p99_tol=1.0,     # e.g. allow p99 absolute error up to 1 photon
    rel_err_p99_tol=1e-3,    # e.g. allow p99 relative error up to 0.1%
    max_abs_err_tol=None,    # optionally cap worst-case absolute error
    suitability_samples=16,
    suitability_t_block=64,
    convert_t_block=64,
    verbose=True,
):
    """
    Recursively finds 'reconinfo.zarr' folders and converts 'flux' to float32,
    then (optionally) further converts to float16 if suitability checks pass.
    """
    for root, dirs, files in os.walk(root_directory):
        for dirname in dirs:
            if dirname != "reconinfo.zarr":
                continue

            zarr_path = os.path.join(root, dirname)
            if verbose:
                print(f"Processing: {zarr_path}")

            try:
                group = zarr.open_group(zarr_path, mode="a")

                if "flux" not in group:
                    if verbose:
                        print(f"  - Warning: 'flux' array not found")
                    continue

                flux = group["flux"]

                # Step 1: ensure float32
                if flux.dtype != np.float32:
                    if verbose:
                        print(f"  - Converting 'flux' {flux.dtype} -> float32 (block-wise)")

                    T, H, W = flux.shape
                    chunks = getattr(flux, "chunks", (min(400, T), H, W))

                    # create new array directly as 'flux_f32'
                    flux_f32 = group.create_array(
                        "flux_f32",
                        shape=flux.shape,
                        chunks=chunks,
                        dtype=np.float32,
                        overwrite=True,
                    )

                    for t0 in range(0, T, convert_t_block):
                        t1 = min(T, t0 + convert_t_block)
                        flux_f32[t0:t1] = flux[t0:t1].astype(np.float32, copy=False)

                    # remove old and replace
                    group["flux"] = flux_f32
                    del group["flux_f32"]

                else:
                    if verbose:
                        print(f"  - 'flux' already float32")

                # Step 2: optionally attempt float16
                if not try_float16:
                    continue

                ok, stats = _float16_suitability_check(
                    flux,
                    n_samples=suitability_samples,
                    t_block=suitability_t_block,
                    abs_err_p99_tol=abs_err_p99_tol,
                    rel_err_p99_tol=rel_err_p99_tol,
                    max_abs_err_tol=max_abs_err_tol,
                )

                if verbose:
                    if ok:
                        print("  - float16 suitability: PASS")
                    else:
                        print("  - float16 suitability: FAIL")
                    print(f"    stats: {stats}")

                if not ok:
                    continue

                if verbose:
                    print("  - Converting 'flux' float32 -> float16 (block-wise overwrite)")

                # Convert in-place (delete + recreate)
                # If you’d rather keep both, change out_name to e.g. "flux_f16".
                _convert_array_to_float16_in_blocks(
                    group, name="flux", out_name="flux_f16", t_block=convert_t_block
                )

                group["flux"] = group["flux_f16"]
                del group["flux_f16"]

                if verbose:
                    print("  - Successfully converted 'flux' to float16.")

            except Exception as e:
                print(f"  - Error processing {zarr_path}: {e}")


if __name__ == "__main__":
    # target_dir = "output/results/quanta/"
    convert_flux_to_float32_or_float16(
        target_dir,
        try_float16=True,
        abs_err_p99_tol=1.0,    # tune: acceptable photon absolute error at p99
        rel_err_p99_tol=1e-2,   # tune: acceptable relative error at p99
        max_abs_err_tol=None,   # e.g. 4.0 if you want a hard cap
        suitability_samples=16,
        suitability_t_block=1000,
        convert_t_block=1000,
        verbose=True,
    )

In [ ]:
asdf = zarr.open_group("output/results/quanta/gun-2s/thin1.000/fps1000/gun-2s-keep1.000-res512x512-playfps1000-bins1000x512x512-chunks20000t32x32-hops10000x16x16-window=hann-cfar2.0/reconinfo.zarr")

In [ ]:
fluxarr = asdf["flux"][:]

In [ ]:
plt.imshow(fluxarr[400, :, :])
plt.colorbar()

# reading flux results

In [ ]:
import zarr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sptflux import ihpp_data, utils, config
from pathlib import Path

In [ ]:
path = Path("/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb28_balloonBounce_acq0_2221ppps/thin1.000/fps1000/Feb28_balloonBounce_acq0_2221ppps-slicefull-chunks5000x64x64-hops1250x16x16-bins1000x512x512-keep1.000-playfps1000-window_hann-alpha0.00100")
path = path / "reconinfo.zarr"

In [ ]:
g = zarr.open_group(path)
flux = g["flux"][:]

In [ ]:
utils.to_video(flux, path.parent / "gamma-14", gamma=1/4, cmap="grey")

## auto converting and reading (not analysis)

In [ ]:
path = Path("/Volumes/CrucialX10/uwb3d-files/output/results/spike/rotation2x/fps5000/rotation2x-slicefull-chunks1000x64x64-hops250x16x16-bins625x250x400-keep1.000-playfps5000-window_hann-alpha0.00010/gamma")
frames = utils.read_imgdir(path, grayscale=True)
ihpp_data.to_video(frames[:200], path.parent / "rotation2x-website-gamma.mp4", playback_fps=30, cmap="gray")

In [ ]:
# b2b inference save xd
respath = Path("/Volumes/CrucialX10/uwb3d-files/output/results/bit2bit/balloon-laser-acq00000-thin1.000/inference.zarr")
flux_zarr = zarr.open_array(respath)
flux = flux_zarr[:]

frameoffset = flux_zarr.attrs["frame_start"]
framenames = [f"frame_{(i + frameoffset):05d}" for i in range(len(flux))]
# ihpp_data.to_video(flux, respath.parent / "gamma", gamma=1/2.2, cmap="grey", vmin=0, framenames=framenames)
# ihpp_data.to_video(flux, respath.parent / "gamma14.mp4", gamma=1/4, playback_fps=30, cmap="grey", vmin=0)
ihpp_data.to_video(flux, respath.parent / "gamma14", gamma=1/4, cmap="grey", vmin=0, framenames=framenames)

In [ ]:
ihpp_data.to_video(flux, respath.parent / "gamma14.mp4", gamma=1/4, cmap="grey", vmin=0, playback_fps=30)

In [ ]:
ihpp_data.to_video(flux, respath.parent / "gamma-fixed.mp4", gamma=1/2.2, playback_fps=30, cmap="grey", vmin=0)

In [ ]:
ihpp_data.to_video(flux, respath.parent / "gamma", gamma=1/2.2, fileformat="png", cmap="grey", vmin=0)

In [ ]:
ihpp_data.to_video(flux, respath.parent / "gamma14.mp4", gamma=1/4, playback_fps=30, cmap="grey", vmin=0)

In [ ]:
# uwb 3d + 1d save
import zarr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sptflux import ihpp_data

resdirs = [
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb29_balloonbounce_acq0_2875ppps/"
    # "/Volumes/CrucialX10/uwb3d-files/output/results/uwb1d/quanta/Feb27_blender_acq15_4351ppps/thin0.100",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Monkey/thin1.000/fps100000/Monkey-slice0_10000-chunks1000x64x64-hops250x16x16-bins24800x512x512-keep1.000-playfps100000-window_hann-alpha0.00100",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Monkey/thin0.100/fps100000/Monkey-slice0_10000-chunks1000x64x64-hops250x16x16-bins24800x512x512-keep0.100-playfps100000-window_hann-alpha0.00010",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb28_balloonBounce_acq0_2221ppps/thin1.000/fps100000/Feb28_balloonBounce_acq0_2221ppps-slice19600_29500-chunks5000x64x64-hops1250x16x16-bins100000x512x512-keep1.000-playfps100000-window_hann-alpha0.00100",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb28_balloonBounce_acq0_2221ppps/thin1.000/fps100000/Feb28_balloonBounce_acq0_2221ppps-slice19600_29500-chunks5000x64x64-hops1250x16x16-bins100000x512x512-keep1.000-playfps100000-window_hann-alpha0.00100",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb28_balloonBounce_acq0_2221ppps/thin1.000/fps1000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb28_balloonBounce_acq0_2221ppps/",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb28_balloonBounce_acq0_2221ppps/thin1.000/fps100000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb28_balloonBounce_acq0_2221ppps/thin0.050/fps100000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb28_balloonBounce_acq0_2221ppps/thin0.020/fps100000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb28_balloonBounce_acq0_2221ppps/thin0.010/fps100000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/uwb1d/quanta/Feb28_balloonBounce_acq0_2221ppps/thin1.000/fps1000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/uwb1d/quanta/Feb28_balloonBounce_acq0_2221ppps/thin0.050/fps1000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/uwb1d/quanta/Feb28_balloonBounce_acq0_2221ppps/thin0.020/fps1000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/uwb1d/quanta/Feb28_balloonBounce_acq0_2221ppps/thin0.010/fps1000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/0116-guitar-f4-tworow-aa-3/thin0.050",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/0116-guitar-f4-tworow-aa-3/thin1.000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/uwb1d/quanta/Feb28_balloonBounce_acq0_2221ppps",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/uwb1d/quanta/Feb28_balloonBounce_acq0_2221ppps/thin1.000/fps100000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/uwb1d/quanta/Feb28_balloonBounce_acq0_2221ppps/thin0.050/fps100000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/uwb1d/quanta/Feb28_balloonBounce_acq0_2221ppps/thin0.020/fps100000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/uwb1d/quanta/Feb28_balloonBounce_acq0_2221ppps/thin0.010/fps100000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/phantom-synth/stirfry-fps1000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/phantom-synth/horseride-fps1000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb28_balloonBounce_acq0_2221ppps/thin0.050",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb28_balloonBounce_acq0_2221ppps/thin0.020",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb27_blender_acq15_4351ppps/thin1.000/fps100000/Feb27_blender_acq15_4351ppps-slice65000_75000-chunks1000x64x64-hops250x16x16-bins80000x512x512-keep1.000-playfps100000-window_hann-alpha0.00100",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/uwb1d/quanta/Feb27_blender_acq15_4351ppps/thin1.000/",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/fan/thin1.000/fps100000/fan-slice70000_80000_50-chunks1000x64x64-hops250x16x16-bins100000x512x512-keep1.000-playfps100000-window_hann-alpha0.00100",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/fan/thin0.100/fps100000/fan-slice70000_80000_50-chunks1000x64x64-hops250x16x16-bins100000x512x512-keep0.100-playfps100000-window_hann-alpha0.00010",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/uwb1d/quanta/fan/thin0.100/fps100000/uwb1d-fan-slice70000_80000_step50-bins100000x512x512-keep0.100-playfps100000-alphaNone",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/uwb1d/quanta/fan/thin1.000/fps100000/uwb1d-fan-slice70000_80000_step50-bins100000x512x512-keep1.000-playfps100000-alphaNone",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/uwb1d/phantom-synth",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb29_balloonbounce_acq0_2875ppps/thin1.000/fps100000/Feb29_balloonbounce_acq0_2875ppps-slice18500_28100-chunks5000x64x64-hops1250x16x16-bins100000x512x512-keep1.000-playfps100000-window_hann-alpha0.00100",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb29_balloonbounce_acq0_2875ppps/thin0.020",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb29_balloonbounce_acq0_2875ppps/thin0.007",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb29_balloonbounce_acq0_2875ppps/thin0.005",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/quanta/balloon-laser-acq00000/thin1.000/",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/spike/viaduct-bridge/fps10000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/spike/car-100kmh/fps10000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/spike/balloon/lowpassed-balloon-slicefull-chunks1000x64x64-hops250x16x16-bins200x250x400-keep1.000-playfps10000-window_hann-alpha0.00010",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/async-spad/AC_fan/spadfan-thin1.000-freqharmx45x69-alpha0.00001/fps100K-bins1000x91x138",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/async-spad/AC_fan/spadfan-thin0.033-freqharmx45x69-alpha0.00001/fps250G-bins1250x91x138",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/async-spad/AC_fan/spadfan-thin1.000-freqharmx45x69-alpha0.00001/fps250G-bins1250x91x138",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/async-spad/AC_fan/spadfan-thin0.033-freqharmx45x69-alpha0.00001/fps100K-bins1000x91x138",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/bit2bit/Monkey-thin1.000",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/bit2bit/Monkey-thin0.100",
    # "/Volumes/CrucialX10/uwb3d-files/output/results/bit2bit/balloon-laser-acq00000-thin1.000",
    "output/results/phantom-synth/",
]

# target_framenums = [19600, 19930, 19960, 21100, 23200, 23440, 23920, 24370]
# target_framenums = slice(None, None, 50)
# target_framenums = slice(None, 100)
# feb28 acq0 balloon pop 100k fps time window
# target_framenums = slice(3600, 4300)
# feb28 acq0 balloon bounce 100k fps time window
# target_framenums = slice(1179, 2445)
# feb28 acq0 1k fps time window
# target_framenums = slice(196, 295)
target_framenums = None
# i saved the bit2bit data in a different format and filename for some reason...
bit2bit_output = False

for resdir in resdirs:
    print(f"recursively reading {resdir} for recon zarrs")
    resdir = Path(resdir)
    if bit2bit_output:
        resultzarrpaths = list(resdir.rglob("inference.zarr"))
    else:
        resultzarrpaths = list(resdir.rglob("reconinfo.zarr"))
    for resultzarrpath in resultzarrpaths:
        print(f"processing {resultzarrpath}...")

        if bit2bit_output:
            g = zarr.open_array(resultzarrpath, mode="r")
            flux = g[:]
            frameoffset = g.attrs.get("frame_start", 0)
        else:
            g = zarr.open_group(resultzarrpath, mode="r")
            flux = g["flux"][:]
            frameoffset = g.attrs["frame_start"]
        if frameoffset is None:
            frameoffset = 0
        maxflux = np.max(flux)
        if target_framenums is None:
            framenames = [f"frame_{(i + frameoffset):05d}" for i in range(len(flux))]
            # ihpp_data.to_video(flux, resultzarrpath.parent / "gamma", gamma=1/2.2, fileformat="png", cmap="grey", vmin=0, framenames=framenames)
            # ihpp_data.to_video(flux, resultzarrpath.parent / "gamma.avi", gamma=1/2.2, playback_fps=30, cmap="grey", vmin=0)
            # ihpp_data.to_video(flux, resultzarrpath.parent / "gamma.mp4", gamma=1/2.2, playback_fps=30, cmap="grey", vmin=0)
            ihpp_data.to_video(flux, resultzarrpath.parent / "flux.mp4", gamma=1.0, playback_fps=30, cmap="grey", vmin=0)
            # ihpp_data.to_video(flux, resultzarrpath.parent / "gamma14", gamma=1/4, fileformat="png", cmap="grey", vmin=0, framenames=framenames)
            # ihpp_data.to_video(flux, resultzarrpath.parent / "gamma14.mov", gamma=1/4, playback_fps=30, cmap="grey", vmin=0)
            # ihpp_data.to_video(flux, resultzarrpath.parent / "gamma1_1.2", gamma=1/1.2, fileformat="png", cmap="grey", vmin=0, framenames=framenames)
            # ihpp_data.to_video(flux, resultzarrpath.parent / "gamma1_1.8.mp4", gamma=1/1.8, playback_fps=30, cmap="grey", vmin=0)
            # ihpp_data.to_video(flux, resultzarrpath.parent / "gamma1_1.8", gamma=1/1.8, fileformat="png", cmap="grey", vmin=0, framenames=framenames)
            # ihpp_data.to_video(flux, resultzarrpath.parent / "gamma1_1.6.mp4", gamma=1/1.6, playback_fps=30, cmap="grey", vmin=0)
        else:
            targframes = []
            framenames = []
            this_target_framenums = None
            if isinstance(target_framenums, slice):
                this_target_framenums = np.arange(*target_framenums.indices(flux.shape[0])) + frameoffset
            else:
                this_target_framenums = target_framenums
            for target_framenum in this_target_framenums:
                idx = target_framenum - frameoffset
                if 0 <= idx < len(flux):
                    targframes.append(flux[idx])
                    framenames.append(f"frame_{target_framenum:05d}")
                else:
                    print(f"  - Warning: target frame {target_framenum} out of range (frameoffset {frameoffset}, flux length {len(flux)})")
            targframes = np.array(targframes)
            # ihpp_data.to_video(
            #     targframes, resultzarrpath.parent / "gamma1_1.8", gamma=1/1.8,
            #     fileformat="png", cmap="grey", vmin=0, vmax=maxflux, framenames=framenames
            # )
            ihpp_data.to_video(
                targframes, resultzarrpath.parent / "gamma_select.mp4", gamma=1/2.2,
                playback_fps=30, cmap="grey", vmin=0, vmax=maxflux
            )
        del flux

        # for the iccv fan data
        # g = zarr.open_group(resultzarrpath, mode="r")
        # flux_r = g["flux_r"][:]
        # flux_g = g["flux_g"][:]
        # flux_b = g["flux_b"][:]
        # Nt, Ny, Nx = flux_r.shape
        # flux_rgb = np.zeros((Nt, Ny, Nx, 3), dtype=np.float64)
        # # correct RGB response for filters in scene (refer to UWB supp)
        # flux_rgb[:, :, :, 0] = flux_r * 1.0
        # flux_rgb[:, :, :, 1] = flux_g * 0.75
        # flux_rgb[:, :, :, 2] = flux_b * 0.4
        # # ihpp_data.to_video(flux_rgb, resultzarrpath.parent / "gamma-rgb", gamma=1/2.2, vmin=0)
        # ihpp_data.to_video(flux_rgb, resultzarrpath.parent / "gamma-rgb.mp4", gamma=1/2.2, vmin=0, playback_fps=30)

In [ ]:
9900 * 39.3/(5*60+30), 9900 * (1*60+21.533)/(5*60+30)

In [ ]:
9900*(120.9/(5*60+30)), 9900*((2*60+21.867)/(5*60+30))

In [ ]:
frameoffset

In [ ]:
frameoffset + 3600, frameoffset + 4300

## analysis below

In [ ]:
from pathlib import Path
import zarr
import numpy as np
import matplotlib.pyplot as plt
# dataset_name = "balloon-laser-acq00000"
dataset_name = "Feb28_balloonBounce_acq0_2221ppps"
# path_b2b = Path("/Volumes/CrucialX10/uwb3d-files/output/results/bit2bit/Monkey-thin1.000/inference.zarr")
# path_uwb3d = Path("/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Monkey/thin1.000/fps100000/Monkey-slice0_10000-chunks1000x64x64-hops250x16x16-bins24800x512x512-keep1.000-playfps100000-window_hann-alpha0.00100/reconinfo.zarr")
# path_b2b = Path("/Volumes/CrucialX10/uwb3d-files/output/results/bit2bit/Monkey-thin0.100/inference.zarr")
# path_uwb3d = Path("/Volumes/CrucialX10/uwb3d-files/output/results/quanta/balloon-laser-acq00000/thin1.000/fps100000/balloon-laser-acq00000-slice5000_15000-chunks1000x64x64-hops250x16x16-bins100000x512x512-keep1.000-playfps100000-window_hann-alpha0.00100/reconinfo.zarr")
path_uwb3d = Path("/Volumes/CrucialX10/uwb3d-files/output/results/quanta/Feb28_balloonBounce_acq0_2221ppps/thin1.000/fps100000/Feb28_balloonBounce_acq0_2221ppps-slice19600_29500-chunks5000x64x64-hops1250x16x16-bins100000x512x512-keep1.000-playfps100000-window_hann-alpha0.00100/reconinfo.zarr")
# path_uwb1d = Path("/Volumes/CrucialX10/uwb3d-files/output/results/uwb1d/quanta/Feb27_blender_acq15_4351ppps/thin1.000/uwb1d-Feb27_blender_acq15_4351ppps-slice65000_75000-bins80000x512x512-keep1.000-playfps100000-alphaNone/reconinfo.zarr")
thinlevel = 1.0
# flux_b2b = zarr.open_array(path_b2b)[:]
flux_uwb3d = zarr.open_group(path_uwb3d, mode="r")["flux"][:]
# flux_uwb1d = zarr.open_group(path_uwb1d, mode="r")["flux"][:]

In [ ]:
# balloon test bright spot
# y = 200
# x = 300
# balloon test wall
# y = 100
# x = 400
# balloonbounce_bright bright spot
# y = 250
# x = 435
# balloonbounce_dark bright spot
# y = 230
# x = 476
# monkey corner spot
# y = 45
# x = 470
# random blender spot
# y = 180
# x = 150
# Feb28_balloonBounce_acq0_2221ppps bright spot
y = 270
x = 330
# Feb28_balloonBounce_acq0_2221ppps bottom right wall
# y = 400
# x = 460
# plt.imshow(flux_uwb1d[0] ** (1/2.2))
plt.imshow(flux_uwb3d[0] ** (1/2.2))
# plt.imshow(flux_b2b[0] ** (1/2.2))
plt.scatter([x], [y], c="r", s=10)

In [ ]:
# ts_b2b = flux_b2b[:, y, x]
ts_uwb3d = flux_uwb3d[:, y, x]
# ts_uwb1d = flux_uwb1d[:, y, x]

In [ ]:
# np.save(f"output/{dataset_name}-flux-b2b-x{x}-y{y}-thin{thinlevel}", ts_b2b)
# np.save(f"output/{dataset_name}-flux-uwb3d-x{x}-y{y}-thin{thinlevel}", ts_uwb3d)
np.save(f"output/{dataset_name}-flux-uwb1d-x{x}-y{y}-thin{thinlevel}", ts_uwb1d)

In [ ]:
plt.plot(ts_uwb3d)
# plt.xlim(100, 200)

In [ ]:
ts_uwb3d = ts_uwb3d[:]
ft = np.fft.rfftfreq(len(ts_uwb3d), 1/100000)
F_uwb3d = np.fft.rfft(ts_uwb3d)
plt.plot(ft, np.abs(F_uwb3d))
plt.yscale("log")
plt.xlim(0, 400)
plt.axvline(120, color="red", linestyle="--")

In [ ]:
plt.plot(flux[:, y, x])
# plt.xlim(0, 100)

In [ ]:
ft = np.fft.rfftfreq(flux.shape[0], d=1/100000)
plt.plot(ft, np.abs(np.fft.rfft(flux[:, y, x])))
plt.yscale("log")

In [ ]:
# np.save(config.OUTPUT_DIR / "b2b-monkey-flux-pixel.npy", flux[:, y, x])

In [ ]:
respath2 = Path("/Volumes/CrucialX10/uwb3d-files/output/results/quanta/teaser_balloonbounce_dark/thin1.000/fps100000/teaser_balloonbounce_dark-slice24300_37900-chunks1000x64x64-hops250x16x16-bins100000x512x512-keep1.000-playfps100000-window_hann-alpha0.00100/reconinfo.zarr")
flux2 = zarr.open_group(respath2, mode="r")["flux"][:]
# flux2 = utils.read_video(respath2)

In [ ]:
# y = 200
# x = 300
# y = 100
# x = 400
plt.imshow(flux2[10] ** (1/2.2))
plt.scatter([x], [y], c="r", s=1)

In [ ]:
ts_b2b = flux[:2000, y, x]
ts_uwb3d = flux2[:2000, y, x]
plt.figure(figsize=(12, 5))
plt.plot((ts_b2b - np.min(ts_b2b)) / (np.max(ts_b2b) - np.min(ts_b2b)), label="normalized bit2bit flux")
plt.plot((ts_uwb3d - np.min(ts_uwb3d)) / (np.max(ts_uwb3d) - np.min(ts_uwb3d)), label="normalized uwb3d flux", alpha=0.9)
# plt.xlim(0, 5000)
plt.xlim(500, 600)
# plt.xlim(0, 500)
# change figsize to be wider
plt.legend()

In [ ]:
fig, axs = plt.subplots(1, 1)
F_uwb3d = np.fft.rfft(ts_uwb3d)
ft = np.fft.rfftfreq(len(ts_uwb3d), d=1/100000)

axs.plot(ft, np.abs(F_uwb3d))
axs.set_title("uwb3d flux FFT")
axs.set_xlabel("Frequency (Hz)")
axs.set_ylabel("|F|")
axs.set_yscale("log")
# axs[1].set_xlim(-10, 500)

plt.tight_layout()

In [ ]:
dataname = "balloon-laser-acq00001"

data_path = config.DATA_DIR / f"quanta/{dataname}"
data_name = data_path.stem
frames, metadata = ihpp_data.read_quanta_auto(data_path, load_data=False)
frames = frames[:]

In [ ]:
rad = 2
superpix = frames[:, y-rad:y+rad+1, x-rad:x+rad+1]
superpix_sum = superpix.sum(axis=(1, 2))
abssum = np.zeros(superpix.shape[0] // 2 + 1)
for i in range(superpix.shape[1]):
    for j in range(superpix.shape[2]):
        abssum += np.abs(np.fft.rfft(superpix[:, i, j]))
plt.plot(abssum)
plt.yscale("log")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
F_b2b = np.fft.rfft(ts_b2b)
F_uwb3d = np.fft.rfft(ts_uwb3d)
ft = np.fft.rfftfreq(len(ts_b2b), d=1/100000)

axs[0].plot(ft, np.abs(F_b2b), color="C0")
axs[0].set_title("bit2bit flux FFT")
axs[0].set_ylabel("|F|")
axs[0].set_yscale("log")
# axs[0].set_xlim(-10, 500)

axs[1].plot(ft, np.abs(F_uwb3d), color="C1")
axs[1].set_title("uwb3d flux FFT")
axs[1].set_xlabel("Frequency (Hz)")
axs[1].set_ylabel("|F|")
axs[1].set_yscale("log")
# axs[1].set_xlim(-10, 500)

plt.tight_layout()

In [ ]:
F_flux = np.fft.rfft(flux2[:, y, x])
ft_flux = np.fft.rfftfreq(len(flux2[:, y, x]), d=1/100000)
plt.plot(ft_flux, np.abs(F_flux))
plt.yscale("log")
plt.xlim(0, 1000)
# plt.axvline(31050, color='red', linestyle='--')
# plt.axvline(120, color='red', linestyle='--')

In [ ]:
binframepath = Path("/Volumes/CrucialX10/uwb3d-files/data/quanta/balloon-laser-acq00000.zarr")
binframes = zarr.open_group(binframepath, mode="r")["frames"]

In [ ]:
ts = binframes[:, y, x]
F_bin = np.fft.rfft(ts)
plt.plot(np.abs(F))
plt.yscale("log")

In [ ]:
np.save(config.OUTPUT_DIR / "uwb3d-monkey-flux-pixel.npy", flux2[:, y, x])

In [ ]:
y = 220
x = 478
plt.imshow(flux2[5] ** (1/2.2))
plt.scatter([x], [y], c="r", s=1)

In [ ]:
plt.plot(flux2[:, y, x] / flux2[:, y, x].max())
plt.plot(flux[:, y, x] / flux[:, y, x].max())
plt.xlim(0, 100)

In [ ]:
F2 = np.fft.rfft(flux2[:, 50, 50])

In [ ]:
ft = np.fft.rfftfreq(flux2.shape[0], 1/flux2.shape[0]) / (flux2.shape[0] / 100000)

In [ ]:
plt.plot(ft, np.abs(F2))
plt.yscale("log")
# plt.xlim(0, 500)

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12,8))
axs[0, 0].plot(flux2[:, 50, 50])
axs[0, 0].set_xlim(-20, 1500)
axs[0, 0].set_title("3D UWB pixel flux")
axs[0, 0].set_xlabel("frame num")
axs[0, 1].plot(ft, np.abs(F2))
axs[0, 1].set_xlabel("Hz")
axs[0, 1].set_title("3D UWB pixel power spec")
axs[0, 1].set_yscale("log")
axs[0, 1].set_xlim(-500, 15000)
axs[1, 0].plot(flux[:, 50, 50])
axs[1, 0].set_xlim(-20, 1500)
axs[1, 0].set_ylim(2050, 2400)
axs[1, 0].set_xlabel("frame num")
axs[1, 0].set_title("bit2bit pixel value")
axs[1, 1].plot(ft, np.abs(F))
axs[1, 1].set_yscale("log")
axs[1, 1].set_xlim(-500, 15000)
axs[1, 1].set_xlabel("Hz")
axs[1, 1].set_title("bit2bit pixel power spec")
plt.tight_layout()
plt.show()

In [ ]:
from sptflux import utils
vidpath = Path("output/results/async-spad/AC_fan/spadfan-keep1.000-freqharmx45x69-cfar2.0/fps100K-bins1000x91x138/gamma-rgb.avi")
vid = utils.read_video(vidpath, return_framerate=False)

In [ ]:
vid.shape

In [ ]:
plt.imshow(vid[100])

In [ ]:
import cv2
cv2.imwrite(vidpath.parent / "frame.png", cv2.cvtColor(vid[100], cv2.COLOR_RGB2BGR))

In [ ]:
import zarr
import numpy as np
import matplotlib.pyplot as plt
blender_bright_3duwb_path = "output/results/quanta/blender_lasagna_bright/thin1.000/fps1000/blender_lasagna_bright-keep1.000-playfps1000-chunks5000x32x32-hops2500x16x16-bins1000x512x512-window_hann-cfar1.0/reconinfo.zarr"
blender_bright_1duwb_path = "output/results/quanta/blender_lasagna_bright/thin1.000/fps1000/blender_lasagna_bright-keep1.000-playfps1000-chunksNonex1x1-hopsNonex1x1-bins1000x512x512-window_hann-cfar1.0/reconinfo.zarr"
blender_dark_3duwb_path = "output/results/quanta/blender_lasagna_dark/thin1.000/fps1000/blender_lasagna_dark-keep1.000-playfps1000-chunks5000x32x32-hops2500x16x16-bins1000x512x512-window_hann-cfar1.0/reconinfo.zarr"
blender_dark_1duwb_path = "output/results/quanta/blender_lasagna_dark/thin1.000/fps1000/blender_lasagna_dark-keep1.000-playfps1000-chunksNonex1x1-hopsNonex1x1-bins1000x512x512-window_hann-cfar1.0/reconinfo.zarr"
blender_bright_3duwb_flux = zarr.open_group(blender_bright_3duwb_path, mode="r")["flux"][:]
blender_bright_1duwb_flux = zarr.open_group(blender_bright_1duwb_path, mode="r")["flux"][:]
blender_dark_3duwb_flux = zarr.open_group(blender_dark_3duwb_path, mode="r")["flux"][:]
blender_dark_1duwb_flux = zarr.open_group(blender_dark_1duwb_path, mode="r")["flux"][:]

In [ ]:
y, x = 150, 200
plt.imshow(blender_bright_3duwb_flux[50])
plt.scatter([x], [y], color="red")

In [ ]:
blender_bright_3duwb_powerspec = np.abs(np.fft.rfftn(blender_bright_3duwb_flux[:, y, x]))
blender_bright_1duwb_powerspec = np.abs(np.fft.rfftn(blender_bright_1duwb_flux[:, y, x]))
blender_dark_3duwb_powerspec = np.abs(np.fft.rfftn(blender_dark_3duwb_flux[:, y, x]))
blender_dark_1duwb_powerspec = np.abs(np.fft.rfftn(blender_dark_1duwb_flux[:, y, x]))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
time = np.linspace(0, 1, blender_bright_3duwb_flux.shape[0])
axs[0].plot(time, blender_bright_3duwb_flux[:, y, x], label="bright 3D UWB")
axs[0].plot(time, blender_bright_1duwb_flux[:, y, x], label="bright 1D UWB")
axs[0].set_title("Bright scene flux")
axs[0].legend()
axs[1].plot(time, blender_dark_3duwb_flux[:, y, x], label="dark 3D UWB")
axs[1].plot(time, blender_dark_1duwb_flux[:, y, x], label="dark 1D UWB")
axs[1].set_title("Dark scene flux")
axs[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 10))
time = np.linspace(0, 1, blender_bright_3duwb_flux.shape[0])
freq = np.fft.rfftfreq(blender_bright_3duwb_flux.shape[0], d=1/blender_bright_3duwb_flux.shape[0])
axs[0, 0].plot(freq, blender_bright_3duwb_powerspec, label="bright 3D UWB")
axs[0, 0].set_title("3D UWB bright power spectrum")
axs[0, 0].set_xlabel("Frequency (Hz)")
axs[0, 0].set_yscale("log")
axs[1, 0].plot(freq, blender_bright_1duwb_powerspec, label="bright 1D UWB")
axs[1, 0].set_title("1D UWB bright power spectrum")
axs[1, 0].set_xlabel("Frequency (Hz)")
axs[1, 0].set_yscale("log")
axs[0, 1].plot(freq, blender_dark_3duwb_powerspec, label="dark 3D UWB")
axs[0, 1].set_title("3D UWB dark power spectrum")
axs[0, 1].set_xlabel("Frequency (Hz)")
axs[0, 1].set_yscale("log")
axs[1, 1].plot(freq, blender_dark_1duwb_powerspec, label="dark 1D UWB")
axs[1, 1].set_title("1D UWB dark power spectrum")
axs[1, 1].set_xlabel("Frequency (Hz)")
axs[1, 1].set_yscale("log")
plt.tight_layout()
plt.show()

# 1d vs 3d uwb synthetic comparisons

In [ ]:
import finufft
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from sptflux import ihpp_sim, ihpp_data, flux_probing

def flux_func_tapered_square_plus_wavy_bg(
    start_x: float,
    start_y: float,
    width: int,
    height: int,
    num_t: int,
    T_exp: float,
    square_side: float,
    taper: float,
    fg_amplitude: float,
    bg_amplitude: float,
    normalize: bool = True,
    vx_fg: float = 0.0,
    vx_bg: float = 0.0,
    vy_fg: float = 0.0,
    vy_bg: float = 0.0,
    bg_cycles_x: float = 4.0,
    bg_cycles_y: float = 0.0,
    bg_phase: float = 0.0,
    tex_strength: float = 0.8,     # how strong texture is (0=no texture)
):
    volume = width * height * T_exp
    scale = volume if normalize else 1.0

    # --- make square slightly bigger ---
    square_side = 1.25 * square_side   # 👈 25% larger
    half = 0.5 * square_side
    taper = float(np.clip(taper, 0.0, 1.0))
    eps = 1e-12

    # --- fixed random texture (object frame) ---
    rng = np.random.default_rng(42)
    texture = rng.standard_normal((height, width))
    texture -= texture.min()
    texture /= texture.max() + 1e-12     # map to [0,1]

    # turn into multiplicative contrast
    tex_mult = 1.0 + tex_strength * (2.0 * texture - 1.0)   # in [1-α, 1+α]

    def _tukey01(u, a):
        u = np.clip(u, 0.0, 1.0)
        if a <= 0.0:
            return np.ones_like(u)
        if a >= 1.0:
            return 0.5 * (1.0 - np.cos(2.0 * np.pi * u))
        w = np.ones_like(u)
        left = u < (a / 2.0)
        right = u > (1.0 - a / 2.0)
        w[left] = 0.5 * (1.0 + np.cos(np.pi * (2.0 * u[left] / a - 1.0)))
        w[right] = 0.5 * (1.0 + np.cos(np.pi * (2.0 * u[right] / a - 2.0 / a + 1.0)))
        return w

    def compute_flux(x, y, t):
        x = np.asarray(x, float).copy()
        y = np.asarray(y, float).copy()
        t = np.asarray(t, float).copy()

        if normalize:
            x *= width
            y *= height
            t *= T_exp

        frame_idx = t * num_t / T_exp

        # --- moving square center ---
        x_fg = start_x + vx_fg * frame_idx
        y_fg = start_y + vy_fg * frame_idx

        ux = (x - x_fg + half) / (2 * half + eps)
        uy = (y - y_fg + half) / (2 * half + eps)

        in_sq = (np.abs(x - x_fg) <= half) & (np.abs(y - y_fg) <= half)

        env = _tukey01(ux, taper) * _tukey01(uy, taper)
        env *= in_sq.astype(float)

        # --- texture sampled in object coordinates (moves with square) ---
        xi = np.mod(np.round(x - x_fg + width / 2).astype(int), width)
        yi = np.mod(np.round(y - y_fg + height / 2).astype(int), height)

        fg = env * tex_mult[yi, xi]

        # --- background wave ---
        xn = x / max(width, 1)
        yn = y / max(height, 1)

        x_bg = xn - (vx_bg * frame_idx) / max(width, 1)
        y_bg = yn - (vy_bg * frame_idx) / max(height, 1)

        phase = 2*np.pi*(bg_cycles_x * x_bg + bg_cycles_y * y_bg) + bg_phase
        bg = 0.5 + 0.5*np.sin(phase)

        return scale * (fg_amplitude * fg + bg_amplitude * bg)

    upper_bound = scale * (fg_amplitude * (1 + tex_strength) + bg_amplitude)
    return compute_flux, upper_bound

def make_flux_function(
    *,
    amplitude: float = 1.0,
    bg_mean: float = 0.6,
    bg_contrast: float = 0.4,
    base_fx: float = 3.0,
    base_fy: float = 5.0,
    fx_mod: float = 2.0,
    fy_mod: float = 2.0,
    phase_speed: float = 2.0,
    circle_radius: float = 0.12,
    circle_intensity: float = 0.9,
    circle_path_amp: float = 0.28,
    circle_path_freq: float = 1.0,
    circle_path_phase: float = 0.0,
) -> callable:
    """
    Returns a continuous spatiotemporal flux(x, y, t) defined on [0,1]^3.

    Components:
      1) Background: spatial sinusoid whose frequency and intensity vary with time.
      2) Moving solid circle: disk indicator moving along a sinusoidal path.

    Output is guaranteed non-negative (clipped) and scaled by `amplitude`.

    Parameters assume x,y,t in [0,1]. Outside that range, behavior is still defined
    but not constrained (it will just evaluate the formulas).
    """

    # Small helper: keep things smooth-ish at boundaries without being mandatory.
    def _clip01(u):
        return np.clip(u, 0.0, 1.0)

    def flux(x, y, t):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        t = np.asarray(t, dtype=float)

        # --- Time-varying background intensity + frequencies ---
        # Time-varying mean and contrast (kept >=0 mean by construction if bg_mean >= bg_contrast).
        mean_t = bg_mean * (0.85 + 0.15 * np.cos(2.0 * np.pi * 0.7 * t))
        contrast_t = bg_contrast * (0.6 + 0.4 * np.sin(2.0 * np.pi * 1.1 * t + 0.3))

        # Time-varying frequencies (cycles per unit in x/y)
        fx_t = base_fx + fx_mod * np.sin(2.0 * np.pi * 0.9 * t + 0.2)
        fy_t = base_fy + fy_mod * np.cos(2.0 * np.pi * 0.6 * t + 0.5)

        # A drifting phase term so the wave field "moves" over time
        phi_t = 2.0 * np.pi * (phase_speed * t)

        bg = mean_t + contrast_t * np.sin(2.0 * np.pi * (fx_t * x + fy_t * y) + phi_t)

        # --- Moving solid circle (disk) ---
        # Sinusoidal path around center (0.5, 0.5), clipped to stay within [0,1]
        cx = (0.5
            + circle_path_amp/6 * np.sin(2.0 * np.pi * circle_path_freq * t + circle_path_phase)
            + circle_path_amp/4 * np.cos(2.0 * np.pi * 5 * t)
            + circle_path_amp/3 * np.sin(2.0 * np.pi * 7 * t)
            + circle_path_amp/5 * np.sin(2.0 * np.pi * 10 * t)
        )
        # cy = 0.5 + circle_path_amp * np.cos(2.0 * np.pi * circle_path_freq * t + circle_path_phase + 0.7)
        cy = np.full_like(cx, 0.5)
        cx = _clip01(cx)
        cy = _clip01(cy)

        r2 = (x - cx) ** 2 + (y - cy) ** 2
        disk = (r2 <= (circle_radius ** 2)).astype(float)

        # Additive circle (you can also make it multiplicative if you prefer)
        circle = circle_intensity * disk

        # Combine + enforce non-negativity + scale
        out = bg + circle
        out = np.clip(out, 0.0, None)  # ensure non-negative
        return amplitude * out

    return flux


# Example usage:
# flux = make_flux_function(amplitude=2.0)
# val = flux(0.2, 0.7, 0.4)              # scalar
# grid_val = flux(X, Y, T)               # numpy arrays/broadcastable inputs


In [ ]:
t_exp = 1.0  # seconds
velocity_x = 2  # pixels/frame
velocity_y = 1  # pixels/frame
amplitude_fg = 5.0e3
amplitude_bg = 5.0e2
# Parameters for reconstruction
reconstructed_bins = (500, 128, 128)  # (time, height, width)
nt, ny, nx = reconstructed_bins
volume = t_exp * nx * ny
dt = t_exp / nt  # seconds per time bin
# Setting up coordinate grids
t_grid = np.linspace(0, 1, nt)
x_grid = np.linspace(0, 1, nx)
y_grid = np.linspace(0, 1, ny)
seed = 42
alpha = 1.0
tex_strength = 0.8

In [ ]:
flux, l_max = flux_func_tapered_square_plus_wavy_bg(
    start_x=nx/10,
    start_y=ny/10,
    width=nx,
    height=ny,
    num_t=nt,
    T_exp=t_exp,
    square_side=30,
    taper=0.4,
    fg_amplitude=amplitude_fg,
    bg_amplitude=amplitude_bg,
    vx_fg=+velocity_x,
    vx_bg=-velocity_x,
    vy_fg=+velocity_y,
    bg_cycles_x=6.0,
    tex_strength=tex_strength,
)

In [ ]:
TT, YY, XX = np.meshgrid(t_grid, y_grid, x_grid, indexing="ij")
# flux_samp = flux(XX, YY, TT)
flux = make_flux_function(
    amplitude=1_000_000,
    circle_path_freq=10,
    circle_intensity=1.5,
)
flux_samp = flux(XX, YY, TT)

In [ ]:
TT, YY, XX = np.meshgrid(t_grid, y_grid, x_grid, indexing="ij")
flux_samp = flux(XX, YY, TT)

In [ ]:
ihpp_data.to_video(
    flux_samp, "output/synthetic/compare/simulated_flux.avi", playback_fps=30, cmap="gray"
)

In [ ]:
# (x, y, t)
pts = ihpp_sim.simulate_ihpp3d(
    lambda_fn=flux,
    s_region=[[0, 0], [1, 0], [1, 1], [0, 1]],
    t_range=(0.0, 1.0),
    rng=seed
)
print(f"Simulated {len(pts)} events")

In [ ]:
# discretize pts pixel-wise
pixel_pos = (pts[:, :2] * [nx, ny]).astype(int)
pts_disc = pts.copy()
pts_disc[:, :2] = pixel_pos / [nx, ny]

In [ ]:
nmodes = (nt, ny, nx)
F = finufft.nufft3d1(
    pts_disc[:, 2] * 2 * np.pi, pts_disc[:, 1] * 2 * np.pi, pts_disc[:, 0] * 2 * np.pi,
    np.ones(pts.shape[0], dtype="complex128"),
    nmodes, eps=1e-6, isign=-1
)
bound = flux_probing.compute_amplitude_bound(pts.size, 1, alpha=1 / (np.prod(nmodes) / 2))
F_bd = F.copy()
F_bd[np.abs(F) < bound] = 0.0
recon_3dfft = np.fft.ifftn(np.fft.ifftshift(F_bd)).real / np.sqrt(V)
recon_3dfft = np.clip(recon_3dfft, 0, None)

In [ ]:
ihpp_data.to_video(
    recon_3dfft, "output/synthetic/compare/recon_3dfft.avi", playback_fps=30, cmap="gray"
)

In [ ]:
F_t = np.fft.ifftn(np.fft.ifftshift(F_bd), axes=(1, 2), norm="forward")

In [ ]:
px = 63
py = 63
# mask = (pixel_pos[:, 0] == px) & (pixel_pos[:, 1] == py)
superpix_size = 4
mask = (pixel_pos[:, 0] >= px-superpix_size) & (pixel_pos[:, 0] <= px+superpix_size) & (pixel_pos[:, 1] >= py-superpix_size) & (pixel_pos[:, 1] <= py+superpix_size)
timestamps = np.sort(pts_disc[mask, 2])
F_1d = finufft.nufft1d1(timestamps * 2 * np.pi, np.ones(timestamps.shape[0], dtype="complex128"), nt, eps=1e-6, isign=-1) / np.sqrt(t_exp)
F_1d_bd = F_1d.copy()
bound_1d = flux_probing.compute_amplitude_bound(timestamps.size, t_exp, alpha=1/(nt/2))
F_1d_bd[np.abs(F_1d) < bound_1d] = 0.0

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 5))
ft = np.fft.fftshift(np.fft.fftfreq(nt, d=1/nt))
posfreqmask = ft >= 0
ft_pos = ft[posfreqmask]
F_gt = np.fft.fftshift(np.fft.fft(flux_samp[:, py, px], norm="forward"))
F_t_pos = np.fft.fftshift(F_t[:, py, px])[posfreqmask]
F_gt_pos = F_gt[posfreqmask]
F_1d_bd_pos = F_1d_bd[posfreqmask]

axs[0].plot(ft_pos, np.abs(F_1d_bd_pos), label="1D UWB")
# axs[0].plot(ft_pos, np.abs(F_gt_pos), alpha=0.6, label="GT spectrum")
axs[0].axhline(bound_1d, color="red", linestyle="--", label="1D UWB bound")
axs[0].set_title("1D UWB CFAR spectrum")
axs[0].set_xlabel("Frequency (Hz)")
axs[0].set_yscale("log")
axs[0].legend()
axs[1].plot(ft_pos, np.abs(F_t_pos), label="3D UWB CFAR")
# axs[1].plot(ft_pos, np.abs(F_gt_pos), alpha=0.6, label="GT spectrum")
axs[1].set_title("3D UWB CFAR spectrum")
axs[1].set_xlabel("Frequency (Hz)")
axs[1].set_yscale("log")
axs[1].legend()
plt.show()

In [ ]:
plt.plot(ft_pos, np.abs(F_1d[posfreqmask])*0.01, "--", label=f"1D UWB {superpix_size*2+1}x{superpix_size*2+1} superpixel (scaled)")
plt.plot(ft_pos, np.abs(F_gt_pos), alpha=0.6, label="GT spectrum")
plt.plot(ft_pos, np.abs(F_t_pos), linestyle="-.", label="3D UWB CFAR")
plt.title("Spectrum comparison")
plt.xlabel("Frequency (Hz)")
plt.yscale("log")
plt.legend()

In [ ]:
plt.plot(t_grid, flux_samp[:, py, px])

In [ ]:
out = np.zeros((nt, ny, nx), dtype="float64")
pbar = tqdm(total=nx*ny)
for i in range(nx):
    for j in range(ny):
        timestamps_ij = np.sort(pts_disc[(pixel_pos[:, 0] == i) & (pixel_pos[:, 1] == j), 2])
        F_1d = finufft.nufft1d1(timestamps_ij * 2 * np.pi, np.ones(timestamps_ij.shape[0], dtype="complex128"), nt, eps=1e-6, isign=-1) / np.sqrt(t_exp)
        F_1d_bd = F_1d.copy()
        bound_1d = flux_probing.compute_amplitude_bound(timestamps_ij.size, t_exp, nfreqs=nt)
        F_1d_bd[np.abs(F_1d) < bound_1d] = 0.0
        recon_1d = np.fft.ifft(np.fft.ifftshift(F_1d_bd), norm="forward").real / np.sqrt(t_exp)
        out[:, j, i] = recon_1d
    pbar.update(ny)

In [ ]:
ihpp_data.to_video(np.clip(out, 0, None), "output/synthetic/compare/recon_1d.avi", playback_fps=30, cmap="gray")

# superpixel 1d uwb

In [ ]:
import zarr
import numpy as np
import cupy as cp
from pathlib import Path
from tqdm import tqdm
from sptflux import flux_probing, ihpp_data

dataname = "gun-2s"
g = zarr.open_group(f"data/quanta/{dataname}.zarr", mode="r")
frames = g["frames"][:]
frames = cp.asarray(frames, dtype="uint8")
nt, ny, nx = 1000, frames.shape[1], frames.shape[2]
out = np.zeros((nt, ny, nx), dtype="float64")
pbar = tqdm(total=nx*ny)
superpix_radius = 1
texp = 1.0
for i in range(nx):
    for j in range(ny):
        superpixel = frames[:, max(0, j-superpix_radius):j+superpix_radius+1, max(0, i-superpix_radius):i+superpix_radius+1]
        # superpixel = cp.asarray(superpixel, dtype="uint8")
        numpix = superpixel.shape[1] * superpixel.shape[2]
        superpix_photons = cp.sum(superpixel, axis=(1, 2))
        nphotons = cp.sum(superpix_photons)
        F = cp.fft.rfft(superpix_photons) / np.sqrt(texp)
        bound = flux_probing.compute_amplitude_bound(nphotons, texp, nfreqs=nt/2)
        F[np.abs(F) < bound] = 0.0
        recon = cp.fft.irfft(F, n=nt, norm="forward").real / np.sqrt(texp) / numpix
        out[:, j, i] = recon.get()
    pbar.update(ny)

In [ ]:
clipped = np.clip(out, 0, None)
gamma = clipped ** (1/2.2)

In [ ]:
ihpp_data.to_video(gamma, f"output/results/quanta/{dataname}/superpix{superpix_radius}.mp4", playback_fps=30, cmap="gray")

# 2d fft video

In [ ]:
import zarr
import numpy as np
from sptflux import ihpp_data, flux_probing, plotting
from pathlib import Path

dataname = "balloon_gun3"
path = Path(f"data/quanta/{dataname}.zarr")
g = zarr.open_group(path, mode="r")
frames = g["frames"][1000:1500]
fftskip = 10

In [ ]:
F = np.fft.fftn(frames, axes=(1, 2))
mag = np.abs(np.fft.fftshift(F, axes=(1, 2)))
logmag = np.log1p(mag)
ihpp_data.to_video(logmag[::fftskip], f"output/fft2d-tests/{dataname}/frames_fft.mp4", playback_fps=30, cmap="magma")

In [ ]:
F = np.fft.fftn(frames, axes=(0, 1, 2))

In [ ]:
ft_lowpass_cutoff = 10
nt, ny, nx = frames.shape
ft = np.fft.fftfreq(nt, 1/nt)
ft_mask = np.abs(ft) < ft_lowpass_cutoff
F_low = F.copy()
F_low[~ft_mask] = 0.0
F_spatial = np.fft.ifftn(F_low, axes=(0,))
mag_spatial = np.abs(np.fft.fftshift(F_spatial, axes=(1, 2)))
logmag_spatial = np.log1p(mag_spatial)
ihpp_data.to_video(logmag_spatial[::fftskip], f"output/fft2d-tests/{dataname}/frames_fft_lowpass.mp4", playback_fps=30, cmap="magma")

In [ ]:
nt, ny, nx = frames.shape
F_bd = F.copy()
bd = flux_probing.compute_amplitude_bound(np.count_nonzero(frames), V=1, nfreqs=nt*ny*nx)
bd_mask = np.abs(F) >= bd * 2.0
F_bd[~bd_mask] = 0.0
F_spatial_bd = np.fft.ifftn(F_bd, axes=(0,))
mag_spatial_bd = np.abs(np.fft.fftshift(F_spatial_bd, axes=(1, 2)))
logmag_spatial_bd = np.log1p(mag_spatial_bd)
ihpp_data.to_video(logmag_spatial_bd[::fftskip], f"output/fft2d-tests/{dataname}/frames_fft_cfar.mp4", playback_fps=30, cmap="magma")

In [ ]:
ani = plotting.ani_fluxvids(
    [[logmag[::fftskip], logmag_spatial[::fftskip], logmag_spatial_bd[::fftskip]]],
    playback_fps=10,
    titles=[["Original FFT", "temporal lowpass", "CFAR thresholded"]],
    cmap="magma",
    figsize=(5, 4)
)
ani.save(f"output/fft2d-tests/{dataname}/fft_comparison_2x.mp4", fps=10, dpi=300)

# chunk velocity segmentation thing

In [ ]:
import zarr
import numpy as np
from pathlib import Path
from sptflux import ihpp_data, plotting, flux_probing

dataname = "balloon_gun2"
path = Path(f"data/quanta/{dataname}.zarr")
g = zarr.open_group(path, mode="r")
frames = g["frames"][:]
nt, ny, nx = frames.shape

In [ ]:
patch = frames[70000:, 162:162+32, 352:352+32]
# patch = frames[60000:65000, 148:248, 89:189]
superpix_photons = np.sum(patch, axis=(1, 2))
F_1d = np.fft.rfft(superpix_photons)

In [ ]:
superpix_photons

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fts = np.fft.rfftfreq(patch.shape[0], d=1/patch.shape[0]) / (patch.shape[0] / 100000)
plt.plot(fts, np.abs(F_1d))
plt.yscale("log")
bound = flux_probing.compute_amplitude_bound(np.sum(superpix_photons), V=1, nfreqs=nt) * 2
plt.axhline(bound, color="red", linestyle="--", label="1D UWB bound")
plt.xlim(0, 1000)
plt.legend()

In [ ]:
plt.plot(superpix_photons)

In [ ]:
np.max(fts[np.abs(F_1d) > bound])

In [ ]:
patch_H = 32
patch_W = 32
hop_H = 32
hop_W = 32
npatches_H = (ny - patch_H) // hop_H + 1
npatches_W = (nx - patch_W) // hop_W + 1
out = np.zeros((npatches_H, npatches_W), dtype="float64")
for h0 in range(0, ny-patch_H+1, hop_H):
    for w0 in range(0, nx-patch_W+1, hop_W):
        patch = frames[80000:85000, h0:h0+patch_H, w0:w0+patch_W]
        fts = np.fft.rfftfreq(patch.shape[0], d=1/patch.shape[0]) / (patch.shape[0] / 100000)
        superpix_photons = np.sum(patch, axis=(1, 2))
        F_1d = np.fft.rfft(superpix_photons)
        bound = flux_probing.compute_amplitude_bound(np.sum(superpix_photons), V=1, nfreqs=nt) * 2.0
        highest_ft = np.max(fts[np.abs(F_1d) > bound])
        out[h0//hop_H, w0//hop_W] = highest_ft

In [ ]:
import matplotlib
# map the colors to log scale for better visibility
plt.imshow(out, norm=matplotlib.colors.LogNorm())
plt.colorbar()

In [ ]:
np.quantile(out, 0.9)

# cavitations experiments or something idk

In [ ]:
from PIL import Image
from sptflux import config, utils, ihpp_data, ihpp_sim, flux_probing
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# tifpath = config.DATA_DIR / "phantom/cavitation1/cam1__C001H001S0001000001.tif"
tifpath = "data/phantom-synth/separated_videos/cavitation1/cam1__C001H001S0001000008.tif"

a = np.array(Image.open(tifpath))

In [ ]:
plt.imshow(a[200:250, 110:190], vmax=20)
plt.colorbar()

In [ ]:
a[200:250, 110:190]

In [ ]:
# imgs = utils.read_imgdir(config.DATA_DIR / "phantom/cavitation1/", grayscale=True)
imgs = utils.read_imgdir("data/phantom-synth/separated_videos/cavitation1/", grayscale=True)

In [ ]:
fluxfunc, discr_flux, maxflux = ihpp_data.fluxfunc_from_frames(imgs * (4096/255), conversion_gain=1, quant_eff=0.46, fps=100000, normalize=False)
# fluxfunc, maxflux = ihpp_data.fluxfunc_from_frames(imgs * (4096/255), conversion_gain=1, quant_eff=0.46, fps=100000, normalize=False)

In [ ]:
discr_flux

In [ ]:
maxflux

In [ ]:
np.mean(discr_flux)

In [ ]:
photon_counts = np.random.poisson(lam=discr_flux/100_000)

In [ ]:
from sptflux import config, utils
import zarr
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# # path = config.DATA_DIR / "phantom-synth/upsampled/cavitation1_rife4_linear1_ppp0.1.zarr"
# # path = "data/phantom-synth/upsampled/cavitation1_rife4_linear8_ppp1.zarr"
# path = Path("data/phantom-synth/upsampled/cavitation1_rife4_linear8_ppp0.1.zarr")
# g = zarr.open_group(path, mode="r")
# frames = g["frames"][:]

In [ ]:
import numpy as np
from typing import Optional

def simulate_conventional_camera(
    spad_frames: np.ndarray,          # shape [N_frames, H, W], dtype uint8 (0 or 1)
    n_quanta_frames: int,             # how many SPAD frames = one conventional exposure
    sigma_read: float = None,          # read noise in electrons RMS
    qe_cmos: float = 1.0,            # QE of conventional sensor (0–1); set 1.0 to skip
    gain: float = 1.0,               # conversion gain, e⁻ per DN
    full_well_capacity: int = 65535, # in electrons (clip before gain division)
    frame_start: int = 0,            # which SPAD frame to start from
    output_dtype=np.float32,         # use float to keep it linear; or np.uint16
    rng: Optional[np.random.Generator] = None,
) -> dict:
    """
    Simulate a conventional CMOS camera image from SPAD binary quanta frames.

    Parameters
    ----------
    spad_frames : ndarray [N_frames, H, W]
        Binary quanta frames (0 or 1) or photon counts (if already summed).
        This should already encode Poisson shot noise and dark counts.
    n_quanta_frames : int
        Number of SPAD frames to integrate over (defines exposure time T = n * dt).
    sigma_read : float
        Read noise standard deviation in electrons.
    qe_cmos : float
        Quantum efficiency of the conventional sensor (applied via binomial draw).
    gain : float
        Sensor gain in e⁻/DN. Output is in DN.
    full_well_capacity : int
        Full well capacity in electrons. Pixels saturate above this.
    frame_start : int
        Starting frame index within spad_frames.
    output_dtype : numpy dtype
        Output image dtype. Use float32 to keep full linearity.
    rng : np.random.Generator, optional
        Random number generator for reproducibility.

    Returns
    -------
    dict with keys:
        'image'         : simulated camera image in DN [H, W]
        'photoelectrons': photoelectron map before read noise [H, W]
        'n_frames_used' : actual number of frames used
        'exposure_info' : dict of exposure parameters
    """
    if rng is None:
        rng = np.random.default_rng()

    N_total, H, W = spad_frames.shape
    frame_end = frame_start + n_quanta_frames

    if frame_end > N_total:
        raise ValueError(
            f"Requested frames [{frame_start}, {frame_end}) exceed "
            f"available frames ({N_total}). Reduce n_quanta_frames or frame_start."
        )

    # ── Step 1: Sum binary frames → raw photon counts ──────────────────────────
    # This is Poisson-distributed by construction (sum of Bernoulli trials).
    # Shape: [H, W], dtype int32
    photon_counts = spad_frames[frame_start:frame_end].sum(axis=0).astype(np.int32)

    # ── Step 2: Apply CMOS quantum efficiency (optional) ───────────────────────
    # Each incident photon is detected with probability qe_cmos.
    # Binomial draw preserves Poisson statistics (Poisson × Binomial = Poisson).
    if qe_cmos < 1.0:
        photoelectrons = rng.binomial(photon_counts, qe_cmos).astype(np.int32)
    else:
        photoelectrons = photon_counts.copy()

    # ── Step 3: Clip to full well capacity ─────────────────────────────────────
    photoelectrons = np.clip(photoelectrons, 0, full_well_capacity)

    # ── Step 4: Add Gaussian read noise ────────────────────────────────────────
    # Read noise is in electrons, applied before gain conversion.
    # Use float here to allow negative values from noise (physically real for low signal).
    signal_with_noise = photoelectrons.astype(np.float64) + rng.normal(
        loc=0.0, scale=sigma_read, size=(H, W)
    )

    # ── Step 5: Convert to DN and clip ─────────────────────────────────────────
    image_dn = signal_with_noise / gain

    # Clip to valid DN range [0, FWC/gain]
    image_dn = np.clip(image_dn, 0.0, full_well_capacity / gain)

    return {
        'image': image_dn.astype(output_dtype),
        'photoelectrons': photoelectrons,
        'photon_counts_raw': photon_counts,
        'n_frames_used': n_quanta_frames,
        'exposure_info': {
            'frame_start': frame_start,
            'frame_end': frame_end,
            'sigma_read_electrons': sigma_read,
            'qe_cmos': qe_cmos,
            'gain_e_per_dn': gain,
            'full_well_capacity_e': full_well_capacity,
        }
    }


def simulate_multiple_exposures(
    spad_frames: np.ndarray,
    n_quanta_frames: int,
    n_exposures: Optional[int] = None,
    **kwargs
) -> np.ndarray:
    """
    Simulate a burst of conventional camera exposures, each of length n_quanta_frames.
    Useful for computing SNR or comparing with theoretical predictions.

    Returns array of shape [n_exposures, H, W].
    """
    N_total = spad_frames.shape[0]
    max_exposures = N_total // n_quanta_frames
    if n_exposures is None:
        n_exposures = max_exposures

    if n_exposures > max_exposures:
        raise ValueError(
            f"Only {max_exposures} non-overlapping exposures possible "
            f"from {N_total} frames with n_quanta_frames={n_quanta_frames}."
        )

    images = []
    for i in range(n_exposures):
        result = simulate_conventional_camera(
            spad_frames,
            n_quanta_frames=n_quanta_frames,
            frame_start=i * n_quanta_frames,
            **kwargs
        )
        images.append(result['image'])

    return np.stack(images, axis=0)

In [ ]:
keep_prob = 1.0
photon_counts_thin = ihpp_data.thin_frames_counts_da(photon_counts, keep_prob=keep_prob, dcr_rate=25/100000, seed=42)
cmos_frames = simulate_multiple_exposures(
    photon_counts_thin,
    n_quanta_frames=1,
    sigma_read=27.0,  # phantom camera read noise, probably close enough to fastcam?
    qe_cmos=0.46,  # fastcam QE at 630 nm
    gain=1.0,  # guessed, couldn't find
)
sum_frames = simulate_multiple_exposures(
    photon_counts_thin,
    n_quanta_frames=1,
    sigma_read=0.0,
    qe_cmos=1.0,
    gain=1.0,
)

V = photon_counts.shape[0] / 100000 * photon_counts.shape[1] * photon_counts.shape[2]
reconflux = flux_probing.uwb3d_patch_quanta(
    photon_counts_thin,
    patch_sizes=(400, 64, 64), hop_sizes=(100, 16, 16),
    window="hann", recon_shape=(201, 640, 280), V=V, alpha=1e-4
)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(10, 4))
for ax, img in zip(axes, [cmos_frames[10], sum_frames[10], reconflux[10], discr_flux[10] * keep_prob]):
    im = ax.imshow(img)
    fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
utils.to_video(cmos_frames, f"output/cavitation/cmos_frames_thin_{keep_prob}.mp4", playback_fps=30, cmap="gray")
utils.to_video(sum_frames, f"output/cavitation/sum_frames_thin_{keep_prob}.mp4", playback_fps=30, cmap="gray")
utils.to_video(reconflux, f"output/cavitation/reconflux_thin_{keep_prob}.mp4", playback_fps=30, cmap="gray")

In [ ]:
for keep_prob in [1.0, 0.1, 0.01, 0.001, 0.0001]:
    if keep_prob < 1.0:
        sigma_read = 27.0  # phantom camera read noise, probably close enough to fastcam?
        dcr_rate = 25/100000
    else:
        sigma_read = 0.0
        dcr_rate = 0.0
    photon_counts_thin = ihpp_data.thin_frames_counts_da(photon_counts, keep_prob=keep_prob, dcr_rate=dcr_rate, seed=42)
    cmos_frames = simulate_multiple_exposures(
        photon_counts_thin,
        n_quanta_frames=1,
        sigma_read=sigma_read,
        qe_cmos=0.46,  # fastcam QE at 630 nm
        gain=1.0,  # guessed, couldn't find
    )
    photon_counts_thin_clipped = np.clip(photon_counts_thin, 0, 1)
    # binary only no read noise
    sum_frames = simulate_multiple_exposures(
        photon_counts_thin_clipped,
        n_quanta_frames=1,
        sigma_read=0.0,
        qe_cmos=1.0,
        gain=1.0,
    )

    V = photon_counts.shape[0] / 100000 * photon_counts.shape[1] * photon_counts.shape[2]
    reconflux = flux_probing.uwb3d_patch_quanta(
        photon_counts_thin_clipped,
        patch_sizes=(400, 64, 64), hop_sizes=(100, 16, 16),
        window="hann", recon_shape=(201, 640, 280), V=V, alpha=1e-4
    )
    utils.to_video(cmos_frames, f"output/cavitation/cmos-sim-frames/cmos_frames_thin_{keep_prob}.mp4", playback_fps=30, cmap="gray")
    utils.to_video(sum_frames, f"output/cavitation/single-photon-sum-frames/sum_frames_thin_{keep_prob}.mp4", playback_fps=30, cmap="gray")
    utils.to_video(reconflux, f"output/cavitation/single-photon-reconstr-results/reconflux_thin_{keep_prob}.mp4", playback_fps=30, cmap="gray")

In [ ]:
imgs = utils.read_imgdir("data/phantom-synth/separated_videos/cavitation1", grayscale=True)
utils.to_video(imgs, "data/phantom-synth/separated_videos/cavitation1.mp4", playback_fps=30, cmap="gray")

In [ ]:
imgs

In [ ]:
plt.figure(figsize=(20, 10))  # adjust width and height as needed
plt.imshow(frames[1000], cmap="gray")
plt.show()

In [ ]:
frames.shape[0]

In [ ]:
frames.shape[0] / 201

In [ ]:
T, H, W = frames.shape
nframesum = 128
frames_grouped = frames[:T - T % nframesum].reshape(-1, nframesum, H, W).sum(axis=1)

In [ ]:
plt.imshow(frames_grouped[10], cmap="gray")

In [ ]:
# utils.to_video(frames_grouped, config.OUTPUT_DIR / "results/phantom-synth/cavitation1/cavitation1_16xsum.mp4", playback_fps=30, cmap="gray")
utils.to_video(frames_grouped, f"output/results/phantom-synth/cavitation1/cavitation1_{nframesum}xsum.mp4", playback_fps=30, cmap="gray")

In [ ]:
#!/usr/bin/env python3
"""
Crop all MP4 videos in the specified folders and root file with the same crop.
Uses ffmpeg under the hood via subprocess.
"""

import subprocess
import sys
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────────

OUTPUT_ROOT = Path("output/cavitation")  # <-- change this

# Crop parameters: width:height:x:y  (x, y = top-left corner of crop region)
CROP_W = 280
CROP_H = 136
CROP_X = 0
CROP_Y = 158

# Folders containing MP4s (all .mp4 files inside will be cropped)
VIDEO_FOLDERS = [
    "cmos-sim-frames",
    "single-photon-reconstr-results",
    "single-photon-sum-frames",
]

# Individual video files at the root
ROOT_VIDEOS = [
    "cavitation1.mp4",
]

# Where to write cropped outputs (mirrors the source structure)
CROPPED_SUFFIX = "_cropped"   # appended to each output filename stem
OUTPUT_SUBDIR  = "cropped"    # sub-folder created next to each source folder
                               # set to "" to write alongside originals

# ── Helpers ───────────────────────────────────────────────────────────────────

def crop_video(src: Path, dst: Path, w: int, h: int, x: int, y: int) -> bool:
    """Run ffmpeg to crop src → dst. Returns True on success."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        "ffmpeg", "-y",
        "-i", str(src),
        "-vf", f"crop={w}:{h}:{x}:{y}",
        "-c:v", "libx264",   # re-encode; swap for "copy" only if crop aligns to codec blocks
        "-crf", "18",         # quality (0=lossless, 51=worst); 18 is visually lossless
        "-preset", "fast",
        "-c:a", "copy",
        str(dst),
    ]
    print(f"  {src.name}  →  {dst}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  [ERROR] ffmpeg failed:\n{result.stderr}", file=sys.stderr)
        return False
    return True


def dest_path(src: Path, root: Path) -> Path:
    """Compute destination path for a source video."""
    if OUTPUT_SUBDIR:
        return src.parent / OUTPUT_SUBDIR / (src.stem + CROPPED_SUFFIX + src.suffix)
    else:
        return src.parent / (src.stem + CROPPED_SUFFIX + src.suffix)


# ── Main ──────────────────────────────────────────────────────────────────────

sources: list[Path] = []

# Collect videos from folders
for folder in VIDEO_FOLDERS:
    folder_path = OUTPUT_ROOT / folder
    if not folder_path.is_dir():
        print(f"[WARN] Folder not found, skipping: {folder_path}")
        continue
    mp4s = sorted(folder_path.glob("*.mp4"))
    if not mp4s:
        print(f"[WARN] No MP4s found in: {folder_path}")
    sources.extend(mp4s)

# Collect root-level videos
for name in ROOT_VIDEOS:
    p = OUTPUT_ROOT / name
    if not p.is_file():
        print(f"[WARN] File not found, skipping: {p}")
        continue
    sources.append(p)

if not sources:
    print("No videos found. Check OUTPUT_ROOT and folder names.")
    sys.exit(1)

print(f"Found {len(sources)} video(s). Crop: {CROP_W}x{CROP_H} at ({CROP_X},{CROP_Y})\n")

ok = err = 0
for src in sources:
    dst = dest_path(src, OUTPUT_ROOT)
    if crop_video(src, dst, CROP_W, CROP_H, CROP_X, CROP_Y):
        ok += 1
    else:
        err += 1

print(f"\nDone. {ok} succeeded, {err} failed.")

# iccv fan thing

In [ ]:
from sptflux import ihpp_data, flux_probing, utils, plotting, config, datastructs
import matplotlib.pyplot as plt

In [ ]:
import os
import json
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np


# -----------------------------
# Window helpers (sqrt Hann)
# -----------------------------

def hann_1d(n: int) -> np.ndarray:
    """Symmetric Hann. Matches np.hanning."""
    if n <= 1:
        return np.ones((n,), dtype=np.float32)
    return np.hanning(n+1).astype(np.float32)[:-1]

def sqrt_hann_2d(ph: int, pw: int, eps: float = 0.0) -> np.ndarray:
    """
    2D separable sqrt Hann window:
      w2 = hann_y ⊗ hann_x
      w  = sqrt(w2)
    eps can be used if you want to avoid exact zeros on edges.
    """
    wy = hann_1d(ph)
    wx = hann_1d(pw)
    w2 = np.outer(wy, wx).astype(np.float32)
    if eps > 0:
        w2 = np.clip(w2, eps, None)
    return np.sqrt(w2).astype(np.float32)  # (ph, pw)


# -----------------------------
# Data model / manifest
# -----------------------------

@dataclass
class SpatialNUSTFTConfig:
    H: int
    W: int
    patch_h: int
    patch_w: int
    hop_h: int
    hop_w: int
    pad_h: int
    pad_w: int
    # coordinate conventions
    # We treat pixel indices as spanning [0, H) and [0, W), and normalize within a patch:
    #   x_local = (x_pix_in_patch + 0.5) / patch_w
    #   y_local = (y_pix_in_patch + 0.5) / patch_h
    # and time is already normalized to [0,1].
    time_is_normalized_01: bool = True


@dataclass
class PatchRecord:
    patch_id: int
    patch_start_y_padded: int  # top-left in padded pixel coords
    patch_start_x_padded: int
    # for debugging / convenience:
    n_events: int
    npz_file: str


def _ensure_dir(p: Union[str, Path]) -> Path:
    p = Path(p)
    p.mkdir(parents=True, exist_ok=True)
    return p


# -----------------------------
# Event cube access
# -----------------------------

def _iter_patch_events(
    events_hw: Union[List[List[np.ndarray]], np.ndarray],
    H: int,
    W: int,
    patch_start_y_padded: int,
    patch_start_x_padded: int,
    patch_h: int,
    patch_w: int,
    pad_h: int,
    pad_w: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Gather all events that fall into a spatial patch.

    events_hw: HxW jagged container where events_hw[y][x] is 1D array of timestamps in [0,1].
    Patch is defined in *padded pixel coords*.
    Original pixel (y,x) is located at padded coords (y+pad_h, x+pad_w).

    Returns:
        x_local, y_local, t_local, w_sqrt_per_event
    where x_local,y_local,t_local are in [0,1] patch coordinates.
    """
    # Determine which original pixels overlap this patch:
    # patch covers padded y in [sy, sy+patch_h), x in [sx, sx+patch_w)
    sy, sx = patch_start_y_padded, patch_start_x_padded

    y0_orig = sy - pad_h
    x0_orig = sx - pad_w
    y1_orig = y0_orig + patch_h
    x1_orig = x0_orig + patch_w

    # Overlap with original [0,H) x [0,W)
    oy0 = max(0, y0_orig)
    ox0 = max(0, x0_orig)
    oy1 = min(H, y1_orig)
    ox1 = min(W, x1_orig)

    if oy0 >= oy1 or ox0 >= ox1:
        return (
            np.empty((0,), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
        )

    # Local pixel offsets within patch for each contributing original pixel
    # local_y_pix = (orig_y - y0_orig), local_x_pix = (orig_x - x0_orig)
    # Note y0_orig/x0_orig can be negative (because of padding), so local indices can start >0.
    local_y_start = oy0 - y0_orig
    local_x_start = ox0 - x0_orig

    # Precompute sqrt Hann weights on the patch grid (per pixel)
    w_sqrt = sqrt_hann_2d(patch_h, patch_w)  # (ph,pw)

    xs = []
    ys = []
    ts = []
    ws = []

    for oy in range(oy0, oy1):
        ly = oy - y0_orig  # in [0, patch_h)
        for ox in range(ox0, ox1):
            lx = ox - x0_orig  # in [0, patch_w)

            t_arr = events_hw[oy][ox]
            if t_arr is None:
                continue
            t_arr = np.asarray(t_arr)
            if t_arr.size == 0:
                continue

            # Normalize spatial coords to [0,1] within patch using pixel centers
            # x_local = (lx + 0.5)/patch_w, y_local = (ly + 0.5)/patch_h
            x_local = (lx) / float(patch_w)
            y_local = (ly) / float(patch_h)

            # Repeat per-event
            n = t_arr.size
            xs.append(np.full((n,), x_local, dtype=np.float32))
            ys.append(np.full((n,), y_local, dtype=np.float32))
            ts.append(t_arr.astype(np.float32, copy=False))

            # sqrt Hann weight at that pixel repeated per-event
            w = float(w_sqrt[ly, lx])
            ws.append(np.full((n,), w, dtype=np.float32))

    if len(ts) == 0:
        return (
            np.empty((0,), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
        )

    x_local = np.concatenate(xs, axis=0)
    y_local = np.concatenate(ys, axis=0)
    t_local = np.concatenate(ts, axis=0)
    w_local = np.concatenate(ws, axis=0)

    # Optionally: enforce [0,1] just in case (timestamps should already be)
    t_local = np.clip(t_local, 0.0, 1.0)

    return x_local, y_local, t_local, w_local

def starts_1d(L, P, S):
    if L <= P:
        return [0]
    starts = list(range(0, L - P + 1, S))
    if starts[-1] != (L - P):
        starts.append(L - P)
    return starts

# -----------------------------
# Analysis
# -----------------------------

def spatial_nustft_analysis(
    events_hw: Union[List[List[np.ndarray]], np.ndarray],
    fx: np.ndarray,
    fy: np.ndarray,
    ft: np.ndarray,
    out_dir: Union[str, Path],
    patch_size: Tuple[int, int],
    hop: Tuple[int, int],
    *,
    # pass-through to uwb3d_global_NU
    alpha: Optional[float] = None,
    eps: float = 1e-6,
    cuda: bool = False,
    chunk_size: Optional[int] = None,
    ft_range_max: Optional[int] = None,
    clip_flux: bool = True,
    use_recon_nyquist: bool = True,
    # padding policy: "patch_size" on both sides
    pad: Optional[Tuple[int, int]] = None,
) -> Dict[str, Any]:
    """
    Spatial-only NU-STFT analysis.

    Saves one NPZ per spatial patch with:
      - F_bd (complex), fx_bd, fy_bd, ft_bd (float) for dominant frequencies only
      - patch metadata (start indices in padded coords, patch size)
      - config snapshot

    Returns:
      manifest dict that is also written to out_dir/manifest.json
    """
    out_dir = _ensure_dir(out_dir)

    H = len(events_hw)
    W = len(events_hw[0]) if H > 0 else 0
    ph, pw = patch_size
    sh, sw = hop

    if pad is None:
        pad_h, pad_w = ph, pw
    else:
        pad_h, pad_w = pad

    cfg = SpatialNUSTFTConfig(
        H=H, W=W,
        patch_h=ph, patch_w=pw,
        hop_h=sh, hop_w=sw,
        pad_h=pad_h, pad_w=pad_w,
    )

    Hp = H + 2 * pad_h
    Wp = W + 2 * pad_w

    patch_records: List[PatchRecord] = []
    patch_id = 0

    # Patch grid over padded coordinates
    # top-left starts at 0, ends so patch fits: <= Hp-ph
    ys = starts_1d(Hp, ph, sh)
    xs = starts_1d(Wp, pw, sw)
    for sy in ys:
        for sx in xs:
            x_local, y_local, t_local, w_local = _iter_patch_events(
                events_hw, H, W,
                patch_start_y_padded=sy,
                patch_start_x_padded=sx,
                patch_h=ph, patch_w=pw,
                pad_h=pad_h, pad_w=pad_w,
            )

            if x_local.size == 0:
                # still record empty patches? usually no; skip to save space
                continue

            # weights correspond to sqrt Hann at each event location
            # Domain is normalized to [0,1]^3 within the patch, so V = 1.0 is consistent.
            # If your uwb3d_global_NU expects weights already scaled by 1/sqrt(V), this is fine.
            weights = w_local.astype(np.complex64)  # allow complex weights if needed

            # Call user-provided NUFFT analysis (dominant freqs only)
            # IMPORTANT: full_spectrum=False => returns (flux_3dfft?, F_bd, bound, (ft_bd, fy_bd, fx_bd))
            # Since we aren't reconstructing in analysis, do not pass xx/yy/tt.
            ret = flux_probing.uwb3d_global_NU(
                x_local, y_local, t_local,
                fx=fx, fy=fy, ft=ft,
                xx=None, yy=None, tt=None,
                grid_freqs=True,
                grid_coords=False,
                V=1.0,
                weights=weights,
                alpha=alpha,
                full_spectrum=False,
                herm_dropped=None,
                use_recon_nyquist=use_recon_nyquist,
                clip_flux=clip_flux,
                chunk_size=chunk_size,
                ft_range_max=ft_range_max,
                eps=eps,
                cuda=cuda,
            )

            # Unpack robustly: some implementations may omit flux_3dfft when no coords are provided.
            # Expected:
            #   (F_bd, bound, (ft_bd, fy_bd, fx_bd))
            # or (flux_none, F_bd, bound, (ft_bd, fy_bd, fx_bd))
            if len(ret) == 3:
                F_bd, bound, freq_triplets = ret
            elif len(ret) == 4:
                _flux, F_bd, bound, freq_triplets = ret
            else:
                raise ValueError(f"Unexpected uwb3d_global_NU return length: {len(ret)}")

            ft_bd, fy_bd, fx_bd = freq_triplets

            # Save patch
            npz_name = f"patch_{patch_id:06d}.npz"
            npz_path = out_dir / npz_name

            np.savez_compressed(
                npz_path,
                # dominant spectrum
                F_bd=np.asarray(F_bd),
                fx_bd=np.asarray(fx_bd),
                fy_bd=np.asarray(fy_bd),
                ft_bd=np.asarray(ft_bd),
                bound=np.asarray(bound),
                # patch placement
                patch_id=np.int32(patch_id),
                patch_start_y_padded=np.int32(sy),
                patch_start_x_padded=np.int32(sx),
                patch_h=np.int32(ph),
                patch_w=np.int32(pw),
                # counts (optional)
                n_events=np.int32(x_local.size),
                # store config snapshot for safety
                config_json=json.dumps(asdict(cfg)),
            )

            patch_records.append(PatchRecord(
                patch_id=patch_id,
                patch_start_y_padded=sy,
                patch_start_x_padded=sx,
                n_events=int(x_local.size),
                npz_file=npz_name,
            ))
            patch_id += 1

    manifest = {
        "config": asdict(cfg),
        "n_patches": len(patch_records),
        "patches": [asdict(r) for r in patch_records],
        "files": {
            "type": "per-patch-npz",
            "pattern": "patch_{patch_id:06d}.npz",
        }
    }

    with open(out_dir / "manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)

    return manifest


# -----------------------------
# Synthesis (WOLA)
# -----------------------------

def spatial_nustft_synthesis(
    analysis_dir: Union[str, Path],
    xx: np.ndarray,
    yy: np.ndarray,
    tt: np.ndarray,
    *,
    eps: float = 1e-6,
    cuda: bool = False,
    chunk_size: int = 10000,
    ft_range_max: int = 5000,
    # optional: if you want to ignore patches with too few freqs
    min_nfreq: int = 1,
) -> np.ndarray:
    """
    Weighted overlap-add synthesis on an arbitrary reconstruction grid.

    Args:
      analysis_dir: directory created by spatial_nustft_analysis (contains manifest + patch_*.npz)
      xx, yy, tt: 1D arrays of global reconstruction coordinates.
        Convention here: xx spans [0,1] over the *original* (unpadded) width, yy spans [0,1] over height.
        Time tt is whatever you want (recommended normalized to [0,1] to match your analysis).

    Returns:
      recon with shape (len(tt), len(yy), len(xx))
    """
    analysis_dir = Path(analysis_dir)
    with open(analysis_dir / "manifest.json", "r") as f:
        manifest = json.load(f)

    cfg = SpatialNUSTFTConfig(**manifest["config"])
    H, W = cfg.H, cfg.W
    ph, pw = cfg.patch_h, cfg.patch_w
    pad_h, pad_w = cfg.pad_h, cfg.pad_w
    sh, sw = cfg.hop_h, cfg.hop_w

    # Precompute patch sqrt Hann window
    w_sqrt_patch = sqrt_hann_2d(ph, pw)  # (ph,pw)
    w_power_patch = w_sqrt_patch ** 2    # Hann (ph,pw)

    # Output accumulator and normalization buffer
    T = len(tt)
    Y = len(yy)
    X = len(xx)
    accum = np.zeros((T, Y, X), dtype=np.float32)
    norm = np.zeros((Y, X), dtype=np.float32)

    # Map global coords (xx,yy in [0,1]) into padded pixel-coordinate space
    # We interpret xx=0 as left edge of original image and xx=1 as right edge.
    # Using continuous pixel coords in [0,W) and [0,H):
    x_pix_padded = xx.astype(np.float32) * float(W) + float(pad_w)
    y_pix_padded = yy.astype(np.float32) * float(H) + float(pad_h)

    # For each patch, determine which xx,yy indices lie inside its support in padded pixel coords.
    # Patch covers x in [sx, sx+pw), y in [sy, sy+ph).
    for pr in manifest["patches"]:
        npz_path = analysis_dir / pr["npz_file"]
        data = np.load(npz_path, allow_pickle=False)

        F_bd = data["F_bd"]
        if F_bd.size < min_nfreq:
            continue
        fx_bd = data["fx_bd"]
        fy_bd = data["fy_bd"]
        ft_bd = data["ft_bd"]
        F_bd[ft_bd > 0] *= 2.0

        sy = int(data["patch_start_y_padded"])
        sx = int(data["patch_start_x_padded"])

        # Find index ranges in xx,yy that fall in this patch support
        ix = np.where((x_pix_padded >= sx) & (x_pix_padded < sx + pw))[0]
        iy = np.where((y_pix_padded >= sy) & (y_pix_padded < sy + ph))[0]
        if ix.size == 0 or iy.size == 0:
            continue

        # Convert selected coords to local patch-normalized coords in [0,1]
        x_local = (x_pix_padded[ix] - float(sx)) / float(pw)  # (len(ix),)
        y_local = (y_pix_padded[iy] - float(sy)) / float(ph)  # (len(iy),)

        # Build target coordinate triplets for the inverse NUFFT:
        # mesh over (t,y,x) and then flatten
        # NOTE: adapt ordering if your spt_ihpp_inufft3_chunked expects different conventions.
        tt_local = tt.astype(np.float32)

        # Create broadcast grids
        # shapes: (T, Iy, Ix)
        Tt = tt_local[:, None, None]
        Yy = y_local[None, :, None]
        Xx = x_local[None, None, :]

        # Flatten into 1D arrays for coordgrid=None call
        t_flat = np.broadcast_to(Tt, (T, iy.size, ix.size)).ravel()
        y_flat = np.broadcast_to(Yy, (T, iy.size, ix.size)).ravel()
        x_flat = np.broadcast_to(Xx, (T, iy.size, ix.size)).ravel()

        # Reconstruct patch values on just the needed subgrid
        g_flat = flux_probing.spt_ihpp_inufft3_chunked(
            fx=fx_bd, fy=fy_bd, ft=ft_bd,
            F=F_bd,
            coordgrid=None,
            x=x_flat, y=y_flat, t=t_flat,
            chunk_size=chunk_size,
            ft_range_max=ft_range_max,
            cuda=cuda,
            eps=eps,
        )

        # Reshape to (T, Iy, Ix)
        g_patch = np.asarray(g_flat.real, dtype=np.float32).reshape(T, iy.size, ix.size)

        # Apply synthesis sqrt window on the corresponding subwindow of the patch
        # Need local integer pixel indices within the patch for each selected y/x coordinate.
        # We approximate by mapping continuous coords back to pixel bins:
        #   px = floor(x_pix_padded - sx), py = floor(y_pix_padded - sy)
        px = np.floor(x_pix_padded[ix] - float(sx)).astype(int)
        py = np.floor(y_pix_padded[iy] - float(sy)).astype(int)
        px = np.clip(px, 0, pw - 1)
        py = np.clip(py, 0, ph - 1)

        w_sqrt_sub = w_sqrt_patch[np.ix_(py, px)]   # (Iy, Ix)
        w_pow_sub = w_power_patch[np.ix_(py, px)]   # (Iy, Ix)

        # Accumulate with WOLA:
        # accum += w_sqrt * g_patch
        # norm  += w_sqrt^2
        accum[:, iy[:, None], ix[None, :]] += g_patch * w_sqrt_sub[None, :, :]
        norm[iy[:, None], ix[None, :]] += w_pow_sub

    # Normalize (avoid divide-by-zero)
    denom = np.maximum(norm, 1e-12).astype(np.float32)
    recon = accum / denom[None, :, :]
    return recon

In [ ]:
########## 3D FFT variable setup ##########
T_exp = 0.1  # exposure time (how much data to read from the start time)

# max frequencies to sweep
step = 1  # multiply by 10 for actual Hz step along time
fy_max = 45
fx_max = 69

# chunk sizes for processing
chunk_size = 100_000_000
ft_range_max = 10_000

# filepaths
output_parent = config.OUTPUT_DIR / "results/async-spad/AC_fan"
data_folder = config.DATA_DIR / "async-spad/AC_fan"

alpha = 0.00001
eps = 1e-12
keep_prob = 1.0
########## End variable setup ##########

# data appears to have HxW = 91x138 pixels
src_W = 138
src_H = 91
V = src_W * src_H * T_exp

fx = np.fft.fftshift(np.fft.fftfreq(src_W, 1 / src_W))
fy = np.fft.fftshift(np.fft.fftfreq(src_H, 1 / src_H))
fan_freq = 105.7
laser_freq = 1.9999440e7
freqs_ab_harmonics = np.arange(0, 15e9 + laser_freq, laser_freq).reshape(1,-1) + np.arange(-10, 11).reshape(-1,1) * fan_freq
freqs_ab_new = np.sort(np.unique((freqs_ab_harmonics.flatten())))
ft_harmonic = freqs_ab_new[freqs_ab_new >= 0] / (1 / T_exp)
# ft_gb = np.arange(0, 11) * fan_freq
ft_gb = np.arange(0, fan_freq * 11 / (1 / T_exp))

# points_r, _ = ihpp_data.read_spadfolder(data_folder / "R", h=src_H, w=src_W, T_exp=T_exp, normalize=True)
# t_r, y_r, x_r = points_r[:, 0], points_r[:, 1], points_r[:, 2]
# points_g, _ = ihpp_data.read_spadfolder(data_folder / "G", h=src_H, w=src_W, T_exp=T_exp, normalize=True)
# t_g, y_g, x_g = points_g[:, 0], points_g[:, 1], points_g[:, 2]
# points_b, _ = ihpp_data.read_spadfolder(data_folder / "B", h=src_H, w=src_W, T_exp=T_exp, normalize=True)
# t_b, y_b, x_b = points_b[:, 0], points_b[:, 1], points_b[:, 2]
(t_r, y_r, x_r), pix_ts_r, _ = ihpp_data.read_asyncspad_zarr(data_folder / "R.zarr", load_data=True)
t_r, y_r, x_r = t_r / T_exp, y_r / src_H, x_r / src_W
(t_g, y_g, x_g), pix_ts_g, _ = ihpp_data.read_asyncspad_zarr(data_folder / "G.zarr", load_data=True)
t_g, y_g, x_g = t_g / T_exp, y_g / src_H, x_g / src_W
(t_b, y_b, x_b), pix_ts_b, _ = ihpp_data.read_asyncspad_zarr(data_folder / "B.zarr", load_data=True)
t_b, y_b, x_b = t_b / T_exp, y_b / src_H, x_b / src_W

data_str_format = "-".join(["spadfan", "STFT", f"thin{keep_prob:.3f}", f"freqharmx{fy_max}x{fx_max}", f"alpha{alpha:.5f}"])
output_folder = output_parent / data_str_format
output_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
for i in range(src_H):
    for j in range(src_W):
        pix_ts_r[i][j] = pix_ts_r[i][j] / T_exp
        pix_ts_g[i][j] = pix_ts_g[i][j] / T_exp
        pix_ts_b[i][j] = pix_ts_b[i][j] / T_exp

In [ ]:
spatial_nustft_analysis(
    events_hw=pix_ts_g,
    fx=fx, fy=fy, ft=ft_gb,
    out_dir=output_folder / "analysis-g",
    patch_size=(60, 60),
    hop=(20, 20),
    alpha=alpha,
    eps=eps,
    cuda=False,
    chunk_size=chunk_size,
    ft_range_max=ft_range_max,
)

In [ ]:
all_fps =     [300, 1_000, 2_000, 10_000, 100_000, 200_000, 30_000_000, 40_000_000, 200_000_000, 10_000_000_000, 30_000_000_000, 199_994_400_000]
all_nframes = [30,  100,   200,   1000,   1000,    1000,    1000,       1000,       1000,        1000,           1000,           1000]
all_tstart =  [0,   0,     0,     0,      0,       0,       0,          0,          0,           0,              0,              0.03000004160118163]
all_eps =     [1e-6, 1e-6, 1e-6,  1e-6,   1e-6,    1e-6,    1e-9,       1e-9,       1e-9,        1e-12,          1e-12,          1e-12]
i = 4
fps = all_fps[i]
nframes = all_nframes[i]
tstart = all_tstart[i]
eps = all_eps[i]
Nt = nframes
tend = tstart + Nt / fps
xx = np.linspace(0, 1, src_W, endpoint=False)
yy = np.linspace(0, 1, src_H, endpoint=False)
tt = np.linspace(tstart, tend, Nt)

In [ ]:
flux = spatial_nustft_synthesis(
    output_folder / "analysis-g",
    xx, yy, tt, chunk_size=chunk_size, ft_range_max=ft_range_max
)
flux = np.clip(flux, 0, None)

In [ ]:
plt.imshow((flux[120] / np.max(flux)) ** (1/2.2))
plt.colorbar()

In [ ]:
ihpp_data.to_video(flux, output_folder / f"idk.avi", playback_fps=60, gamma=1/2.2, vmin=0, cmap="grey")

In [ ]:
nfreqs = freqs_ab_new.size * fx.size * fy.size

In [ ]:
from scipy.stats import chi, chi2
chi2.ppf(1-0.00010, 2) ** 0.5, chi2.ppf(1-1/nfreqs, 2) ** 0.5

In [ ]:
2 * (chi.ppf(1-0.00010, 2) ** 0.5), 2*(chi.ppf(1-1/nfreqs, 2) ** 0.5)

In [ ]:
freqinfopath = "output/results/async-spad/AC_fan/spadfan-thin1.000-freqharmx45x69-alpha0.00010/freqinfo-g.npz"
freqinfo = np.load(freqinfopath)
F = freqinfo["F"]
ft = freqinfo["ft"]
fy = freqinfo["fy"]
fx = freqinfo["fx"]

In [ ]:
xx = np.linspace(0, 1, src_W)
yy = np.linspace(0, 1, src_H)
tt = np.linspace(0, 1, 10)
TT, YY, XX = np.meshgrid(tt, yy, xx, indexing="ij")
g = flux_probing.spt_ihpp_inufft3(fx, fy, ft, F, XX.ravel(), YY.ravel(), TT.ravel(), eps=1e-6)

In [ ]:
fx

In [ ]:
ft.max()

In [ ]:
fx_freq = np.fft.fftfreq(139, 1/139)
fy_freq = np.fft.fftfreq(92, 1/92)
ft_freq = np.fft.rfftfreq(232, 1/232)
F_grid = np.zeros((ft_freq.size, fy_freq.size, fx_freq.size), dtype="complex128")

In [ ]:
F_grid[ft.astype("int"), fy.astype("int"), fx.astype("int")] = F

In [ ]:
F_grid_shift = np.fft.fftshift(F_grid)

In [ ]:
F_grid_shift_flat = np.max(np.abs(F_grid_shift), axis=0)

In [ ]:
# use log colorbar for better visibility
from matplotlib import cm
from matplotlib import colors
plt.imshow(F_grid_shift_flat, norm=colors.LogNorm(vmin=1, vmax=F_grid_shift_flat.max()/1))
plt.colorbar()

In [ ]:
np.fft.fftshift(fy_freq)[46]

In [ ]:
plt.plot(F_grid_shift_flat[46, :])

In [ ]:
ft_freq

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(
    go.Scatter3d(
        x=fx, y=fy, z=ft,
        mode="markers",
        marker=dict(
            size=sizes ** 0.5,
            color=absF,
            colorscale="Viridis",
            colorbar=dict(title="|F|"),
            opacity=0.7,
            showscale=True,
        ),
        hovertemplate="ft=%{x}<br>fy=%{y}<br>fx=%{z}<br>|F|=%{marker.color}<extra></extra>",
    )
)

fig.update_layout(
    scene=dict(xaxis_title="ft", yaxis_title="fy", zaxis_title="fx"),
    margin=dict(l=0, r=0, b=0, t=30),
)
fig.show()

In [ ]:
ft.max()

In [ ]:
g = g.reshape((10, src_H, src_W)).real

In [ ]:
plt.imshow(g[0])

In [ ]:
import numpy as np
path = "/Volumes/CrucialX10/uwb3d-files/output/results/async-spad/AC_fan/testing-gpu/freqinfo-r.npz"
data = np.load(path)

In [ ]:
data["F"]

In [ ]:
data["ft"]

In [ ]:
path2 = "/Volumes/CrucialX10/uwb3d-files/output/results/async-spad/AC_fan/spadfan-thin1.000-freqharmx45x69-alpha0.00001/freqinfo-r.npz"
data2 = np.load(path2)

In [ ]:
data2["F"]

In [ ]:
data["ft"]

In [ ]:
import numpy as np
import finufft
import matplotlib.pyplot as plt

def psf_test_finufft_3d(
    H=128, W=128, T=256,
    n_events=2_000_000,
    spot_sigma_xy=0.8,          # in pixel units; 0 => delta-like spot
    spot_center_xy=None,        # (cx, cy) in pixel coords
    random_seed=0,
    # NUFFT frequency grid size (number of Fourier modes)
    # Bigger => sharper PSF, but more compute/memory.
    ms=(128, 128, 256),
    eps=1e-6,
):
    """
    PSF test for a 3D NUFFT pipeline:
      - generate synthetic events for a stationary spatial spot + constant time
      - compute Fourier coefficients via NUFFT type-1
      - reconstruct onto a regular grid via NUFFT type-2
    """

    rng = np.random.default_rng(random_seed)

    if spot_center_xy is None:
        cx, cy = (W - 1) / 2.0, (H - 1) / 2.0
    else:
        cx, cy = spot_center_xy
    print(cx, cy)

    # --- 1) Generate synthetic event coordinates in pixel units ---
    # Time: uniform in [0, 1)
    t_pix = rng.random(n_events) * (T - 1)

    # Space: either delta-like (all at center) or Gaussian jitter around center
    if spot_sigma_xy <= 0:
        x_pix = np.full(n_events, cx)
        y_pix = np.full(n_events, cy)
    else:
        x_pix = rng.normal(loc=cx, scale=spot_sigma_xy, size=n_events)
        y_pix = rng.normal(loc=cy, scale=spot_sigma_xy, size=n_events)

    # Clip to sensor bounds
    x_pix = np.clip(x_pix, 0, W - 1)
    y_pix = np.clip(y_pix, 0, H - 1)

    # --- 2) Map coordinates to FINUFFT domain [-pi, pi) ---
    # FINUFFT expects nonuniform points x,y,z in [-pi,pi).
    # We'll map pixel coordinates linearly:
    def to_pm_pi(u, N):
        return (u / N) * 2.0 * np.pi - np.pi

    x = to_pm_pi(x_pix, W)
    y = to_pm_pi(y_pix, H)
    z = to_pm_pi(t_pix, T)

    # Complex weights for each event (all 1's)
    c = np.ones(n_events, dtype=np.complex128)

    # --- 3) NUFFT type-1: points -> Fourier modes on a uniform k-grid ---
    # Returns F[k1,k2,k3] with k ranges:
    #   k1 = -ms[0]/2 ... ms[0]/2-1 (similarly for others)
    print("Computing NUFFT type-1 (events -> Fourier coeffs)...")
    F = finufft.nufft3d1(x, y, z, c, n_modes=ms, eps=eps, isign=-1)

    # Optional: normalize (depends on your convention).
    # FINUFFT uses an unnormalized convention; for PSF shape it usually doesn't matter.
    # If you want, uncomment:
    # F = F / n_events

    # --- 4) NUFFT type-2: Fourier modes -> values on a uniform voxel grid ---
    # We'll reconstruct at every voxel center.
    # Create uniform grid of points in [-pi,pi)
    print("Computing NUFFT type-2 (Fourier coeffs -> voxel grid)...")
    xs = to_pm_pi(np.arange(W), W)
    ys = to_pm_pi(np.arange(H), H)
    zs = to_pm_pi(np.arange(T), T)

    XX, YY, ZZ = np.meshgrid(xs, ys, zs, indexing="xy")
    # Flatten for finufft call
    xg = XX.ravel()
    yg = YY.ravel()
    zg = ZZ.ravel()

    # Type-2 expects Fourier array input and returns values at given points
    # It computes: f(x_j)=sum_k F_k * exp(± i k·x_j)
    f = finufft.nufft3d2(xg, yg, zg, F, eps=eps, isign=+1)
    f = f.reshape(H, W, T)

    # For visualization: magnitude / real part
    # Since c is real positive and we used symmetric-ish grids, real part is often meaningful.
    f_real = np.real(f)

    # --- 5) Visualize slices to look for cross/grid PSF ---
    mid_t = T // 2
    mid_y = H // 2
    mid_x = W // 2

    fig, axs = plt.subplots(2, 2, figsize=(10, 9))
    axs = axs.ravel()

    im0 = axs[0].imshow(f_real[:, :, mid_t], cmap="gray")
    axs[0].set_title(f"Spatial slice at t={mid_t}")
    plt.colorbar(im0, ax=axs[0], fraction=0.046, pad=0.04)

    im1 = axs[1].imshow(f_real[mid_y, :, :], cmap="gray", aspect="auto")
    axs[1].set_title(f"X-T slice at y={mid_y}")
    plt.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)

    im2 = axs[2].imshow(f_real[:, mid_x, :], cmap="gray", aspect="auto")
    axs[2].set_title(f"Y-T slice at x={mid_x}")
    plt.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04)

    # Log magnitude of Fourier coefficients (helps see anisotropy near axes)
    F_shift = np.fft.fftshift(F)
    im3 = axs[3].imshow(np.log1p(np.abs(F_shift[:, :, ms[2]//2])), cmap="gray")
    axs[3].set_title("log(1+|F(kx,ky,kz=0)|) (center kz slice)")
    plt.colorbar(im3, ax=axs[3], fraction=0.046, pad=0.04)

    for ax in axs:
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.show()

    return f_real, F


if __name__ == "__main__":
    f_real, F = psf_test_finufft_3d(
        H=128, W=128, T=256,
        n_events=500_000,      # start smaller; increase once it runs
        spot_sigma_xy=0.0,     # delta-like spot
        spot_center_xy=(63.00,63.0),
        ms=(128, 128, 256),
        eps=1e-6,
    )

In [ ]:
import numpy as np
import finufft
import matplotlib.pyplot as plt

def analytic_delta_spectrum(ms, x0, y0, z0):
    # integer mode indices in FINUFFT ordering: k = -M/2,...,M/2-1
    kx = np.fft.fftfreq(ms[0]) * ms[0]
    ky = np.fft.fftfreq(ms[1]) * ms[1]
    kz = np.fft.fftfreq(ms[2]) * ms[2]
    # fftfreq gives 0,1,...,M/2-1,-M/2,...,-1 ordering; FINUFFT uses centered integer modes.
    # We'll build centered then ifftshift to match FINUFFT array ordering.
    kx_c = np.arange(-ms[0]//2, ms[0]//2)
    ky_c = np.arange(-ms[1]//2, ms[1]//2)
    kz_c = np.arange(-ms[2]//2, ms[2]//2)

    KX, KY, KZ = np.meshgrid(kx_c, ky_c, kz_c, indexing="ij")
    F_c = np.exp(-1j*(KX*x0 + KY*y0 + KZ*z0))
    # convert centered -> FINUFFT ordering
    F = np.fft.ifftshift(F_c)
    return F

def type2_only_recon(H=128, W=128, T=256, ms=(128,128,256), eps=1e-12, center_xy=(63.5,63.5), center_t=127.5):
    # coords in [-pi,pi)
    def to_pm_pi(u, N):
        return (u / N) * 2*np.pi - np.pi

    x0 = to_pm_pi(center_xy[0], W)
    y0 = to_pm_pi(center_xy[1], H)
    z0 = to_pm_pi(center_t, T)

    F = analytic_delta_spectrum(ms, x0, y0, z0)

    xs = to_pm_pi(np.arange(W), W)
    ys = to_pm_pi(np.arange(H), H)
    zs = to_pm_pi(np.arange(T), T)
    XX, YY, ZZ = np.meshgrid(xs, ys, zs, indexing="xy")

    f = finufft.nufft3d2(XX.ravel(), YY.ravel(), ZZ.ravel(), F, isign=+1, eps=eps)
    f = np.real(f.reshape(H, W, T))

    plt.imshow(f[:, :, T//2], cmap="gray")
    plt.title("Type-2 only recon (analytic delta spectrum)")
    plt.axis("off")
    plt.show()
    return f

# Run
_ = type2_only_recon()

In [ ]:
from pathlib import Path
from scipy.io import loadmat
from sptflux import ihpp_data
respath = Path("/Volumes/CrucialX10/uwb3d-files/output/results/uwb1d/async-spad/AC_fan/uwb_fan_thin10/249800.000MHz")
r = loadmat(respath / "rates_R.mat")["rates"].transpose(2, 0, 1)
g = loadmat(respath / "rates_G.mat")["rates"].transpose(2, 0, 1)
b = loadmat(respath / "rates_B.mat")["rates"].transpose(2, 0, 1)

In [ ]:
out_rgbvals = np.zeros((r.shape[0], r.shape[1], r.shape[2], 3), dtype=np.float32)
out_rgbvals[..., 1] = g
ihpp_data.to_video(out_rgbvals, respath / "whatthefcuk.avi", playback_fps=30, gamma=1/2.2)

# losing my sanity with scaling

In [ ]:
import numpy as np
from sptflux import flux_probing, ihpp_sim
from matplotlib import pyplot as plt

amp = 10000.0
max_rate = amp * 2

def fluxfunc(t):
    return amp * (np.sin(2 * np.pi * 5 * t) + 1)  # example: 5 Hz sine wave flux

tt = np.linspace(0, 1, 1000)
flux = fluxfunc(tt)
plt.plot(tt, flux)

In [ ]:
pts = ihpp_sim.simulate_ihpp1d(fluxfunc, 1.0, max_rate=max_rate, seed=42)

In [ ]:
pts

In [ ]:
plt.plot(tt, flux)
plt.scatter(pts, np.zeros_like(pts), s=10, color="red", alpha=0.5)

In [ ]:
n_bins = 100_000
bins = np.zeros(n_bins, dtype=np.int8)
indices = (pts * n_bins).astype(int).clip(0, n_bins - 1)
bins[indices] = 1

In [ ]:
f, F, bd = flux_probing.spt_ihpp_uwb1d([[pts]], tt, np.arange(0, 500+1, 1), T_exp=1.0, alpha=1e-4, type3=True)
recon_flux_quanta = flux_probing.uwb1d_quanta(bins.reshape(-1, 1, 1), alpha=1e-4, T_exp=1.0, Nt=10000)

In [ ]:
plt.plot(tt, flux)
plt.plot(tt, f[:, 0, 0])
plt.plot(np.linspace(0, 1, 10000), recon_flux_quanta[:, 0, 0])

In [ ]:
A = np.random.random((1000, 50, 50))

In [ ]:
mask = np.zeros(1000, dtype=bool)
mask[::10] = True
A[mask].shape

In [ ]:
round(0.5 / np.array(0.99))

In [ ]:
0.5 / np.array(0.99)

In [ ]:
np.array(0.99)

# intensity vs poisson spectra vis

In [ ]:
from sptflux import ihpp_data, utils
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
imgs = utils.read_imgdir("data/phantom-synth/separated_videos/stirfry-fps1000", grayscale=True)

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

outdir = Path("output/poster-vis")
outdir.mkdir(parents=True, exist_ok=True)

nframes = 41  # frames 0-40 inclusive
# imgs is (T, H, W)
frame_idxs = [0, 20, 40]


def save_frames(vol, prefix, idxs=frame_idxs, outdir=outdir):
    """Save frames at `idxs` as pngs, sharing one vmax across the set."""
    vmax = max(vol[i].max() for i in idxs)
    for i in idxs:
        plt.imsave(outdir / f"{prefix}_frame{i}.png", vol[i], cmap="gray", vmin=0, vmax=vmax)


# save the original frames 0, 20, 40 as pngs
save_frames(imgs, "orig")


In [ ]:
def save_tf_power_spectra(vol, prefix, outdir=outdir, nslices=3):
    """vol: (T, H, W) with time on axis 0.

    Do a 3D rfftn with time as the real (rfft) axis: full complex FFT over the
    spatial (H, W) axes, real FFT over time. Passing axes=(1, 2, 0) makes the
    time axis last in the axes list, so rfft is applied to it and axis 0 of the
    output holds the time frequencies. Save the log1p power spectra of the first
    `nslices` spatial slices along the time-frequency axis (spatially fftshifted),
    sharing one vmax across the set so the slices are comparable.
    """
    F = np.fft.rfftn(vol, axes=(1, 2, 0))  # -> (nfreq_t, H, W)
    powers = [np.log1p(np.abs(np.fft.fftshift(F[k])) ** 2) for k in range(nslices)]
    vmax = max(p.max() for p in powers)  # shared scale across the 3 slices
    for k, power in enumerate(powers):
        plt.imsave(outdir / f"{prefix}_tf{k}.png", power, cmap="magma", vmin=0, vmax=vmax)
    return F


In [ ]:
# original intensity frames -> time-frequency power spectra
F_intensity = save_tf_power_spectra(imgs[:nframes].astype(np.float64), "intensity")


In [ ]:
# Poisson-sample the frames relative to their intensity to simulate what a
# single-photon (SPAD) camera reads, at two thinning levels.
rng = np.random.default_rng(0)
norm = imgs[:nframes] / 255.0  # intensity in [0, 1]

poisson_levels = {
    # more photons: multi-photon counts, generally >1 photon/px but not binary
    "poisson_hi": dict(scale=20.0, clip=False),
    # fewer photons, clipped to 1 -> binary single-photon frames
    "poisson_lo": dict(scale=0.5, clip=True),
}

for prefix, cfg in poisson_levels.items():
    counts = rng.poisson(norm * cfg["scale"]).astype(np.float64)
    if cfg["clip"]:
        counts = np.clip(counts, 0, 1)  # binary: at most 1 photon per pixel
    print(f"{prefix}: mean={counts.mean():.3f} photons/px, max={counts.max():.0f}")
    save_frames(counts, prefix)
    save_tf_power_spectra(counts, prefix)


# 16x16 pixel thing with circle lol

In [ ]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# 16x16 image: slightly gray background + white "circle-like" center
img16 = np.full((16, 16), 0.35, dtype=np.float32)

# 4x4 core in the center
img16[6:10, 6:10] = 1.0

# 1-pixel protrusions on each side (to make it more circle-like)
img16[5, 6:10] = 1.0    # top
img16[10, 6:10] = 1.0   # bottom
img16[6:10, 5] = 1.0    # left
img16[6:10, 10] = 1.0   # right

save_path = Path("output/poster-vis/circle16x16.png")
save_path.parent.mkdir(parents=True, exist_ok=True)
plt.imsave(save_path, img16, cmap="gray", vmin=0, vmax=1)

plt.figure(figsize=(2, 2))
plt.imshow(img16, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
plt.axis("off")
plt.show()

# barebones velocity plane shear vis

In [ ]:
"""
Visualize the spectral support of a constant-velocity moving pattern:

    v_x*f_x + v_y*f_y + f_t = 0

This is a plane through the origin in (f_x, f_y, f_t) space, tilted
according to the velocity (v_x, v_y).
"""

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Times New Roman",
    "font.size": 20,
})

# ---------------------------------------------------------------- settings
extent = 1.0            # half-width of the fx,fy grid
n = 25                  # grid resolution
speed = 0.6             # velocity magnitude used for the tilted cases
wireframe = False       # draw grid wireframe on the plane
show_ft0_plane = False  # draw a transparent reference plane at ft = 0


# ---------------------------------------------------------------- geometry
lin = np.linspace(-extent, extent, n)
fx, fy = np.meshgrid(lin, lin)


def bare_axes(ax, L):
    """Draw plain fx, fy, ft axis lines through the origin, nothing else."""
    ax.set_axis_off()
    ax.plot([-L, L], [0, 0], [0, 0], color='k', lw=1)
    ax.plot([0, 0], [-L, L], [0, 0], color='k', lw=1)
    ax.plot([0, 0], [0, 0], [-L, L], color='k', lw=1)
    ax.text(L * 1.2, 0, 0, r'$f_x$', ha='center', va='center')
    ax.text(0, L * 1.1, 0, r'$f_y$', ha='center', va='center')
    ax.text(0, 0, L * 1.15, r'$f_t$', ha='center', va='center')
    ax.set_xlim(-L, L)
    ax.set_ylim(-L, L)
    ax.set_zlim(-L, L)


def plot_plane(ax, vx, vy, wireframe=False, show_ft0_plane=False):
    # tilted plane: vx*fx + vy*fy + ft = 0  ->  ft = -(vx*fx + vy*fy)
    ft = -(vx * fx + vy * fy)
    ax.plot_surface(fx, fy, ft, color='tab:blue', alpha=0.55,
                    linewidth=0, antialiased=True)
    if wireframe:
        ax.plot_wireframe(fx, fy, ft, color='k', linewidth=0.4, alpha=0.6,
                          rcount=10, ccount=10)
    # outline around the plane: perimeter of the fx,fy square lifted onto it
    e = extent
    cx = np.array([-e, e, e, -e, -e])
    cy = np.array([-e, -e, e, e, -e])
    ax.plot(cx, cy, -(vx * cx + vy * cy), color='k', lw=1.2)
    if show_ft0_plane:
        # transparent reference plane at ft = 0 (the fx, fy plane)
        ax.plot_surface(fx, fy, np.zeros_like(fx), color='gray', alpha=0.2,
                        linewidth=0, antialiased=True)
        ax.plot(cx, cy, np.zeros_like(cx), color='gray', lw=1.0)
    # dashed line where the tilted plane meets ft = 0: vx*fx + vy*fy = 0,
    # i.e. the line through the origin perpendicular to (vx, vy)
    if vx != 0 or vy != 0:
        dx, dy = -vy, vx
        t = e / max(abs(dx), abs(dy))
        ax.plot([-t * dx, t * dx], [-t * dy, t * dy], [0, 0],
                color='k', lw=1.2, ls='--')
    bare_axes(ax, extent * 1.3)
    ax.view_init(elev=18, azim=-20)


# ---------------------------------------------------------------- three planes
cases = [
    (0.0, speed, 'vx0'),      # vx = 0
    (speed, 0.0, 'vy0'),      # vy = 0
    (speed, speed, 'vxvy'),   # vx = vy
    (0, 0, '00'),
]
for vx, vy, tag in cases:
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(1, 1, 1, projection='3d')
    plot_plane(ax, vx, vy, wireframe=wireframe, show_ft0_plane=show_ft0_plane)
    plt.tight_layout()
    plt.savefig(f'output/poster-vis/motion_plane_{tag}.svg', dpi=300, transparent=True)
    plt.show()


# should've been visualizing these fourier cubes as isosurfaces or something

In [ ]:
from sptflux import ihpp_data, utils

In [ ]:
import numpy as np
import plotly.graph_objects as go


def _disk(yy, xx, cy, cx, radius):
    """Hard-edge disk sampled on the grid: 1.0 inside, 0.0 outside (no AA)."""
    return ((yy - cy) ** 2 + (xx - cx) ** 2 <= radius ** 2).astype(np.float64)


def moving_circle_flux(T, H, W, radius, center0, velocity):
    """Hard-edge disk of `radius` moving in a straight line, sampled on a grid.

    Returns a (T, H, W) volume. Sampling is binary (no anti-aliasing / partial
    coverage). center0 = (cy, cx) at t=0; velocity = (vy, vx) in pixels/frame.
    """
    yy, xx = np.mgrid[0:H, 0:W]
    vol = np.zeros((T, H, W), dtype=np.float64)
    for t in range(T):
        cy = center0[0] + velocity[0] * t
        cx = center0[1] + velocity[1] * t
        vol[t] = _disk(yy, xx, cy, cx, radius)
    return vol


def orbiting_circle_flux(T, H, W, radius, center0, orbit_radius, n_rev=1.0, phase=0.0):
    """Hard-edge disk of `radius` whose center orbits a circle, grid-sampled.

    The center traces a circle of `orbit_radius` around `center0 = (cy, cx)`,
    completing `n_rev` revolutions over the T frames. Sampling is binary.
    """
    yy, xx = np.mgrid[0:H, 0:W]
    vol = np.zeros((T, H, W), dtype=np.float64)
    theta = 2 * np.pi * n_rev * np.arange(T) / T + phase
    for t in range(T):
        cy = center0[0] + orbit_radius * np.sin(theta[t])
        cx = center0[1] + orbit_radius * np.cos(theta[t])
        vol[t] = _disk(yy, xx, cy, cx, radius)
    return vol


def power_isosurface(vol, title):
    """3D rfftn (time as the real axis) -> log-power isosurface plot.

    axes=(1, 2, 0) makes time last in the axes list, so rfft is applied to it;
    output axis 0 holds the time frequencies -> shape (T//2+1, H, W).
    """
    T, H, W = vol.shape
    F = np.fft.rfftn(vol, axes=(1, 2, 0))
    power = np.log1p(np.abs(np.fft.fftshift(F, axes=(1, 2))) ** 2)  # shift spatial axes only

    ft = np.fft.rfftfreq(T)                  # non-negative time freqs (rfft axis)
    fy = np.fft.fftshift(np.fft.fftfreq(H))  # centered spatial freqs
    fx = np.fft.fftshift(np.fft.fftfreq(W))
    FT, FY, FX = np.meshgrid(ft, fy, fx, indexing="ij")

    fig = go.Figure(go.Isosurface(
        x=FX.ravel(), y=FY.ravel(), z=FT.ravel(),
        value=power.ravel(),
        isomin=float(np.percentile(power, 99.0)),
        isomax=float(power.max()),
        surface_count=3,
        opacity=0.5,
        caps=dict(x_show=False, y_show=False, z_show=False),
        colorscale="Magma",
        colorbar=dict(title="log power"),
    ))
    fig.update_layout(
        scene=dict(xaxis_title="fx", yaxis_title="fy", zaxis_title="ft"),
        width=650, height=650,
        title=title,
    )
    fig.show()
    return fig


T, H, W = 64, 64, 64

# straight-line motion: energy concentrates on the tilted plane
# vx*fx + vy*fy + ft = 0, so the isosurface reads as a tilted slab.
vol_line = moving_circle_flux(
    T, H, W, radius=8.0,
    center0=(H / 2 - 15, W / 2 - 15),  # start off-center
    velocity=(0.4, 0.6),               # straight line, pixels/frame
)
power_isosurface(vol_line, "Power spectrum: circle moving in a straight line")

# circular (orbital) motion: periodic in time, so energy lands on discrete
# time-frequency harmonics of the orbit -> stacked rings instead of one plane.
vol_orbit = orbiting_circle_flux(
    T, H, W, radius=8.0,
    center0=(H / 2, W / 2),
    orbit_radius=14.0,
    n_rev=1.0,
)
power_isosurface(vol_orbit, "Power spectrum: circle moving in a circle")
